In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 30)

# ============================================================
# 1. Carregamento da base patrimonial clusterizada
# ============================================================

df_pat = pd.read_csv(
    "features_patrimoniais_por_candidato_cluster.csv",
    sep=";",
    encoding="utf-8"
)

df_pat.head()

,SQ_CANDIDATO,valor_ativos_financeiros,valor_bens_luxo_colecao,valor_creditos_direitos,valor_dinheiro_especie,valor_direitos_intangiveis,valor_imoveis,valor_outros,valor_outros_atividade_profissional,valor_participacoes_societarias,valor_rural_agropecuario,valor_veiculos,perc_ativos_financeiros,perc_bens_luxo_colecao,perc_creditos_direitos,perc_dinheiro_especie,perc_direitos_intangiveis,perc_imoveis,perc_outros,perc_outros_atividade_profissional,perc_participacoes_societarias,perc_rural_agropecuario,perc_veiculos,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,patrimonio_total,estrato,perfil_cluster
0,10001595335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.000000,2,1,1.000000,60000.0,01_baixo,Baixo patrimônio - veicular concentrado com ru...
1,10001595336,0.0,0.0,0.0,0.0,0.0,400000.0,0.0,0.0,0.0,0.0,113900.0,0.0,0.0,0.0,0.0,0.0,0.778362,0.0,0.0,0.0,0.0,0.221638,3,2,0.654970,513900.0,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...
2,10001595338,0.0,0.0,0.0,0.0,0.0,300000.0,0.0,0.0,0.0,0.0,12000.0,0.0,0.0,0.0,0.0,0.0,0.961538,0.0,0.0,0.0,0.0,0.038462,2,2,0.926036,312000.0,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...
3,10001595339,0.0,0.0,0.0,0.0,0.0,250000.0,0.0,0.0,0.0,0.0,100000.0,0.0,0.0,0.0,0.0,0.0,0.714286,0.0,0.0,0.0,0.0,0.285714,2,2,0.591837,350000.0,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...
4,10001595340,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32000.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.000000,2,1,1.000000,32000.0,01_baixo,Baixo patrimônio - veicular concentrado com ru...


In [2]:
# ============================================================
# 2. Checagens iniciais
# ============================================================

resumo_base = pd.DataFrame({
    "indicador": [
        "linhas",
        "colunas",
        "candidatos_unicos",
        "duplicatas_sq_candidato",
        "estratos",
        "clusters"
    ],
    "valor": [
        len(df_pat),
        df_pat.shape[1],
        df_pat["SQ_CANDIDATO"].nunique(),
        len(df_pat) - df_pat["SQ_CANDIDATO"].nunique(),
        df_pat["estrato"].nunique(),
        df_pat["perfil_cluster"].nunique()
    ]
})

resumo_base

,indicador,valor
0,linhas,18219
1,colunas,29
2,candidatos_unicos,18219
3,duplicatas_sq_candidato,0
4,estratos,5
5,clusters,35


In [3]:
# ============================================================
# 3. Composição geral dos estratos
# ============================================================

tabela_estratos = (
    df_pat
    .groupby("estrato", observed=True)
    .agg(
        candidatos=("SQ_CANDIDATO", "count"),
        patrimonio_min=("patrimonio_total", "min"),
        patrimonio_p25=("patrimonio_total", lambda x: x.quantile(0.25)),
        patrimonio_mediano=("patrimonio_total", "median"),
        patrimonio_p75=("patrimonio_total", lambda x: x.quantile(0.75)),
        patrimonio_medio=("patrimonio_total", "mean"),
        patrimonio_max=("patrimonio_total", "max"),
        qtd_bens_mediana=("qtd_bens", "median"),
        macros_mediana=("qtd_macros_presentes", "median"),
        concentracao_mediana=("indice_concentracao_macro", "median")
    )
    .reset_index()
)

tabela_estratos["perc_base"] = (
    tabela_estratos["candidatos"] / tabela_estratos["candidatos"].sum()
)

tabela_estratos = tabela_estratos[
    [
        "estrato",
        "candidatos",
        "perc_base",
        "patrimonio_min",
        "patrimonio_p25",
        "patrimonio_mediano",
        "patrimonio_p75",
        "patrimonio_medio",
        "patrimonio_max",
        "qtd_bens_mediana",
        "macros_mediana",
        "concentracao_mediana"
    ]
]

tabela_estratos

,estrato,candidatos,perc_base,patrimonio_min,patrimonio_p25,patrimonio_mediano,patrimonio_p75,patrimonio_medio,patrimonio_max,qtd_bens_mediana,macros_mediana,concentracao_mediana
0,01_baixo,5365,0.294473,0.01,1.100000e+04,3.200000e+04,60206.00,3.833371e+04,1.000000e+05,1.0,1.0,1.000000
1,02_medio_baixo,4186,0.229760,100021.05,1.426898e+05,1.880000e+05,240718.39,1.920508e+05,3.000000e+05,2.0,2.0,0.777410
2,03_medio_alto,4069,0.223338,300005.52,3.761770e+05,4.657272e+05,594152.57,4.879935e+05,7.500000e+05,4.0,2.0,0.698386
3,04_alto,4410,0.242055,750149.76,1.010638e+06,1.440935e+06,2500000.00,2.188159e+06,1.300000e+07,8.0,3.0,0.581909
4,05_ultra_alto,189,0.010374,13114000.00,1.708115e+07,2.465718e+07,46569840.76,5.839267e+07,1.267951e+09,24.0,5.0,0.535014


In [4]:
# ============================================================
# 4. Composição patrimonial agregada por estrato
# ============================================================

valor_cols = [col for col in df_pat.columns if col.startswith("valor_")]

valores_por_estrato = (
    df_pat
    .groupby("estrato", observed=True)[valor_cols]
    .sum()
)

patrimonio_por_estrato = (
    df_pat
    .groupby("estrato", observed=True)["patrimonio_total"]
    .sum()
)

composicao_agregada_estrato = (
    valores_por_estrato
    .div(patrimonio_por_estrato, axis=0)
    .mul(100)
    .reset_index()
)

composicao_agregada_estrato

,estrato,valor_ativos_financeiros,valor_bens_luxo_colecao,valor_creditos_direitos,valor_dinheiro_especie,valor_direitos_intangiveis,valor_imoveis,valor_outros,valor_outros_atividade_profissional,valor_participacoes_societarias,valor_rural_agropecuario,valor_veiculos
0,01_baixo,8.604781,0.071604,0.707752,3.779732,0.003000,26.595327,1.949799,0.298596,7.270331,2.012087,48.706990
1,02_medio_baixo,6.053612,0.025541,1.000420,2.543973,0.027092,58.682675,1.292479,0.072932,6.041239,3.613761,20.646277
2,03_medio_alto,7.044459,0.057255,1.457699,2.605640,0.000252,65.057097,1.123435,0.032372,5.128939,5.115498,12.377354
3,04_alto,13.165401,0.126170,3.247155,1.576625,0.035959,49.494868,2.196801,0.017333,13.205694,11.057442,5.876552
4,05_ultra_alto,17.752016,0.299422,8.990199,0.636156,0.000000,11.035351,0.948797,0.001168,48.769774,10.264501,1.302616


In [5]:
# ============================================================
# 5. Principais componentes patrimoniais de cada estrato
# ============================================================

composicao_long = (
    composicao_agregada_estrato
    .melt(
        id_vars="estrato",
        var_name="macro_patrimonial",
        value_name="perc_patrimonio_estrato"
    )
)

composicao_long["macro_patrimonial"] = (
    composicao_long["macro_patrimonial"]
    .str.replace("valor_", "", regex=False)
)

top_componentes_estrato = (
    composicao_long
    .sort_values(
        ["estrato", "perc_patrimonio_estrato"],
        ascending=[True, False]
    )
    .groupby("estrato", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_componentes_estrato

,estrato,macro_patrimonial,perc_patrimonio_estrato
0,01_baixo,veiculos,48.706990
1,01_baixo,imoveis,26.595327
2,01_baixo,ativos_financeiros,8.604781
3,01_baixo,participacoes_societarias,7.270331
4,01_baixo,dinheiro_especie,3.779732
5,02_medio_baixo,imoveis,58.682675
6,02_medio_baixo,veiculos,20.646277
7,02_medio_baixo,ativos_financeiros,6.053612
8,02_medio_baixo,participacoes_societarias,6.041239
9,02_medio_baixo,rural_agropecuario,3.613761


In [6]:
# ============================================================
# 6. Composição dos clusters dentro de cada estrato
# ============================================================

clusters_por_estrato = (
    df_pat
    .groupby(["estrato", "perfil_cluster"], observed=True)
    .size()
    .reset_index(name="candidatos")
)

clusters_por_estrato["perc_estrato"] = (
    clusters_por_estrato["candidatos"]
    / clusters_por_estrato.groupby("estrato")["candidatos"].transform("sum")
)

top_clusters_estrato = (
    clusters_por_estrato
    .sort_values(["estrato", "candidatos"], ascending=[True, False])
    .groupby("estrato", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_clusters_estrato

,estrato,perfil_cluster,candidatos,perc_estrato
0,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,0.408388
1,01_baixo,Baixo patrimônio - imobiliário concentrado,718,0.133830
2,01_baixo,Baixo patrimônio - financeiro concentrado,690,0.128611
3,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,0.079590
4,01_baixo,Baixo patrimônio - societário concentrado,411,0.076608
5,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1154,0.275681
6,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1033,0.246775
7,02_medio_baixo,Patrimônio médio-baixo - imobiliário + veicular,443,0.105829
8,02_medio_baixo,Patrimônio médio-baixo - veicular concentrado,364,0.086957
9,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,0.086001


In [7]:
!pip install plotly

In [8]:
# ============================================================
# 7. Configurações para gráficos
# ============================================================

import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import textwrap

PASTA_GRAFICOS = Path("plot")
PASTA_GRAFICOS.mkdir(exist_ok=True)

RENDERIZAR_NO_NOTEBOOK = True  # troque para False se quiser apenas salvar HTML

def quebra_label(texto, largura=45):
    if pd.isna(texto):
        return texto
    return "<br>".join(textwrap.wrap(str(texto), width=largura))

In [9]:
# ============================================================
# 8. Base de composição cluster x estrato
# ============================================================

clusters_por_estrato = (
    df_pat
    .groupby(["estrato", "perfil_cluster"], observed=True)
    .size()
    .reset_index(name="candidatos")
)

clusters_por_estrato["total_estrato"] = (
    clusters_por_estrato
    .groupby("estrato")["candidatos"]
    .transform("sum")
)

clusters_por_estrato["perc_estrato"] = (
    clusters_por_estrato["candidatos"]
    / clusters_por_estrato["total_estrato"]
)

clusters_por_estrato = clusters_por_estrato.sort_values(
    ["estrato", "candidatos"],
    ascending=[True, False]
)

clusters_por_estrato.head(10)

,estrato,perfil_cluster,candidatos,total_estrato,perc_estrato
7,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,5365,0.408388
3,01_baixo,Baixo patrimônio - imobiliário concentrado,718,5365,0.133830
1,01_baixo,Baixo patrimônio - financeiro concentrado,690,5365,0.128611
6,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,5365,0.079590
5,01_baixo,Baixo patrimônio - societário concentrado,411,5365,0.076608
0,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,364,5365,0.067847
2,01_baixo,Baixo patrimônio - imobiliário + veicular,306,5365,0.057036
4,01_baixo,Baixo patrimônio - outros ativos diversificado,258,5365,0.048089
11,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1154,4186,0.275681
10,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1033,4186,0.246775


In [10]:
# ============================================================
# 9. Gráfico 1 — Composição percentual dos clusters por estrato
# Top N clusters dentro de cada estrato + Outros
# ============================================================

TOP_N = 7

df_top = clusters_por_estrato.copy()

df_top["rank_no_estrato"] = (
    df_top
    .groupby("estrato")["candidatos"]
    .rank(method="first", ascending=False)
)

df_top["cluster_plot"] = np.where(
    df_top["rank_no_estrato"] <= TOP_N,
    df_top["perfil_cluster"],
    "Outros clusters"
)

df_compacta = (
    df_top
    .groupby(["estrato", "cluster_plot"], observed=True)
    .agg(candidatos=("candidatos", "sum"))
    .reset_index()
)

df_compacta["total_estrato"] = (
    df_compacta
    .groupby("estrato")["candidatos"]
    .transform("sum")
)

df_compacta["perc_estrato"] = (
    df_compacta["candidatos"]
    / df_compacta["total_estrato"]
)

df_compacta["cluster_plot_quebrado"] = df_compacta["cluster_plot"].apply(
    lambda x: quebra_label(x, 45)
)

fig = px.bar(
    df_compacta,
    x="estrato",
    y="perc_estrato",
    color="cluster_plot_quebrado",
    text=df_compacta["perc_estrato"].map(lambda x: f"{x:.1%}"),
    hover_data={
        "estrato": True,
        "cluster_plot_quebrado": False,
        "candidatos": True,
        "perc_estrato": ":.2%",
    },
    labels={
        "estrato": "Estrato patrimonial",
        "perc_estrato": "% dentro do estrato",
        "cluster_plot_quebrado": "Cluster"
    },
    title=f"Composição dos clusters dentro de cada estrato — Top {TOP_N} + Outros"
)

fig.update_layout(
    barmode="stack",
    yaxis_tickformat=".0%",
    legend_title_text="Cluster",
    height=700
)

fig.write_html(PASTA_GRAFICOS / "01_composicao_clusters_por_estrato.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [11]:
# ============================================================
# 13. Tabela auxiliar — cluster dominante por estratplotlyo
# ============================================================

cluster_dominante_estrato = (
    clusters_por_estrato
    .sort_values(["estrato", "candidatos"], ascending=[True, False])
    .groupby("estrato", observed=True)
    .head(1)
    .reset_index(drop=True)
    [
        [
            "estrato",
            "perfil_cluster",
            "candidatos",
            "perc_estrato"
        ]
    ]
)

cluster_dominante_estrato

,estrato,perfil_cluster,candidatos,perc_estrato
0,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,0.408388
1,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1154,0.275681
2,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,1152,0.283116
3,04_alto,Alto patrimônio - imobiliário + societário com...,1060,0.240363
4,05_ultra_alto,Patrimônio ultra-alto - societário concentrado,57,0.301587


In [12]:
# ============================================================
# 14. Tabela auxiliar — diversidade de clusters por estrato
# ============================================================

diversidade_clusters_estrato = (
    clusters_por_estrato
    .groupby("estrato", observed=True)
    .agg(
        candidatos=("candidatos", "sum"),
        clusters_presentes=("perfil_cluster", "nunique"),
        maior_cluster_candidatos=("candidatos", "max")
    )
    .reset_index()
)

diversidade_clusters_estrato["peso_maior_cluster"] = (
    diversidade_clusters_estrato["maior_cluster_candidatos"]
    / diversidade_clusters_estrato["candidatos"]
)

diversidade_clusters_estrato

,estrato,candidatos,clusters_presentes,maior_cluster_candidatos,peso_maior_cluster
0,01_baixo,5365,8,2191,0.408388
1,02_medio_baixo,4186,8,1154,0.275681
2,03_medio_alto,4069,7,1152,0.283116
3,04_alto,4410,7,1060,0.240363
4,05_ultra_alto,189,5,57,0.301587


In [13]:
# ============================================================
# SEÇÃO 1 — Integração da base eleitoral com a base patrimonial
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 40)

# Ajuste este caminho conforme o nome real da sua base eleitoral
ARQUIVO_CANDIDATOS = Path("consulta_cand_2022_BRASIL.csv")

# Base patrimonial clusterizada
ARQUIVO_PATRIMONIAL = Path("features_patrimoniais_por_candidato_cluster.csv")

In [14]:
# ============================================================
# 1.1 Função auxiliar para leitura de arquivos
# ============================================================

def ler_base(caminho):
    caminho = Path(caminho)
    
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")
    
    if caminho.suffix.lower() == ".csv":
        tentativas = [
            {"sep": ";", "encoding": "utf-8"},
            {"sep": ";", "encoding": "latin1"},
            {"sep": ",", "encoding": "utf-8"},
            {"sep": ",", "encoding": "latin1"},
        ]
        
        ultimo_erro = None
        
        for params in tentativas:
            try:
                return pd.read_csv(caminho, low_memory=False, **params)
            except Exception as erro:
                ultimo_erro = erro
        
        raise ultimo_erro
    
    elif caminho.suffix.lower() in [".parquet", ".pq"]:
        return pd.read_parquet(caminho)
    
    elif caminho.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(caminho)
    
    else:
        raise ValueError(f"Formato não suportado: {caminho.suffix}")

In [15]:
# ============================================================
# 1.2 Carregamento das bases
# ============================================================

df_candidatos = ler_base(ARQUIVO_CANDIDATOS)
df_pat = ler_base(ARQUIVO_PATRIMONIAL)

print("Base eleitoral:", df_candidatos.shape)
print("Base patrimonial clusterizada:", df_pat.shape)

Base eleitoral: (29314, 50)
Base patrimonial clusterizada: (18219, 29)


In [16]:
# ============================================================
# 1.3 Padronização mínima das chaves
# ============================================================

df_candidatos["SQ_CANDIDATO"] = df_candidatos["SQ_CANDIDATO"].astype(str)
df_pat["SQ_CANDIDATO"] = df_pat["SQ_CANDIDATO"].astype(str)

print("Candidatos únicos na base eleitoral:", df_candidatos["SQ_CANDIDATO"].nunique())
print("Candidatos únicos na base patrimonial:", df_pat["SQ_CANDIDATO"].nunique())

Candidatos únicos na base eleitoral: 29262
Candidatos únicos na base patrimonial: 18219


In [17]:
# ============================================================
# 1.4 Remoção de duplicatas na base eleitoral
# ============================================================

duplicatas_candidatos = (
    len(df_candidatos) 
    - df_candidatos["SQ_CANDIDATO"].nunique()
)

print("Duplicatas em SQ_CANDIDATO na base eleitoral:", duplicatas_candidatos)

df_candidatos = (
    df_candidatos
    .sort_values("SQ_CANDIDATO")
    .drop_duplicates(subset="SQ_CANDIDATO", keep="first")
    .copy()
)

print("Base eleitoral após remoção de duplicatas:", df_candidatos.shape)

Duplicatas em SQ_CANDIDATO na base eleitoral: 52
Base eleitoral após remoção de duplicatas: (29262, 50)


In [18]:
# ============================================================
# 1.5 Seleção das colunas eleitorais principais
# ============================================================

colunas_eleitorais_desejadas = [
    "SQ_CANDIDATO",
    "NM_CANDIDATO",
    "DS_CARGO",
    "SG_PARTIDO",
    "NM_PARTIDO",
    "DS_GENERO",
    "DS_GRAU_INSTRUCAO",
    "DS_COR_RACA",
    "DS_OCUPACAO",
    "DS_SIT_TOT_TURNO",
    "SG_UF"
]

colunas_eleitorais_existentes = [
    col for col in colunas_eleitorais_desejadas
    if col in df_candidatos.columns
]

colunas_faltantes = sorted(
    set(colunas_eleitorais_desejadas) 
    - set(colunas_eleitorais_existentes)
)

print("Colunas eleitorais encontradas:", len(colunas_eleitorais_existentes))
print("Colunas eleitorais faltantes:", colunas_faltantes)

df_candidatos_sel = df_candidatos[colunas_eleitorais_existentes].copy()
df_candidatos_sel.head()

Colunas eleitorais encontradas: 11
Colunas eleitorais faltantes: []


,SQ_CANDIDATO,NM_CANDIDATO,DS_CARGO,SG_PARTIDO,NM_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,DS_COR_RACA,DS_OCUPACAO,DS_SIT_TOT_TURNO,SG_UF
26626,100001599072,ADRIANO AURÉLIO DE MENEZES BRAGA,DEPUTADO FEDERAL,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,MASCULINO,SUPERIOR COMPLETO,BRANCA,ADVOGADO,NÃO ELEITO,MA
7510,100001599073,ALAIR BATISTA FIRMIANO,DEPUTADO FEDERAL,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,MASCULINO,SUPERIOR COMPLETO,BRANCA,MÉDICO,SUPLENTE,MA
7496,100001599074,CARLOS ROBERTO DINIZ ARAÚJO,DEPUTADO FEDERAL,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,MASCULINO,SUPERIOR COMPLETO,PRETA,ADMINISTRADOR,SUPLENTE,MA
7492,100001599075,CLAUDIO JORGE VIEIRA PINTO,DEPUTADO FEDERAL,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,MASCULINO,ENSINO MÉDIO COMPLETO,PARDA,OUTROS,SUPLENTE,MA
25874,100001599076,EDISON LOBÃO FILHO,DEPUTADO FEDERAL,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,MASCULINO,SUPERIOR COMPLETO,BRANCA,EMPRESÁRIO,SUPLENTE,MA


In [19]:
# ============================================================
# 1.6 Integração eleitoral + patrimonial
# ============================================================

df_integrado = (
    df_candidatos_sel
    .merge(
        df_pat,
        on="SQ_CANDIDATO",
        how="inner",
        validate="one_to_one"
    )
)

df_candidatos_sem_bens = (
    df_candidatos_sel
    .loc[
        ~df_candidatos_sel["SQ_CANDIDATO"].isin(df_pat["SQ_CANDIDATO"])
    ]
    .copy()
)

print("Base eleitoral original:", df_candidatos_sel.shape)
print("Base integrada:", df_integrado.shape)
print("Candidatos sem bens / sem cluster:", df_candidatos_sem_bens.shape)

Base eleitoral original: (29262, 11)
Base integrada: (18219, 39)
Candidatos sem bens / sem cluster: (11043, 11)


In [20]:
# ============================================================
# 1.7 Checagem da integração
# ============================================================

resumo_integracao = pd.DataFrame({
    "indicador": [
        "candidatos_base_eleitoral",
        "candidatos_base_patrimonial",
        "candidatos_integrados",
        "candidatos_sem_bens_ou_sem_cluster",
        "clusters_integrados",
        "estratos_integrados"
    ],
    "valor": [
        df_candidatos_sel["SQ_CANDIDATO"].nunique(),
        df_pat["SQ_CANDIDATO"].nunique(),
        df_integrado["SQ_CANDIDATO"].nunique(),
        df_candidatos_sem_bens["SQ_CANDIDATO"].nunique(),
        df_integrado["perfil_cluster"].nunique(),
        df_integrado["estrato"].nunique()
    ]
})

resumo_integracao

,indicador,valor
0,candidatos_base_eleitoral,29262
1,candidatos_base_patrimonial,18219
2,candidatos_integrados,18219
3,candidatos_sem_bens_ou_sem_cluster,11043
4,clusters_integrados,35
5,estratos_integrados,5


In [21]:
# ============================================================
# 1.8 Criação da variável de sucesso eleitoral
# ============================================================

situacoes_eleitos = [
    "ELEITO",
    "ELEITO POR QP",
    "ELEITO POR MÉDIA",
    "ELEITO POR MEDIA"
]

df_integrado["DS_SIT_TOT_TURNO_LIMPA"] = (
    df_integrado["DS_SIT_TOT_TURNO"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df_integrado["eleito"] = (
    df_integrado["DS_SIT_TOT_TURNO_LIMPA"]
    .isin(situacoes_eleitos)
    .astype(int)
)

df_integrado[["DS_SIT_TOT_TURNO", "DS_SIT_TOT_TURNO_LIMPA", "eleito"]].drop_duplicates()

,DS_SIT_TOT_TURNO,DS_SIT_TOT_TURNO_LIMPA,eleito
0,NÃO ELEITO,NÃO ELEITO,0
1,SUPLENTE,SUPLENTE,0
7,ELEITO POR QP,ELEITO POR QP,1
34,#NULO,#NULO,0
42,ELEITO POR MÉDIA,ELEITO POR MÉDIA,1
155,ELEITO,ELEITO,1
1452,2º TURNO,2º TURNO,0


In [22]:
# ============================================================
# 1.9 Resumo inicial da variável de sucesso eleitoral
# ============================================================

resumo_sucesso = pd.DataFrame({
    "indicador": [
        "candidatos_integrados",
        "eleitos",
        "nao_eleitos",
        "taxa_eleicao"
    ],
    "valor": [
        len(df_integrado),
        df_integrado["eleito"].sum(),
        len(df_integrado) - df_integrado["eleito"].sum(),
        df_integrado["eleito"].mean()
    ]
})

resumo_sucesso

,indicador,valor
0,candidatos_integrados,18219.000000
1,eleitos,1611.000000
2,nao_eleitos,16608.000000
3,taxa_eleicao,0.088424


In [23]:
# ============================================================
# EIXO 1 — Desempenho eleitoral dos clusters
# ============================================================
#
# Pergunta:
# Quais perfis patrimoniais apresentam maior sucesso eleitoral?
#
# Unidade principal de análise:
# perfil_cluster
#
# Métrica central:
# taxa_eleicao = eleitos / candidatos
# ============================================================

In [24]:
# ============================================================
# 2.1 Tabela principal — desempenho eleitoral por cluster
# ============================================================

desempenho_clusters = (
    df_integrado
    .groupby("perfil_cluster", observed=True)
    .agg(
        candidatos=("SQ_CANDIDATO", "count"),
        eleitos=("eleito", "sum"),
        patrimonio_mediano=("patrimonio_total", "median"),
        patrimonio_medio=("patrimonio_total", "mean"),
        estratos_presentes=("estrato", "nunique")
    )
    .reset_index()
)

desempenho_clusters["taxa_eleicao"] = (
    desempenho_clusters["eleitos"]
    / desempenho_clusters["candidatos"]
)

desempenho_clusters = desempenho_clusters.sort_values(
    ["taxa_eleicao", "candidatos"],
    ascending=[False, False]
)

desempenho_clusters.head(15)

,perfil_cluster,candidatos,eleitos,patrimonio_mediano,patrimonio_medio,estratos_presentes,taxa_eleicao
31,Patrimônio ultra-alto - financeiro + societário,37,13,2.651250e+07,5.229810e+07,1,0.351351
34,Patrimônio ultra-alto - societário concentrado,57,18,3.969496e+07,9.458353e+07,1,0.315789
33,Patrimônio ultra-alto - rural/agropecuário con...,46,14,2.100000e+07,2.646900e+07,1,0.304348
6,Alto patrimônio - imobiliário diversificado co...,474,141,2.575782e+06,3.565677e+06,1,0.297468
30,Patrimônio ultra-alto - créditos e direitos di...,23,6,2.864532e+07,6.529224e+07,1,0.260870
0,Alto patrimônio - imobiliário + financeiro,677,155,1.286471e+06,1.691100e+06,1,0.228951
5,Alto patrimônio - imobiliário diversificado co...,452,100,2.042919e+06,2.952399e+06,1,0.221239
2,Alto patrimônio - imobiliário + societário com...,1060,215,1.530500e+06,2.263245e+06,1,0.202830
17,Patrimônio médio-alto - imobiliário concentrad...,392,77,4.876473e+05,5.040474e+05,1,0.196429
21,Patrimônio médio-alto - veicular diversificado,398,74,4.265297e+05,4.602855e+05,1,0.185930


In [25]:
# ============================================================
# 2.2 Tabela com filtro mínimo de candidatos
# ============================================================
#
# Evita interpretar clusters muito pequenos como se fossem achados robustos.
# Ajuste o limiar conforme o tamanho da base.
# ============================================================

MIN_CANDIDATOS_CLUSTER = 30

desempenho_clusters_filtrado = (
    desempenho_clusters
    .loc[desempenho_clusters["candidatos"] >= MIN_CANDIDATOS_CLUSTER]
    .copy()
)

desempenho_clusters_filtrado = desempenho_clusters_filtrado.sort_values(
    ["taxa_eleicao", "candidatos"],
    ascending=[False, False]
)

desempenho_clusters_filtrado.head(15)

,perfil_cluster,candidatos,eleitos,patrimonio_mediano,patrimonio_medio,estratos_presentes,taxa_eleicao
31,Patrimônio ultra-alto - financeiro + societário,37,13,2.651250e+07,5.229810e+07,1,0.351351
34,Patrimônio ultra-alto - societário concentrado,57,18,3.969496e+07,9.458353e+07,1,0.315789
33,Patrimônio ultra-alto - rural/agropecuário con...,46,14,2.100000e+07,2.646900e+07,1,0.304348
6,Alto patrimônio - imobiliário diversificado co...,474,141,2.575782e+06,3.565677e+06,1,0.297468
0,Alto patrimônio - imobiliário + financeiro,677,155,1.286471e+06,1.691100e+06,1,0.228951
5,Alto patrimônio - imobiliário diversificado co...,452,100,2.042919e+06,2.952399e+06,1,0.221239
2,Alto patrimônio - imobiliário + societário com...,1060,215,1.530500e+06,2.263245e+06,1,0.202830
17,Patrimônio médio-alto - imobiliário concentrad...,392,77,4.876473e+05,5.040474e+05,1,0.196429
21,Patrimônio médio-alto - veicular diversificado,398,74,4.265297e+05,4.602855e+05,1,0.185930
1,Alto patrimônio - imobiliário + rural/agropecu...,636,106,1.479259e+06,2.126573e+06,1,0.166667


In [26]:
# ============================================================
# 2.3 Clusters com melhor desempenho eleitoral
# ============================================================

top_clusters_eleicao = (
    desempenho_clusters_filtrado
    .sort_values(["taxa_eleicao", "candidatos"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True)
)

top_clusters_eleicao

,perfil_cluster,candidatos,eleitos,patrimonio_mediano,patrimonio_medio,estratos_presentes,taxa_eleicao
0,Patrimônio ultra-alto - financeiro + societário,37,13,2.651250e+07,5.229810e+07,1,0.351351
1,Patrimônio ultra-alto - societário concentrado,57,18,3.969496e+07,9.458353e+07,1,0.315789
2,Patrimônio ultra-alto - rural/agropecuário con...,46,14,2.100000e+07,2.646900e+07,1,0.304348
3,Alto patrimônio - imobiliário diversificado co...,474,141,2.575782e+06,3.565677e+06,1,0.297468
4,Alto patrimônio - imobiliário + financeiro,677,155,1.286471e+06,1.691100e+06,1,0.228951
5,Alto patrimônio - imobiliário diversificado co...,452,100,2.042919e+06,2.952399e+06,1,0.221239
6,Alto patrimônio - imobiliário + societário com...,1060,215,1.530500e+06,2.263245e+06,1,0.202830
7,Patrimônio médio-alto - imobiliário concentrad...,392,77,4.876473e+05,5.040474e+05,1,0.196429
8,Patrimônio médio-alto - veicular diversificado,398,74,4.265297e+05,4.602855e+05,1,0.185930
9,Alto patrimônio - imobiliário + rural/agropecu...,636,106,1.479259e+06,2.126573e+06,1,0.166667


In [27]:
# ============================================================
# 2.4 Clusters com pior desempenho eleitoral
# ============================================================

bottom_clusters_eleicao = (
    desempenho_clusters_filtrado
    .sort_values(["taxa_eleicao", "candidatos"], ascending=[True, False])
    .head(10)
    .reset_index(drop=True)
)

bottom_clusters_eleicao

,perfil_cluster,candidatos,eleitos,patrimonio_mediano,patrimonio_medio,estratos_presentes,taxa_eleicao
0,Baixo patrimônio - outros ativos diversificado,258,1,7110.00,22986.005930,1,0.003876
1,Baixo patrimônio - imobiliário + veicular,306,2,74000.00,70245.165196,1,0.006536
2,Baixo patrimônio - veicular concentrado com ru...,2191,22,30000.00,35954.069475,1,0.010041
3,Patrimônio médio-baixo - imobiliário concentra...,1154,16,195000.00,195996.373648,1,0.013865
4,Patrimônio médio-baixo - imobiliário concentra...,1033,19,194500.00,197256.299990,1,0.018393
5,Baixo patrimônio - imobiliário concentrado,718,15,60000.00,60348.946337,1,0.020891
6,Baixo patrimônio - dinheiro em espécie concent...,364,9,10000.00,22230.653681,1,0.024725
7,Baixo patrimônio - societário concentrado,411,12,22036.09,36036.876448,1,0.029197
8,Baixo patrimônio - veicular concentrado com fi...,427,15,49049.18,50664.183419,1,0.035129
9,Patrimônio médio-baixo - veicular concentrado,364,13,146256.00,161321.324780,1,0.035714


In [28]:
# ============================================================
# 2.5 Comparação com a taxa média geral da base integrada
# ============================================================

taxa_media_geral = df_integrado["eleito"].mean()

desempenho_clusters_filtrado["taxa_media_geral"] = taxa_media_geral

desempenho_clusters_filtrado["indice_vs_media_geral"] = (
    desempenho_clusters_filtrado["taxa_eleicao"]
    / desempenho_clusters_filtrado["taxa_media_geral"]
)

ranking_clusters_vs_media = (
    desempenho_clusters_filtrado
    .sort_values("indice_vs_media_geral", ascending=False)
    .reset_index(drop=True)
)

ranking_clusters_vs_media.head(15)

,perfil_cluster,candidatos,eleitos,patrimonio_mediano,patrimonio_medio,estratos_presentes,taxa_eleicao,taxa_media_geral,indice_vs_media_geral
0,Patrimônio ultra-alto - financeiro + societário,37,13,2.651250e+07,5.229810e+07,1,0.351351,0.088424,3.973476
1,Patrimônio ultra-alto - societário concentrado,57,18,3.969496e+07,9.458353e+07,1,0.315789,0.088424,3.571303
2,Patrimônio ultra-alto - rural/agropecuário con...,46,14,2.100000e+07,2.646900e+07,1,0.304348,0.088424,3.441908
3,Alto patrimônio - imobiliário diversificado co...,474,141,2.575782e+06,3.565677e+06,1,0.297468,0.088424,3.364107
4,Alto patrimônio - imobiliário + financeiro,677,155,1.286471e+06,1.691100e+06,1,0.228951,0.088424,2.589238
5,Alto patrimônio - imobiliário diversificado co...,452,100,2.042919e+06,2.952399e+06,1,0.221239,0.088424,2.502019
6,Alto patrimônio - imobiliário + societário com...,1060,215,1.530500e+06,2.263245e+06,1,0.202830,0.088424,2.293832
7,Patrimônio médio-alto - imobiliário concentrad...,392,77,4.876473e+05,5.040474e+05,1,0.196429,0.088424,2.221435
8,Patrimônio médio-alto - veicular diversificado,398,74,4.265297e+05,4.602855e+05,1,0.185930,0.088424,2.102702
9,Alto patrimônio - imobiliário + rural/agropecu...,636,106,1.479259e+06,2.126573e+06,1,0.166667,0.088424,1.884854


In [29]:
# ============================================================
# 2.6 Classificação interpretativa simples
# ============================================================

def classificar_desempenho(indice):
    if indice >= 1.5:
        return "muito acima da média"
    elif indice >= 1.1:
        return "acima da média"
    elif indice > 0.9:
        return "próximo da média"
    elif indice > 0.67:
        return "abaixo da média"
    else:
        return "muito abaixo da média"

ranking_clusters_vs_media["classificacao_desempenho"] = (
    ranking_clusters_vs_media["indice_vs_media_geral"]
    .apply(classificar_desempenho)
)

ranking_clusters_vs_media[
    [
        "perfil_cluster",
        "candidatos",
        "eleitos",
        "taxa_eleicao",
        "taxa_media_geral",
        "indice_vs_media_geral",
        "classificacao_desempenho"
    ]
].head(20)

,perfil_cluster,candidatos,eleitos,taxa_eleicao,taxa_media_geral,indice_vs_media_geral,classificacao_desempenho
0,Patrimônio ultra-alto - financeiro + societário,37,13,0.351351,0.088424,3.973476,muito acima da média
1,Patrimônio ultra-alto - societário concentrado,57,18,0.315789,0.088424,3.571303,muito acima da média
2,Patrimônio ultra-alto - rural/agropecuário con...,46,14,0.304348,0.088424,3.441908,muito acima da média
3,Alto patrimônio - imobiliário diversificado co...,474,141,0.297468,0.088424,3.364107,muito acima da média
4,Alto patrimônio - imobiliário + financeiro,677,155,0.228951,0.088424,2.589238,muito acima da média
5,Alto patrimônio - imobiliário diversificado co...,452,100,0.221239,0.088424,2.502019,muito acima da média
6,Alto patrimônio - imobiliário + societário com...,1060,215,0.202830,0.088424,2.293832,muito acima da média
7,Patrimônio médio-alto - imobiliário concentrad...,392,77,0.196429,0.088424,2.221435,muito acima da média
8,Patrimônio médio-alto - veicular diversificado,398,74,0.185930,0.088424,2.102702,muito acima da média
9,Alto patrimônio - imobiliário + rural/agropecu...,636,106,0.166667,0.088424,1.884854,muito acima da média


In [30]:
# ============================================================
# 2.7 Tabela sintética para exportação / relatório
# ============================================================

tabela_eixo1 = (
    ranking_clusters_vs_media
    [
        [
            "perfil_cluster",
            "candidatos",
            "eleitos",
            "taxa_eleicao",
            "taxa_media_geral",
            "indice_vs_media_geral",
            "patrimonio_mediano",
            "patrimonio_medio",
            "estratos_presentes",
            "classificacao_desempenho"
        ]
    ]
    .copy()
)

tabela_eixo1.head(20)

,perfil_cluster,candidatos,eleitos,taxa_eleicao,taxa_media_geral,indice_vs_media_geral,patrimonio_mediano,patrimonio_medio,estratos_presentes,classificacao_desempenho
0,Patrimônio ultra-alto - financeiro + societário,37,13,0.351351,0.088424,3.973476,2.651250e+07,5.229810e+07,1,muito acima da média
1,Patrimônio ultra-alto - societário concentrado,57,18,0.315789,0.088424,3.571303,3.969496e+07,9.458353e+07,1,muito acima da média
2,Patrimônio ultra-alto - rural/agropecuário con...,46,14,0.304348,0.088424,3.441908,2.100000e+07,2.646900e+07,1,muito acima da média
3,Alto patrimônio - imobiliário diversificado co...,474,141,0.297468,0.088424,3.364107,2.575782e+06,3.565677e+06,1,muito acima da média
4,Alto patrimônio - imobiliário + financeiro,677,155,0.228951,0.088424,2.589238,1.286471e+06,1.691100e+06,1,muito acima da média
5,Alto patrimônio - imobiliário diversificado co...,452,100,0.221239,0.088424,2.502019,2.042919e+06,2.952399e+06,1,muito acima da média
6,Alto patrimônio - imobiliário + societário com...,1060,215,0.202830,0.088424,2.293832,1.530500e+06,2.263245e+06,1,muito acima da média
7,Patrimônio médio-alto - imobiliário concentrad...,392,77,0.196429,0.088424,2.221435,4.876473e+05,5.040474e+05,1,muito acima da média
8,Patrimônio médio-alto - veicular diversificado,398,74,0.185930,0.088424,2.102702,4.265297e+05,4.602855e+05,1,muito acima da média
9,Alto patrimônio - imobiliário + rural/agropecu...,636,106,0.166667,0.088424,1.884854,1.479259e+06,2.126573e+06,1,muito acima da média


In [31]:
# ============================================================
# 2.8 Salvamento das tabelas do Eixo 1
# ============================================================

PASTA_SAIDAS = Path("results")
PASTA_SAIDAS.mkdir(exist_ok=True)

desempenho_clusters.to_csv(
    PASTA_SAIDAS / "eixo1_desempenho_clusters_completo.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo1.to_csv(
    PASTA_SAIDAS / "eixo1_desempenho_clusters_filtrado.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

In [32]:
# ============================================================
# SEÇÃO 2 — EIXO 2
# Desempenho eleitoral dos clusters dentro do estrato
# ============================================================
#
# Pergunta central:
# O cluster agrega informação além do volume total de patrimônio?
#
# Estratégia:
# Comparar a taxa de eleição de cada cluster com a taxa média
# do estrato patrimonial ao qual ele pertence.
#
# Métricas principais:
# taxa_cluster = eleitos_cluster / candidatos_cluster
# taxa_estrato = eleitos_estrato / candidatos_estrato
# indice_vs_estrato = taxa_cluster / taxa_estrato
# delta_pp = taxa_cluster - taxa_estrato, em pontos percentuais
# ============================================================

In [33]:
# ============================================================
# 2.0 Checagem dos objetos e colunas necessárias
# ============================================================

if "df_integrado" not in globals():
    raise NameError(
        "df_integrado não existe. Execute antes a seção de integração "
        "entre a base eleitoral e a base patrimonial."
    )

colunas_necessarias = [
    "SQ_CANDIDATO",
    "estrato",
    "perfil_cluster",
    "patrimonio_total"
]

colunas_faltantes = [
    col for col in colunas_necessarias
    if col not in df_integrado.columns
]

if colunas_faltantes:
    raise ValueError(f"Colunas faltantes em df_integrado: {colunas_faltantes}")

if "eleito" not in df_integrado.columns:
    if "DS_SIT_TOT_TURNO" not in df_integrado.columns:
        raise ValueError(
            "A coluna 'eleito' não existe e 'DS_SIT_TOT_TURNO' também não está disponível."
        )
    
    situacoes_eleitos = [
        "ELEITO",
        "ELEITO POR QP",
        "ELEITO POR MÉDIA",
        "ELEITO POR MEDIA"
    ]

    df_integrado["DS_SIT_TOT_TURNO_LIMPA"] = (
        df_integrado["DS_SIT_TOT_TURNO"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df_integrado["eleito"] = (
        df_integrado["DS_SIT_TOT_TURNO_LIMPA"]
        .isin(situacoes_eleitos)
        .astype(int)
    )

print("Base integrada:", df_integrado.shape)
print("Candidatos:", df_integrado["SQ_CANDIDATO"].nunique())
print("Estratos:", df_integrado["estrato"].nunique())
print("Clusters:", df_integrado["perfil_cluster"].nunique())
print("Taxa média geral:", round(df_integrado["eleito"].mean(), 4))

Base integrada: (18219, 41)
Candidatos: 18219
Estratos: 5
Clusters: 35
Taxa média geral: 0.0884


In [34]:
# ============================================================
# 2.1 Verificação: os clusters estão aninhados nos estratos?
# ============================================================

clusters_estratos = (
    df_integrado
    .groupby("perfil_cluster", observed=True)
    .agg(
        estratos_presentes=("estrato", "nunique"),
        candidatos=("SQ_CANDIDATO", "count")
    )
    .reset_index()
    .sort_values(["estratos_presentes", "candidatos"], ascending=[False, False])
)

resumo_aninhamento = (
    clusters_estratos["estratos_presentes"]
    .value_counts()
    .sort_index()
    .rename_axis("estratos_por_cluster")
    .reset_index(name="clusters")
)

resumo_aninhamento

,estratos_por_cluster,clusters
0,1,35


In [35]:
# ============================================================
# 2.2 Taxa média de eleição por estrato
# ============================================================

desempenho_estratos = (
    df_integrado
    .groupby("estrato", observed=True)
    .agg(
        candidatos_estrato=("SQ_CANDIDATO", "count"),
        eleitos_estrato=("eleito", "sum"),
        patrimonio_mediano_estrato=("patrimonio_total", "median"),
        patrimonio_medio_estrato=("patrimonio_total", "mean")
    )
    .reset_index()
)

desempenho_estratos["taxa_estrato"] = (
    desempenho_estratos["eleitos_estrato"]
    / desempenho_estratos["candidatos_estrato"]
)

desempenho_estratos = desempenho_estratos.sort_values("estrato")

desempenho_estratos

,estrato,candidatos_estrato,eleitos_estrato,patrimonio_mediano_estrato,patrimonio_medio_estrato,taxa_estrato
0,01_baixo,5365,118,3.200000e+04,3.833371e+04,0.021994
1,02_medio_baixo,4186,187,1.880000e+05,1.920508e+05,0.044673
2,03_medio_alto,4069,395,4.657272e+05,4.879935e+05,0.097075
3,04_alto,4410,858,1.440935e+06,2.188159e+06,0.194558
4,05_ultra_alto,189,53,2.465718e+07,5.839267e+07,0.280423


In [36]:
# ============================================================
# 2.3 Taxa de eleição dos clusters dentro de cada estrato
# ============================================================

desempenho_cluster_estrato = (
    df_integrado
    .groupby(["estrato", "perfil_cluster"], observed=True)
    .agg(
        candidatos_cluster=("SQ_CANDIDATO", "count"),
        eleitos_cluster=("eleito", "sum"),
        patrimonio_mediano_cluster=("patrimonio_total", "median"),
        patrimonio_medio_cluster=("patrimonio_total", "mean"),
        qtd_bens_mediana=("qtd_bens", "median"),
        macros_mediana=("qtd_macros_presentes", "median"),
        concentracao_mediana=("indice_concentracao_macro", "median")
    )
    .reset_index()
)

desempenho_cluster_estrato["taxa_cluster"] = (
    desempenho_cluster_estrato["eleitos_cluster"]
    / desempenho_cluster_estrato["candidatos_cluster"]
)

desempenho_cluster_estrato = desempenho_cluster_estrato.merge(
    desempenho_estratos[
        [
            "estrato",
            "candidatos_estrato",
            "eleitos_estrato",
            "taxa_estrato"
        ]
    ],
    on="estrato",
    how="left"
)

desempenho_cluster_estrato["indice_vs_estrato"] = np.where(
    desempenho_cluster_estrato["taxa_estrato"] > 0,
    desempenho_cluster_estrato["taxa_cluster"] / desempenho_cluster_estrato["taxa_estrato"],
    np.nan
)

desempenho_cluster_estrato["delta_pp"] = (
    desempenho_cluster_estrato["taxa_cluster"]
    - desempenho_cluster_estrato["taxa_estrato"]
) * 100

desempenho_cluster_estrato["participacao_no_estrato"] = (
    desempenho_cluster_estrato["candidatos_cluster"]
    / desempenho_cluster_estrato["candidatos_estrato"]
)

desempenho_cluster_estrato["participacao_eleitos_estrato"] = np.where(
    desempenho_cluster_estrato["eleitos_estrato"] > 0,
    desempenho_cluster_estrato["eleitos_cluster"]
    / desempenho_cluster_estrato["eleitos_estrato"],
    np.nan
)

desempenho_cluster_estrato = desempenho_cluster_estrato.sort_values(
    ["estrato", "indice_vs_estrato"],
    ascending=[True, False]
)

desempenho_cluster_estrato.head(15)

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,patrimonio_mediano_cluster,patrimonio_medio_cluster,qtd_bens_mediana,macros_mediana,concentracao_mediana,taxa_cluster,candidatos_estrato,eleitos_estrato,taxa_estrato,indice_vs_estrato,delta_pp,participacao_no_estrato,participacao_eleitos_estrato
1,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,6548.510,16800.479130,2.0,1.0,1.000000,0.060870,5365,118,0.021994,2.767502,3.887516,0.128611,0.355932
6,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,49049.180,50664.183419,3.0,2.0,0.685239,0.035129,5365,118,0.021994,1.597170,1.313440,0.079590,0.127119
5,01_baixo,Baixo patrimônio - societário concentrado,411,12,22036.090,36036.876448,1.0,1.0,1.000000,0.029197,5365,118,0.021994,1.327477,0.720267,0.076608,0.101695
0,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,364,9,10000.000,22230.653681,1.0,1.0,1.000000,0.024725,5365,118,0.021994,1.124162,0.273087,0.067847,0.076271
3,01_baixo,Baixo patrimônio - imobiliário concentrado,718,15,60000.000,60348.946337,1.0,1.0,1.000000,0.020891,5365,118,0.021994,0.949849,-0.110304,0.133830,0.127119
7,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,22,30000.000,35954.069475,1.0,1.0,1.000000,0.010041,5365,118,0.021994,0.456529,-1.195333,0.408388,0.186441
2,01_baixo,Baixo patrimônio - imobiliário + veicular,306,2,74000.000,70245.165196,2.0,2.0,0.571706,0.006536,5365,118,0.021994,0.297164,-1.545846,0.057036,0.016949
4,01_baixo,Baixo patrimônio - outros ativos diversificado,258,1,7110.000,22986.005930,1.0,1.0,1.000000,0.003876,5365,118,0.021994,0.176225,-1.811844,0.048089,0.008475
14,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,170000.000,178927.462889,4.0,2.0,0.593739,0.133333,4186,187,0.044673,2.984670,8.866061,0.086001,0.256684
9,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,199896.720,197147.819031,4.0,2.0,0.751550,0.117647,4186,187,0.044673,2.633533,7.297434,0.069040,0.181818


In [37]:
# ============================================================
# 2.4 Classificação da performance dentro do estrato
# ============================================================

def classificar_vs_estrato(indice):
    if pd.isna(indice):
        return "indeterminado"
    elif indice >= 1.5:
        return "muito acima do estrato"
    elif indice >= 1.1:
        return "acima do estrato"
    elif indice > 0.9:
        return "próximo ao estrato"
    elif indice > 0.67:
        return "abaixo do estrato"
    else:
        return "muito abaixo do estrato"

desempenho_cluster_estrato["classificacao_vs_estrato"] = (
    desempenho_cluster_estrato["indice_vs_estrato"]
    .apply(classificar_vs_estrato)
)

desempenho_cluster_estrato[
    [
        "estrato",
        "perfil_cluster",
        "candidatos_cluster",
        "eleitos_cluster",
        "taxa_cluster",
        "taxa_estrato",
        "indice_vs_estrato",
        "delta_pp",
        "classificacao_vs_estrato"
    ]
].head(20)

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,taxa_cluster,taxa_estrato,indice_vs_estrato,delta_pp,classificacao_vs_estrato
1,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,0.060870,0.021994,2.767502,3.887516,muito acima do estrato
6,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,0.035129,0.021994,1.597170,1.313440,muito acima do estrato
5,01_baixo,Baixo patrimônio - societário concentrado,411,12,0.029197,0.021994,1.327477,0.720267,acima do estrato
0,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,364,9,0.024725,0.021994,1.124162,0.273087,acima do estrato
3,01_baixo,Baixo patrimônio - imobiliário concentrado,718,15,0.020891,0.021994,0.949849,-0.110304,próximo ao estrato
7,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,22,0.010041,0.021994,0.456529,-1.195333,muito abaixo do estrato
2,01_baixo,Baixo patrimônio - imobiliário + veicular,306,2,0.006536,0.021994,0.297164,-1.545846,muito abaixo do estrato
4,01_baixo,Baixo patrimônio - outros ativos diversificado,258,1,0.003876,0.021994,0.176225,-1.811844,muito abaixo do estrato
14,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,0.133333,0.044673,2.984670,8.866061,muito acima do estrato
9,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,0.117647,0.044673,2.633533,7.297434,muito acima do estrato


In [38]:
# ============================================================
# 2.5 Filtro mínimo para interpretação
# ============================================================
#
# Evita destacar clusters muito pequenos dentro do estrato.
# Como os clusters parecem estar aninhados em estratos, esse filtro
# funciona de modo parecido com o filtro mínimo do Eixo 1.
# ============================================================

MIN_CANDIDATOS_CLUSTER_ESTRATO = 30

desempenho_cluster_estrato_filtrado = (
    desempenho_cluster_estrato
    .loc[
        desempenho_cluster_estrato["candidatos_cluster"]
        >= MIN_CANDIDATOS_CLUSTER_ESTRATO
    ]
    .copy()
)

desempenho_cluster_estrato_filtrado = desempenho_cluster_estrato_filtrado.sort_values(
    ["estrato", "indice_vs_estrato"],
    ascending=[True, False]
)

desempenho_cluster_estrato_filtrado.head(20)

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,patrimonio_mediano_cluster,patrimonio_medio_cluster,qtd_bens_mediana,macros_mediana,concentracao_mediana,taxa_cluster,candidatos_estrato,eleitos_estrato,taxa_estrato,indice_vs_estrato,delta_pp,participacao_no_estrato,participacao_eleitos_estrato,classificacao_vs_estrato
1,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,6548.510,16800.479130,2.0,1.0,1.000000,0.060870,5365,118,0.021994,2.767502,3.887516,0.128611,0.355932,muito acima do estrato
6,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,49049.180,50664.183419,3.0,2.0,0.685239,0.035129,5365,118,0.021994,1.597170,1.313440,0.079590,0.127119,muito acima do estrato
5,01_baixo,Baixo patrimônio - societário concentrado,411,12,22036.090,36036.876448,1.0,1.0,1.000000,0.029197,5365,118,0.021994,1.327477,0.720267,0.076608,0.101695,acima do estrato
0,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,364,9,10000.000,22230.653681,1.0,1.0,1.000000,0.024725,5365,118,0.021994,1.124162,0.273087,0.067847,0.076271,acima do estrato
3,01_baixo,Baixo patrimônio - imobiliário concentrado,718,15,60000.000,60348.946337,1.0,1.0,1.000000,0.020891,5365,118,0.021994,0.949849,-0.110304,0.133830,0.127119,próximo ao estrato
7,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,22,30000.000,35954.069475,1.0,1.0,1.000000,0.010041,5365,118,0.021994,0.456529,-1.195333,0.408388,0.186441,muito abaixo do estrato
2,01_baixo,Baixo patrimônio - imobiliário + veicular,306,2,74000.000,70245.165196,2.0,2.0,0.571706,0.006536,5365,118,0.021994,0.297164,-1.545846,0.057036,0.016949,muito abaixo do estrato
4,01_baixo,Baixo patrimônio - outros ativos diversificado,258,1,7110.000,22986.005930,1.0,1.0,1.000000,0.003876,5365,118,0.021994,0.176225,-1.811844,0.048089,0.008475,muito abaixo do estrato
14,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,170000.000,178927.462889,4.0,2.0,0.593739,0.133333,4186,187,0.044673,2.984670,8.866061,0.086001,0.256684,muito acima do estrato
9,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,199896.720,197147.819031,4.0,2.0,0.751550,0.117647,4186,187,0.044673,2.633533,7.297434,0.069040,0.181818,muito acima do estrato


In [39]:
# ============================================================
# 2.6 Clusters que mais performam acima do próprio estrato
# ============================================================

top_sobreperformance_estrato = (
    desempenho_cluster_estrato_filtrado
    .sort_values(
        ["indice_vs_estrato", "delta_pp", "candidatos_cluster"],
        ascending=[False, False, False]
    )
    .head(15)
    .reset_index(drop=True)
)

top_sobreperformance_estrato[
    [
        "estrato",
        "perfil_cluster",
        "candidatos_cluster",
        "eleitos_cluster",
        "taxa_cluster",
        "taxa_estrato",
        "indice_vs_estrato",
        "delta_pp",
        "participacao_no_estrato",
        "participacao_eleitos_estrato"
    ]
]

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,taxa_cluster,taxa_estrato,indice_vs_estrato,delta_pp,participacao_no_estrato,participacao_eleitos_estrato
0,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,0.133333,0.044673,2.984670,8.866061,0.086001,0.256684
1,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,0.060870,0.021994,2.767502,3.887516,0.128611,0.355932
2,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,0.117647,0.044673,2.633533,7.297434,0.069040,0.181818
3,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,392,77,0.196429,0.097075,2.023463,9.935312,0.096338,0.194937
4,03_medio_alto,Patrimônio médio-alto - veicular diversificado,398,74,0.185930,0.097075,1.915311,8.885420,0.097813,0.187342
5,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,0.035129,0.021994,1.597170,1.313440,0.079590,0.127119
6,04_alto,Alto patrimônio - imobiliário diversificado co...,474,141,0.297468,0.194558,1.528946,10.291053,0.107483,0.164336
7,02_medio_baixo,Patrimônio médio-baixo - societário + veicular,326,21,0.064417,0.044673,1.441980,1.974446,0.077879,0.112299
8,02_medio_baixo,Patrimônio médio-baixo - imobiliário + veicular,443,27,0.060948,0.044673,1.364324,1.627536,0.105829,0.144385
9,01_baixo,Baixo patrimônio - societário concentrado,411,12,0.029197,0.021994,1.327477,0.720267,0.076608,0.101695


In [40]:
# ============================================================
# 2.7 Clusters que mais performam abaixo do próprio estrato
# ============================================================

bottom_sobreperformance_estrato = (
    desempenho_cluster_estrato_filtrado
    .sort_values(
        ["indice_vs_estrato", "delta_pp", "candidatos_cluster"],
        ascending=[True, True, False]
    )
    .head(15)
    .reset_index(drop=True)
)

bottom_sobreperformance_estrato[
    [
        "estrato",
        "perfil_cluster",
        "candidatos_cluster",
        "eleitos_cluster",
        "taxa_cluster",
        "taxa_estrato",
        "indice_vs_estrato",
        "delta_pp",
        "participacao_no_estrato",
        "participacao_eleitos_estrato"
    ]
]

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,taxa_cluster,taxa_estrato,indice_vs_estrato,delta_pp,participacao_no_estrato,participacao_eleitos_estrato
0,01_baixo,Baixo patrimônio - outros ativos diversificado,258,1,0.003876,0.021994,0.176225,-1.811844,0.048089,0.008475
1,01_baixo,Baixo patrimônio - imobiliário + veicular,306,2,0.006536,0.021994,0.297164,-1.545846,0.057036,0.016949
2,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1154,16,0.013865,0.044673,0.310364,-3.080790,0.275681,0.085561
3,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1033,19,0.018393,0.044673,0.411728,-2.627969,0.246775,0.101604
4,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,1152,50,0.043403,0.097075,0.447104,-5.367267,0.283116,0.126582
5,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,22,0.010041,0.021994,0.456529,-1.195333,0.408388,0.186441
6,04_alto,Alto patrimônio - imobiliário concentrado com ...,600,57,0.095000,0.194558,0.488287,-9.955782,0.136054,0.066434
7,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,670,42,0.062687,0.097075,0.645751,-3.438888,0.164660,0.106329
8,03_medio_alto,Patrimônio médio-alto - imobiliário + veicular...,427,33,0.077283,0.097075,0.796117,-1.979208,0.104940,0.083544
9,02_medio_baixo,Patrimônio médio-baixo - veicular concentrado,364,13,0.035714,0.044673,0.799465,-0.895843,0.086957,0.069519


In [41]:
# ============================================================
# 2.8 Melhores clusters dentro de cada estrato
# ============================================================

top_por_estrato = (
    desempenho_cluster_estrato_filtrado
    .sort_values(
        ["estrato", "indice_vs_estrato", "candidatos_cluster"],
        ascending=[True, False, False]
    )
    .groupby("estrato", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_por_estrato[
    [
        "estrato",
        "perfil_cluster",
        "candidatos_cluster",
        "eleitos_cluster",
        "taxa_cluster",
        "taxa_estrato",
        "indice_vs_estrato",
        "delta_pp"
    ]
]

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,taxa_cluster,taxa_estrato,indice_vs_estrato,delta_pp
0,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,0.060870,0.021994,2.767502,3.887516
1,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,0.035129,0.021994,1.597170,1.313440
2,01_baixo,Baixo patrimônio - societário concentrado,411,12,0.029197,0.021994,1.327477,0.720267
3,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,364,9,0.024725,0.021994,1.124162,0.273087
4,01_baixo,Baixo patrimônio - imobiliário concentrado,718,15,0.020891,0.021994,0.949849,-0.110304
5,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,0.133333,0.044673,2.984670,8.866061
6,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,0.117647,0.044673,2.633533,7.297434
7,02_medio_baixo,Patrimônio médio-baixo - societário + veicular,326,21,0.064417,0.044673,1.441980,1.974446
8,02_medio_baixo,Patrimônio médio-baixo - imobiliário + veicular,443,27,0.060948,0.044673,1.364324,1.627536
9,02_medio_baixo,Patrimônio médio-baixo - rural/agropecuário + ...,217,9,0.041475,0.044673,0.928411,-0.319806


In [42]:
# ============================================================
# 2.9 Piores clusters dentro de cada estrato
# ============================================================

bottom_por_estrato = (
    desempenho_cluster_estrato_filtrado
    .sort_values(
        ["estrato", "indice_vs_estrato", "candidatos_cluster"],
        ascending=[True, True, False]
    )
    .groupby("estrato", observed=True)
    .head(5)
    .reset_index(drop=True)
)

bottom_por_estrato[
    [
        "estrato",
        "perfil_cluster",
        "candidatos_cluster",
        "eleitos_cluster",
        "taxa_cluster",
        "taxa_estrato",
        "indice_vs_estrato",
        "delta_pp"
    ]
]

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,taxa_cluster,taxa_estrato,indice_vs_estrato,delta_pp
0,01_baixo,Baixo patrimônio - outros ativos diversificado,258,1,0.003876,0.021994,0.176225,-1.811844
1,01_baixo,Baixo patrimônio - imobiliário + veicular,306,2,0.006536,0.021994,0.297164,-1.545846
2,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,22,0.010041,0.021994,0.456529,-1.195333
3,01_baixo,Baixo patrimônio - imobiliário concentrado,718,15,0.020891,0.021994,0.949849,-0.110304
4,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,364,9,0.024725,0.021994,1.124162,0.273087
5,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1154,16,0.013865,0.044673,0.310364,-3.080790
6,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1033,19,0.018393,0.044673,0.411728,-2.627969
7,02_medio_baixo,Patrimônio médio-baixo - veicular concentrado,364,13,0.035714,0.044673,0.799465,-0.895843
8,02_medio_baixo,Patrimônio médio-baixo - rural/agropecuário + ...,217,9,0.041475,0.044673,0.928411,-0.319806
9,02_medio_baixo,Patrimônio médio-baixo - imobiliário + veicular,443,27,0.060948,0.044673,1.364324,1.627536


In [43]:
# ============================================================
# 2.10 Tabela-resumo por estrato
# Melhor e pior cluster de cada estrato
# ============================================================

melhor_cluster_estrato = (
    desempenho_cluster_estrato_filtrado
    .sort_values(
        ["estrato", "indice_vs_estrato", "candidatos_cluster"],
        ascending=[True, False, False]
    )
    .groupby("estrato", observed=True)
    .head(1)
    .rename(columns={
        "perfil_cluster": "melhor_cluster",
        "candidatos_cluster": "melhor_cluster_candidatos",
        "taxa_cluster": "melhor_cluster_taxa",
        "indice_vs_estrato": "melhor_cluster_indice",
        "delta_pp": "melhor_cluster_delta_pp"
    })
    [
        [
            "estrato",
            "melhor_cluster",
            "melhor_cluster_candidatos",
            "melhor_cluster_taxa",
            "melhor_cluster_indice",
            "melhor_cluster_delta_pp"
        ]
    ]
)

pior_cluster_estrato = (
    desempenho_cluster_estrato_filtrado
    .sort_values(
        ["estrato", "indice_vs_estrato", "candidatos_cluster"],
        ascending=[True, True, False]
    )
    .groupby("estrato", observed=True)
    .head(1)
    .rename(columns={
        "perfil_cluster": "pior_cluster",
        "candidatos_cluster": "pior_cluster_candidatos",
        "taxa_cluster": "pior_cluster_taxa",
        "indice_vs_estrato": "pior_cluster_indice",
        "delta_pp": "pior_cluster_delta_pp"
    })
    [
        [
            "estrato",
            "pior_cluster",
            "pior_cluster_candidatos",
            "pior_cluster_taxa",
            "pior_cluster_indice",
            "pior_cluster_delta_pp"
        ]
    ]
)

resumo_eixo2_por_estrato = (
    desempenho_estratos
    .merge(melhor_cluster_estrato, on="estrato", how="left")
    .merge(pior_cluster_estrato, on="estrato", how="left")
)

resumo_eixo2_por_estrato

,estrato,candidatos_estrato,eleitos_estrato,patrimonio_mediano_estrato,patrimonio_medio_estrato,taxa_estrato,melhor_cluster,melhor_cluster_candidatos,melhor_cluster_taxa,melhor_cluster_indice,melhor_cluster_delta_pp,pior_cluster,pior_cluster_candidatos,pior_cluster_taxa,pior_cluster_indice,pior_cluster_delta_pp
0,01_baixo,5365,118,3.200000e+04,3.833371e+04,0.021994,Baixo patrimônio - financeiro concentrado,690,0.060870,2.767502,3.887516,Baixo patrimônio - outros ativos diversificado,258,0.003876,0.176225,-1.811844
1,02_medio_baixo,4186,187,1.880000e+05,1.920508e+05,0.044673,Patrimônio médio-baixo - veicular + financeiro,360,0.133333,2.984670,8.866061,Patrimônio médio-baixo - imobiliário concentra...,1154,0.013865,0.310364,-3.080790
2,03_medio_alto,4069,395,4.657272e+05,4.879935e+05,0.097075,Patrimônio médio-alto - imobiliário concentrad...,392,0.196429,2.023463,9.935312,Patrimônio médio-alto - imobiliário concentrad...,1152,0.043403,0.447104,-5.367267
3,04_alto,4410,858,1.440935e+06,2.188159e+06,0.194558,Alto patrimônio - imobiliário diversificado co...,474,0.297468,1.528946,10.291053,Alto patrimônio - imobiliário concentrado com ...,600,0.095000,0.488287,-9.955782
4,05_ultra_alto,189,53,2.465718e+07,5.839267e+07,0.280423,Patrimônio ultra-alto - financeiro + societário,37,0.351351,1.252932,7.092807,Patrimônio ultra-alto - rural/agropecuário con...,46,0.304348,1.085316,2.392455


In [44]:
# ============================================================
# 2.11 Tabela final do Eixo 2
# ============================================================

tabela_eixo2 = (
    desempenho_cluster_estrato_filtrado
    [
        [
            "estrato",
            "perfil_cluster",
            "candidatos_cluster",
            "eleitos_cluster",
            "taxa_cluster",
            "candidatos_estrato",
            "eleitos_estrato",
            "taxa_estrato",
            "indice_vs_estrato",
            "delta_pp",
            "participacao_no_estrato",
            "participacao_eleitos_estrato",
            "patrimonio_mediano_cluster",
            "patrimonio_medio_cluster",
            "qtd_bens_mediana",
            "macros_mediana",
            "concentracao_mediana",
            "classificacao_vs_estrato"
        ]
    ]
    .sort_values(["estrato", "indice_vs_estrato"], ascending=[True, False])
    .copy()
)

tabela_eixo2.head(20)

,estrato,perfil_cluster,candidatos_cluster,eleitos_cluster,taxa_cluster,candidatos_estrato,eleitos_estrato,taxa_estrato,indice_vs_estrato,delta_pp,participacao_no_estrato,participacao_eleitos_estrato,patrimonio_mediano_cluster,patrimonio_medio_cluster,qtd_bens_mediana,macros_mediana,concentracao_mediana,classificacao_vs_estrato
1,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,0.060870,5365,118,0.021994,2.767502,3.887516,0.128611,0.355932,6548.510,16800.479130,2.0,1.0,1.000000,muito acima do estrato
6,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,0.035129,5365,118,0.021994,1.597170,1.313440,0.079590,0.127119,49049.180,50664.183419,3.0,2.0,0.685239,muito acima do estrato
5,01_baixo,Baixo patrimônio - societário concentrado,411,12,0.029197,5365,118,0.021994,1.327477,0.720267,0.076608,0.101695,22036.090,36036.876448,1.0,1.0,1.000000,acima do estrato
0,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,364,9,0.024725,5365,118,0.021994,1.124162,0.273087,0.067847,0.076271,10000.000,22230.653681,1.0,1.0,1.000000,acima do estrato
3,01_baixo,Baixo patrimônio - imobiliário concentrado,718,15,0.020891,5365,118,0.021994,0.949849,-0.110304,0.133830,0.127119,60000.000,60348.946337,1.0,1.0,1.000000,próximo ao estrato
7,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191,22,0.010041,5365,118,0.021994,0.456529,-1.195333,0.408388,0.186441,30000.000,35954.069475,1.0,1.0,1.000000,muito abaixo do estrato
2,01_baixo,Baixo patrimônio - imobiliário + veicular,306,2,0.006536,5365,118,0.021994,0.297164,-1.545846,0.057036,0.016949,74000.000,70245.165196,2.0,2.0,0.571706,muito abaixo do estrato
4,01_baixo,Baixo patrimônio - outros ativos diversificado,258,1,0.003876,5365,118,0.021994,0.176225,-1.811844,0.048089,0.008475,7110.000,22986.005930,1.0,1.0,1.000000,muito abaixo do estrato
14,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,0.133333,4186,187,0.044673,2.984670,8.866061,0.086001,0.256684,170000.000,178927.462889,4.0,2.0,0.593739,muito acima do estrato
9,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,0.117647,4186,187,0.044673,2.633533,7.297434,0.069040,0.181818,199896.720,197147.819031,4.0,2.0,0.751550,muito acima do estrato


In [45]:
# ============================================================
# 2.12 Salvamento das tabelas do Eixo 2
# ============================================================

from pathlib import Path

PASTA_SAIDAS = Path("results")
PASTA_SAIDAS.mkdir(exist_ok=True)

desempenho_estratos.to_csv(
    PASTA_SAIDAS / "eixo2_taxa_eleicao_por_estrato.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo2.to_csv(
    PASTA_SAIDAS / "eixo2_desempenho_cluster_dentro_estrato.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

resumo_eixo2_por_estrato.to_csv(
    PASTA_SAIDAS / "eixo2_resumo_melhor_pior_cluster_por_estrato.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

In [46]:
# ============================================================
# 2.13 Configuração para gráficos do Eixo 2
# ============================================================

import plotly.express as px
import textwrap

PASTA_GRAFICOS = Path("plot")
PASTA_GRAFICOS.mkdir(exist_ok=True)

if "RENDERIZAR_NO_NOTEBOOK" not in globals():
    RENDERIZAR_NO_NOTEBOOK = False

def quebra_label(texto, largura=55):
    if pd.isna(texto):
        return texto
    return "<br>".join(textwrap.wrap(str(texto), width=largura))

In [47]:
# ============================================================
# 2.14 Gráfico — Índice de desempenho dos clusters vs estrato
# ============================================================

df_plot_eixo2 = tabela_eixo2.copy()

df_plot_eixo2["perfil_cluster_quebrado"] = (
    df_plot_eixo2["perfil_cluster"]
    .apply(lambda x: quebra_label(x, 60))
)

fig = px.bar(
    df_plot_eixo2.sort_values(["estrato", "indice_vs_estrato"]),
    x="indice_vs_estrato",
    y="perfil_cluster_quebrado",
    facet_col="estrato",
    facet_col_wrap=2,
    orientation="h",
    hover_data={
        "estrato": True,
        "perfil_cluster": True,
        "perfil_cluster_quebrado": False,
        "candidatos_cluster": True,
        "eleitos_cluster": True,
        "taxa_cluster": ":.2%",
        "taxa_estrato": ":.2%",
        "indice_vs_estrato": ":.2f",
        "delta_pp": ":.2f"
    },
    labels={
        "indice_vs_estrato": "Índice vs taxa do estrato",
        "perfil_cluster_quebrado": "Cluster"
    },
    title="Desempenho eleitoral dos clusters em relação à média do próprio estrato"
)

fig.add_vline(
    x=1,
    line_dash="dash",
    annotation_text="média do estrato",
    annotation_position="top"
)

fig.update_yaxes(matches=None, showticklabels=True)
fig.update_layout(
    height=1200,
    showlegend=False
)

fig.write_html(PASTA_GRAFICOS / "eixo2_indice_clusters_vs_estrato.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [48]:
# ============================================================
# 2.16 Gráfico — Taxa do cluster vs taxa do estrato
# ============================================================

df_scatter_eixo2 = tabela_eixo2.copy()

df_scatter_eixo2["perfil_cluster_quebrado"] = (
    df_scatter_eixo2["perfil_cluster"]
    .apply(lambda x: quebra_label(x, 60))
)

fig = px.scatter(
    df_scatter_eixo2,
    x="taxa_estrato",
    y="taxa_cluster",
    size="candidatos_cluster",
    color="estrato",
    hover_name="perfil_cluster_quebrado",
    hover_data={
        "perfil_cluster": True,
        "perfil_cluster_quebrado": False,
        "candidatos_cluster": True,
        "eleitos_cluster": True,
        "taxa_cluster": ":.2%",
        "taxa_estrato": ":.2%",
        "indice_vs_estrato": ":.2f",
        "delta_pp": ":.2f"
    },
    labels={
        "taxa_estrato": "Taxa média de eleição do estrato",
        "taxa_cluster": "Taxa de eleição do cluster",
        "estrato": "Estrato"
    },
    title="Taxa de eleição do cluster em relação à taxa média do estrato"
)

fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(tickformat=".0%")

fig.write_html(PASTA_GRAFICOS / "eixo2_taxa_cluster_vs_taxa_estrato.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [49]:
# ============================================================
# SEÇÃO 3 — EIXO 3
# Geografia eleitoral dos clusters
# ============================================================
#
# Pergunta central:
# Em quais estados determinados perfis patrimoniais performam melhor?
#
# Importante:
# A presença territorial é apenas contextual.
# O foco analítico é desempenho eleitoral.
#
# Métricas principais:
# taxa_cluster_uf = eleitos do cluster na UF / candidatos do cluster na UF
# taxa_uf = eleitos na UF / candidatos na UF
# indice_vs_uf = taxa_cluster_uf / taxa_uf
#
# Controle adicional:
# taxa_estrato_uf = eleitos do estrato na UF / candidatos do estrato na UF
# indice_vs_estrato_uf = taxa_cluster_uf / taxa_estrato_uf
# ============================================================

In [50]:
# ============================================================
# 3.0 Checagens iniciais
# ============================================================

if "df_integrado" not in globals():
    raise NameError(
        "df_integrado não existe. Execute antes a seção de integração."
    )

colunas_necessarias_eixo3 = [
    "SQ_CANDIDATO",
    "SG_UF",
    "estrato",
    "perfil_cluster",
    "eleito",
    "patrimonio_total"
]

colunas_faltantes_eixo3 = [
    col for col in colunas_necessarias_eixo3
    if col not in df_integrado.columns
]

if colunas_faltantes_eixo3:
    raise ValueError(f"Colunas faltantes para o Eixo 3: {colunas_faltantes_eixo3}")

df_geo = df_integrado.copy()

df_geo["SG_UF"] = (
    df_geo["SG_UF"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Remove candidaturas nacionais da análise estadual
REMOVER_BR = True

if REMOVER_BR:
    df_geo = df_geo.loc[df_geo["SG_UF"] != "BR"].copy()

print("Base geográfica:", df_geo.shape)
print("UFs:", df_geo["SG_UF"].nunique())
print("Clusters:", df_geo["perfil_cluster"].nunique())
print("Estratos:", df_geo["estrato"].nunique())
print("Taxa média:", round(df_geo["eleito"].mean(), 4))

Base geográfica: (18195, 41)
UFs: 27
Clusters: 35
Estratos: 5
Taxa média: 0.0885


In [51]:
# ============================================================
# 3.1 Desempenho eleitoral geral por UF
# ============================================================

desempenho_uf = (
    df_geo
    .groupby("SG_UF", observed=True)
    .agg(
        candidatos_uf=("SQ_CANDIDATO", "count"),
        eleitos_uf=("eleito", "sum"),
        clusters_presentes=("perfil_cluster", "nunique"),
        estratos_presentes=("estrato", "nunique"),
        patrimonio_mediano_uf=("patrimonio_total", "median")
    )
    .reset_index()
)

desempenho_uf["taxa_uf"] = (
    desempenho_uf["eleitos_uf"]
    / desempenho_uf["candidatos_uf"]
)

desempenho_uf = desempenho_uf.sort_values(
    "taxa_uf",
    ascending=False
)

desempenho_uf.head(15)

,SG_UF,candidatos_uf,eleitos_uf,clusters_presentes,estratos_presentes,patrimonio_mediano_uf,taxa_uf
16,PI,262,43,34,5,199000.000,0.164122
1,AL,239,39,31,5,269000.000,0.163180
15,PE,572,72,32,5,229665.500,0.125874
14,PB,413,51,32,5,265000.000,0.123487
5,CE,611,71,33,5,257000.000,0.116203
0,AC,281,32,30,5,230000.000,0.113879
9,MA,575,63,34,5,315500.000,0.109565
3,AP,267,28,28,5,233000.000,0.104869
13,PA,556,58,34,5,300000.000,0.104317
4,BA,999,102,33,5,243011.000,0.102102


In [52]:
# ============================================================
# 3.2 Presença dos clusters por UF
# Apenas contextual
# ============================================================

presenca_cluster_uf = (
    df_geo
    .groupby(["perfil_cluster", "SG_UF"], observed=True)
    .agg(
        candidatos_cluster_uf=("SQ_CANDIDATO", "count")
    )
    .reset_index()
)

total_cluster = (
    df_geo
    .groupby("perfil_cluster", observed=True)
    .agg(
        candidatos_cluster_total=("SQ_CANDIDATO", "count")
    )
    .reset_index()
)

total_uf = (
    df_geo
    .groupby("SG_UF", observed=True)
    .agg(
        candidatos_uf=("SQ_CANDIDATO", "count")
    )
    .reset_index()
)

presenca_cluster_uf = (
    presenca_cluster_uf
    .merge(total_cluster, on="perfil_cluster", how="left")
    .merge(total_uf, on="SG_UF", how="left")
)

presenca_cluster_uf["perc_cluster_na_uf"] = (
    presenca_cluster_uf["candidatos_cluster_uf"]
    / presenca_cluster_uf["candidatos_cluster_total"]
)

presenca_cluster_uf["perc_uf_no_cluster"] = (
    presenca_cluster_uf["candidatos_cluster_uf"]
    / presenca_cluster_uf["candidatos_uf"]
)

presenca_cluster_uf = presenca_cluster_uf.sort_values(
    ["perfil_cluster", "candidatos_cluster_uf"],
    ascending=[True, False]
)

presenca_cluster_uf.head(20)

,perfil_cluster,SG_UF,candidatos_cluster_uf,candidatos_cluster_total,candidatos_uf,perc_cluster_na_uf,perc_uf_no_cluster
25,Alto patrimônio - imobiliário + financeiro,SP,139,674,2528,0.206231,0.054984
18,Alto patrimônio - imobiliário + financeiro,RJ,53,674,1432,0.078635,0.037011
10,Alto patrimônio - imobiliário + financeiro,MG,49,674,1668,0.072700,0.029376
17,Alto patrimônio - imobiliário + financeiro,PR,49,674,1154,0.072700,0.042461
22,Alto patrimônio - imobiliário + financeiro,RS,43,674,975,0.063798,0.044103
23,Alto patrimônio - imobiliário + financeiro,SC,34,674,752,0.050445,0.045213
6,Alto patrimônio - imobiliário + financeiro,DF,31,674,606,0.045994,0.051155
4,Alto patrimônio - imobiliário + financeiro,BA,26,674,999,0.038576,0.026026
5,Alto patrimônio - imobiliário + financeiro,CE,25,674,611,0.037092,0.040917
9,Alto patrimônio - imobiliário + financeiro,MA,24,674,575,0.035608,0.041739


In [53]:
# ============================================================
# 3.3 Taxa de eleição por cluster e UF
# ============================================================

desempenho_cluster_uf = (
    df_geo
    .groupby(["perfil_cluster", "SG_UF"], observed=True)
    .agg(
        candidatos_cluster_uf=("SQ_CANDIDATO", "count"),
        eleitos_cluster_uf=("eleito", "sum"),
        patrimonio_mediano_cluster_uf=("patrimonio_total", "median"),
        estratos_presentes=("estrato", "nunique")
    )
    .reset_index()
)

desempenho_cluster_uf["taxa_cluster_uf"] = (
    desempenho_cluster_uf["eleitos_cluster_uf"]
    / desempenho_cluster_uf["candidatos_cluster_uf"]
)

desempenho_cluster_uf = desempenho_cluster_uf.merge(
    desempenho_uf[
        [
            "SG_UF",
            "candidatos_uf",
            "eleitos_uf",
            "taxa_uf"
        ]
    ],
    on="SG_UF",
    how="left"
)

desempenho_cluster_uf["indice_vs_uf"] = np.where(
    desempenho_cluster_uf["taxa_uf"] > 0,
    desempenho_cluster_uf["taxa_cluster_uf"] / desempenho_cluster_uf["taxa_uf"],
    np.nan
)

desempenho_cluster_uf["delta_pp_vs_uf"] = (
    desempenho_cluster_uf["taxa_cluster_uf"]
    - desempenho_cluster_uf["taxa_uf"]
) * 100

desempenho_cluster_uf["participacao_candidatos_uf"] = (
    desempenho_cluster_uf["candidatos_cluster_uf"]
    / desempenho_cluster_uf["candidatos_uf"]
)

desempenho_cluster_uf["participacao_eleitos_uf"] = np.where(
    desempenho_cluster_uf["eleitos_uf"] > 0,
    desempenho_cluster_uf["eleitos_cluster_uf"]
    / desempenho_cluster_uf["eleitos_uf"],
    np.nan
)

desempenho_cluster_uf = desempenho_cluster_uf.sort_values(
    ["indice_vs_uf", "delta_pp_vs_uf"],
    ascending=[False, False]
)

desempenho_cluster_uf.head(20)

,perfil_cluster,SG_UF,candidatos_cluster_uf,eleitos_cluster_uf,patrimonio_mediano_cluster_uf,estratos_presentes,taxa_cluster_uf,candidatos_uf,eleitos_uf,taxa_uf,indice_vs_uf,delta_pp_vs_uf,participacao_candidatos_uf,participacao_eleitos_uf
808,Patrimônio ultra-alto - créditos e direitos di...,MS,1,1,1.663284e+07,1,1.000000,394,36,0.091371,10.944444,90.862944,0.002538,0.027778
863,Patrimônio ultra-alto - rural/agropecuário con...,RS,2,2,2.288722e+07,1,1.000000,975,90,0.092308,10.833333,90.769231,0.002051,0.022222
823,Patrimônio ultra-alto - financeiro + societário,MT,3,3,2.284046e+07,1,1.000000,380,37,0.097368,10.270270,90.263158,0.007895,0.081081
830,Patrimônio ultra-alto - financeiro + societário,RN,2,2,3.935703e+07,1,1.000000,356,35,0.098315,10.171429,90.168539,0.005618,0.057143
832,Patrimônio ultra-alto - financeiro + societário,RR,1,1,5.056014e+07,1,1.000000,343,34,0.099125,10.088235,90.087464,0.002915,0.029412
852,Patrimônio ultra-alto - rural/agropecuário con...,BA,1,1,1.591834e+07,1,1.000000,999,102,0.102102,9.794118,89.789790,0.001001,0.009804
874,Patrimônio ultra-alto - societário concentrado,PA,1,1,1.375645e+07,1,1.000000,556,58,0.104317,9.586207,89.568345,0.001799,0.017241
821,Patrimônio ultra-alto - financeiro + societário,MA,1,1,1.576241e+07,1,1.000000,575,63,0.109565,9.126984,89.043478,0.001739,0.015873
839,Patrimônio ultra-alto - imobiliário + outros a...,MA,1,1,2.540680e+07,1,1.000000,575,63,0.109565,9.126984,89.043478,0.001739,0.015873
870,Patrimônio ultra-alto - societário concentrado,MA,1,1,2.023095e+07,1,1.000000,575,63,0.109565,9.126984,89.043478,0.001739,0.015873


In [54]:
# ============================================================
# 3.4 Filtro mínimo para interpretação cluster x UF
# ============================================================
#
# Filtro mais conservador para evitar sobreinterpretação
# de combinações pequenas entre cluster e UF.
# ============================================================

MIN_CANDIDATOS_CLUSTER_UF = 25
MIN_CANDIDATOS_UF = 100
MIN_ELEITOS_UF = 5

desempenho_cluster_uf_filtrado = (
    desempenho_cluster_uf
    .loc[
        (desempenho_cluster_uf["candidatos_cluster_uf"] >= MIN_CANDIDATOS_CLUSTER_UF)
        & (desempenho_cluster_uf["candidatos_uf"] >= MIN_CANDIDATOS_UF)
        & (desempenho_cluster_uf["eleitos_uf"] >= MIN_ELEITOS_UF)
        & (desempenho_cluster_uf["taxa_uf"] > 0)
    ]
    .copy()
)

desempenho_cluster_uf_filtrado = desempenho_cluster_uf_filtrado.sort_values(
    ["indice_vs_uf", "delta_pp_vs_uf", "candidatos_cluster_uf"],
    ascending=[False, False, False]
)

desempenho_cluster_uf_filtrado.head(20)

,perfil_cluster,SG_UF,candidatos_cluster_uf,eleitos_cluster_uf,patrimonio_mediano_cluster_uf,estratos_presentes,taxa_cluster_uf,candidatos_uf,eleitos_uf,taxa_uf,indice_vs_uf,delta_pp_vs_uf,participacao_candidatos_uf,participacao_eleitos_uf
179,Alto patrimônio - imobiliário diversificado co...,PR,44,20,2419047.095,1,0.454545,1154,86,0.074523,6.099366,38.002206,0.038128,0.232558
166,Alto patrimônio - imobiliário diversificado co...,BA,26,14,3025637.760,1,0.538462,999,102,0.102102,5.273756,43.635944,0.026026,0.137255
172,Alto patrimônio - imobiliário diversificado co...,MG,42,14,2372432.580,1,0.333333,1668,133,0.079736,4.180451,25.359712,0.025180,0.105263
139,Alto patrimônio - imobiliário diversificado co...,BA,38,16,2075178.010,1,0.421053,999,102,0.102102,4.123839,31.895053,0.038038,0.156863
143,Alto patrimônio - imobiliário diversificado co...,GO,26,8,2486466.020,1,0.307692,766,60,0.078329,3.928205,22.936333,0.033943,0.133333
185,Alto patrimônio - imobiliário diversificado co...,SC,31,9,2894850.350,1,0.290323,752,56,0.074468,3.898618,21.585450,0.041223,0.160714
170,Alto patrimônio - imobiliário diversificado co...,GO,29,8,3086048.410,1,0.275862,766,60,0.078329,3.521839,19.753309,0.037859,0.133333
59,Alto patrimônio - imobiliário + societário com...,CE,38,15,1633003.775,1,0.394737,611,71,0.116203,3.396961,27.853390,0.062193,0.211268
768,Patrimônio médio-baixo - veicular + financeiro,RJ,38,10,164691.555,1,0.263158,1432,116,0.081006,3.248639,18.215231,0.026536,0.086207
71,Alto patrimônio - imobiliário + societário com...,PR,68,16,1228427.865,1,0.235294,1154,86,0.074523,3.157319,16.077072,0.058925,0.186047


In [55]:
# ============================================================
# 3.5 Maiores sobreperformances por UF
# ============================================================

top_sobreperformance_uf = (
    desempenho_cluster_uf_filtrado
    .sort_values(
        ["indice_vs_uf", "delta_pp_vs_uf", "candidatos_cluster_uf"],
        ascending=[False, False, False]
    )
    .head(30)
    .reset_index(drop=True)
)

top_sobreperformance_uf[
    [
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_uf",
        "indice_vs_uf",
        "delta_pp_vs_uf",
        "participacao_candidatos_uf",
        "participacao_eleitos_uf"
    ]
]

,SG_UF,perfil_cluster,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_uf,indice_vs_uf,delta_pp_vs_uf,participacao_candidatos_uf,participacao_eleitos_uf
0,PR,Alto patrimônio - imobiliário diversificado co...,44,20,0.454545,0.074523,6.099366,38.002206,0.038128,0.232558
1,BA,Alto patrimônio - imobiliário diversificado co...,26,14,0.538462,0.102102,5.273756,43.635944,0.026026,0.137255
2,MG,Alto patrimônio - imobiliário diversificado co...,42,14,0.333333,0.079736,4.180451,25.359712,0.025180,0.105263
3,BA,Alto patrimônio - imobiliário diversificado co...,38,16,0.421053,0.102102,4.123839,31.895053,0.038038,0.156863
4,GO,Alto patrimônio - imobiliário diversificado co...,26,8,0.307692,0.078329,3.928205,22.936333,0.033943,0.133333
5,SC,Alto patrimônio - imobiliário diversificado co...,31,9,0.290323,0.074468,3.898618,21.585450,0.041223,0.160714
6,GO,Alto patrimônio - imobiliário diversificado co...,29,8,0.275862,0.078329,3.521839,19.753309,0.037859,0.133333
7,CE,Alto patrimônio - imobiliário + societário com...,38,15,0.394737,0.116203,3.396961,27.853390,0.062193,0.211268
8,RJ,Patrimônio médio-baixo - veicular + financeiro,38,10,0.263158,0.081006,3.248639,18.215231,0.026536,0.086207
9,PR,Alto patrimônio - imobiliário + societário com...,68,16,0.235294,0.074523,3.157319,16.077072,0.058925,0.186047


In [56]:
# ============================================================
# 3.5 Maiores sobreperformances por UF
# ============================================================

top_sobreperformance_uf = (
    desempenho_cluster_uf_filtrado
    .sort_values(
        ["indice_vs_uf", "delta_pp_vs_uf", "candidatos_cluster_uf"],
        ascending=[False, False, False]
    )
    .head(30)
    .reset_index(drop=True)
)

top_sobreperformance_uf[
    [
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_uf",
        "indice_vs_uf",
        "delta_pp_vs_uf",
        "participacao_candidatos_uf",
        "participacao_eleitos_uf"
    ]
]

,SG_UF,perfil_cluster,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_uf,indice_vs_uf,delta_pp_vs_uf,participacao_candidatos_uf,participacao_eleitos_uf
0,PR,Alto patrimônio - imobiliário diversificado co...,44,20,0.454545,0.074523,6.099366,38.002206,0.038128,0.232558
1,BA,Alto patrimônio - imobiliário diversificado co...,26,14,0.538462,0.102102,5.273756,43.635944,0.026026,0.137255
2,MG,Alto patrimônio - imobiliário diversificado co...,42,14,0.333333,0.079736,4.180451,25.359712,0.025180,0.105263
3,BA,Alto patrimônio - imobiliário diversificado co...,38,16,0.421053,0.102102,4.123839,31.895053,0.038038,0.156863
4,GO,Alto patrimônio - imobiliário diversificado co...,26,8,0.307692,0.078329,3.928205,22.936333,0.033943,0.133333
5,SC,Alto patrimônio - imobiliário diversificado co...,31,9,0.290323,0.074468,3.898618,21.585450,0.041223,0.160714
6,GO,Alto patrimônio - imobiliário diversificado co...,29,8,0.275862,0.078329,3.521839,19.753309,0.037859,0.133333
7,CE,Alto patrimônio - imobiliário + societário com...,38,15,0.394737,0.116203,3.396961,27.853390,0.062193,0.211268
8,RJ,Patrimônio médio-baixo - veicular + financeiro,38,10,0.263158,0.081006,3.248639,18.215231,0.026536,0.086207
9,PR,Alto patrimônio - imobiliário + societário com...,68,16,0.235294,0.074523,3.157319,16.077072,0.058925,0.186047


In [57]:
# ============================================================
# 3.6 Maiores subperformances por UF
# ============================================================

bottom_sobreperformance_uf = (
    desempenho_cluster_uf_filtrado
    .sort_values(
        ["indice_vs_uf", "delta_pp_vs_uf", "candidatos_cluster_uf"],
        ascending=[True, True, False]
    )
    .head(30)
    .reset_index(drop=True)
)

bottom_sobreperformance_uf[
    [
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_uf",
        "indice_vs_uf",
        "delta_pp_vs_uf",
        "participacao_candidatos_uf",
        "participacao_eleitos_uf"
    ]
]

,SG_UF,perfil_cluster,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_uf,indice_vs_uf,delta_pp_vs_uf,participacao_candidatos_uf,participacao_eleitos_uf
0,PI,Baixo patrimônio - veicular concentrado com ru...,38,0,0.0,0.164122,0.0,-16.412214,0.145038,0.0
1,AL,Baixo patrimônio - veicular concentrado com ru...,33,0,0.0,0.163180,0.0,-16.317992,0.138075,0.0
2,PE,Baixo patrimônio - veicular concentrado com ru...,94,0,0.0,0.125874,0.0,-12.587413,0.164336,0.0
3,PB,Baixo patrimônio - veicular concentrado com ru...,51,0,0.0,0.123487,0.0,-12.348668,0.123487,0.0
4,PB,Patrimônio médio-alto - imobiliário concentrad...,31,0,0.0,0.123487,0.0,-12.348668,0.075061,0.0
5,CE,Baixo patrimônio - veicular concentrado com ru...,95,0,0.0,0.116203,0.0,-11.620295,0.155483,0.0
6,CE,Baixo patrimônio - imobiliário concentrado,26,0,0.0,0.116203,0.0,-11.620295,0.042553,0.0
7,AC,Baixo patrimônio - veicular concentrado com ru...,51,0,0.0,0.113879,0.0,-11.387900,0.181495,0.0
8,AC,Patrimônio médio-baixo - imobiliário concentra...,27,0,0.0,0.113879,0.0,-11.387900,0.096085,0.0
9,MA,Baixo patrimônio - veicular concentrado com ru...,60,0,0.0,0.109565,0.0,-10.956522,0.104348,0.0


In [58]:
# ============================================================
# 3.7 Melhores UFs para cada cluster
# ============================================================

top_ufs_por_cluster = (
    desempenho_cluster_uf_filtrado
    .sort_values(
        ["perfil_cluster", "indice_vs_uf", "delta_pp_vs_uf"],
        ascending=[True, False, False]
    )
    .groupby("perfil_cluster", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_ufs_por_cluster[
    [
        "perfil_cluster",
        "SG_UF",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_uf",
        "indice_vs_uf",
        "delta_pp_vs_uf"
    ]
].head(30)

,perfil_cluster,SG_UF,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_uf,indice_vs_uf,delta_pp_vs_uf
0,Alto patrimônio - imobiliário + financeiro,RJ,53,13,0.245283,0.081006,3.027977,16.427743
1,Alto patrimônio - imobiliário + financeiro,BA,26,8,0.307692,0.102102,3.013575,20.559021
2,Alto patrimônio - imobiliário + financeiro,SP,139,26,0.187050,0.063291,2.955396,12.375922
3,Alto patrimônio - imobiliário + financeiro,MG,49,11,0.224490,0.079736,2.815406,14.475358
4,Alto patrimônio - imobiliário + financeiro,SC,34,7,0.205882,0.074468,2.764706,13.141427
5,Alto patrimônio - imobiliário + rural/agropecu...,TO,37,10,0.270270,0.087591,3.085586,18.267903
6,Alto patrimônio - imobiliário + rural/agropecu...,BA,42,13,0.309524,0.102102,3.031513,20.742171
7,Alto patrimônio - imobiliário + rural/agropecu...,RS,25,6,0.240000,0.092308,2.600000,14.769231
8,Alto patrimônio - imobiliário + rural/agropecu...,MA,28,5,0.178571,0.109565,1.629819,6.900621
9,Alto patrimônio - imobiliário + rural/agropecu...,GO,47,6,0.127660,0.078329,1.629787,4.933059


In [59]:
# ============================================================
# 3.8 Clusters mais fortes dentro de cada UF
# ============================================================

top_clusters_por_uf = (
    desempenho_cluster_uf_filtrado
    .sort_values(
        ["SG_UF", "indice_vs_uf", "delta_pp_vs_uf"],
        ascending=[True, False, False]
    )
    .groupby("SG_UF", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_clusters_por_uf[
    [
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_uf",
        "indice_vs_uf",
        "delta_pp_vs_uf"
    ]
].head(40)

,SG_UF,perfil_cluster,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_uf,indice_vs_uf,delta_pp_vs_uf
0,AC,Baixo patrimônio - veicular concentrado com ru...,51,0,0.000000,0.113879,0.000000,-11.387900
1,AC,Patrimônio médio-baixo - imobiliário concentra...,27,0,0.000000,0.113879,0.000000,-11.387900
2,AL,Baixo patrimônio - veicular concentrado com ru...,33,0,0.000000,0.163180,0.000000,-16.317992
3,AM,Alto patrimônio - imobiliário + societário com...,35,8,0.228571,0.085427,2.675630,14.314429
4,AM,Baixo patrimônio - dinheiro em espécie concent...,43,0,0.000000,0.085427,0.000000,-8.542714
5,AM,Baixo patrimônio - veicular concentrado com ru...,37,0,0.000000,0.085427,0.000000,-8.542714
6,AM,Patrimônio médio-baixo - imobiliário concentra...,33,0,0.000000,0.085427,0.000000,-8.542714
7,AM,Patrimônio médio-baixo - imobiliário concentra...,25,0,0.000000,0.085427,0.000000,-8.542714
8,AP,Patrimônio médio-baixo - imobiliário concentra...,25,4,0.160000,0.104869,1.525714,5.513109
9,AP,Baixo patrimônio - veicular concentrado com ru...,35,3,0.085714,0.104869,0.817347,-1.915463


In [60]:
# ============================================================
# 3.9 Desempenho por estrato e UF
# ============================================================

desempenho_estrato_uf = (
    df_geo
    .groupby(["estrato", "SG_UF"], observed=True)
    .agg(
        candidatos_estrato_uf=("SQ_CANDIDATO", "count"),
        eleitos_estrato_uf=("eleito", "sum"),
        patrimonio_mediano_estrato_uf=("patrimonio_total", "median")
    )
    .reset_index()
)

desempenho_estrato_uf["taxa_estrato_uf"] = (
    desempenho_estrato_uf["eleitos_estrato_uf"]
    / desempenho_estrato_uf["candidatos_estrato_uf"]
)

desempenho_estrato_uf = desempenho_estrato_uf.sort_values(
    ["estrato", "SG_UF"]
)

desempenho_estrato_uf.head(20)

,estrato,SG_UF,candidatos_estrato_uf,eleitos_estrato_uf,patrimonio_mediano_estrato_uf,taxa_estrato_uf
0,01_baixo,AC,83,5,38000.000,0.060241
1,01_baixo,AL,75,4,40000.000,0.053333
2,01_baixo,AM,136,2,14853.940,0.014706
3,01_baixo,AP,75,5,54200.000,0.066667
4,01_baixo,BA,339,5,32457.000,0.014749
5,01_baixo,CE,190,4,29975.795,0.021053
6,01_baixo,DF,152,3,37500.000,0.019737
7,01_baixo,ES,148,6,46936.970,0.040541
8,01_baixo,GO,190,3,35647.500,0.015789
9,01_baixo,MA,144,1,45000.000,0.006944


In [61]:
# ============================================================
# 3.10 Sobreperformance por UF dentro do estrato
# ============================================================

desempenho_cluster_estrato_uf = (
    df_geo
    .groupby(["estrato", "perfil_cluster", "SG_UF"], observed=True)
    .agg(
        candidatos_cluster_uf=("SQ_CANDIDATO", "count"),
        eleitos_cluster_uf=("eleito", "sum"),
        patrimonio_mediano_cluster_uf=("patrimonio_total", "median")
    )
    .reset_index()
)

desempenho_cluster_estrato_uf["taxa_cluster_uf"] = (
    desempenho_cluster_estrato_uf["eleitos_cluster_uf"]
    / desempenho_cluster_estrato_uf["candidatos_cluster_uf"]
)

desempenho_cluster_estrato_uf = desempenho_cluster_estrato_uf.merge(
    desempenho_estrato_uf[
        [
            "estrato",
            "SG_UF",
            "candidatos_estrato_uf",
            "eleitos_estrato_uf",
            "taxa_estrato_uf"
        ]
    ],
    on=["estrato", "SG_UF"],
    how="left"
)

desempenho_cluster_estrato_uf["indice_vs_estrato_uf"] = np.where(
    desempenho_cluster_estrato_uf["taxa_estrato_uf"] > 0,
    desempenho_cluster_estrato_uf["taxa_cluster_uf"]
    / desempenho_cluster_estrato_uf["taxa_estrato_uf"],
    np.nan
)

desempenho_cluster_estrato_uf["delta_pp_vs_estrato_uf"] = (
    desempenho_cluster_estrato_uf["taxa_cluster_uf"]
    - desempenho_cluster_estrato_uf["taxa_estrato_uf"]
) * 100

desempenho_cluster_estrato_uf["participacao_candidatos_estrato_uf"] = (
    desempenho_cluster_estrato_uf["candidatos_cluster_uf"]
    / desempenho_cluster_estrato_uf["candidatos_estrato_uf"]
)

desempenho_cluster_estrato_uf["participacao_eleitos_estrato_uf"] = np.where(
    desempenho_cluster_estrato_uf["eleitos_estrato_uf"] > 0,
    desempenho_cluster_estrato_uf["eleitos_cluster_uf"]
    / desempenho_cluster_estrato_uf["eleitos_estrato_uf"],
    np.nan
)

desempenho_cluster_estrato_uf = desempenho_cluster_estrato_uf.sort_values(
    ["indice_vs_estrato_uf", "delta_pp_vs_estrato_uf"],
    ascending=[False, False]
)

desempenho_cluster_estrato_uf.head(20)

,estrato,perfil_cluster,SG_UF,candidatos_cluster_uf,eleitos_cluster_uf,patrimonio_mediano_cluster_uf,taxa_cluster_uf,candidatos_estrato_uf,eleitos_estrato_uf,taxa_estrato_uf,indice_vs_estrato_uf,delta_pp_vs_estrato_uf,participacao_candidatos_estrato_uf,participacao_eleitos_estrato_uf
329,02_medio_baixo,Patrimônio médio-baixo - rural/agropecuário + ...,MS,2,1,189737.700,0.500000,103,2,0.019417,25.750000,48.058252,0.019417,0.500000
165,01_baixo,Baixo patrimônio - veicular concentrado com fi...,MA,6,1,27815.300,0.166667,144,1,0.006944,24.000000,15.972222,0.041667,1.000000
13,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,PE,2,1,32366.555,0.500000,193,5,0.025907,19.300000,47.409326,0.010363,0.200000
19,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,RR,6,1,45000.000,0.166667,111,1,0.009009,18.500000,15.765766,0.054054,1.000000
159,01_baixo,Baixo patrimônio - veicular concentrado com fi...,AM,4,1,54771.040,0.250000,136,2,0.014706,17.000000,23.529412,0.029412,0.500000
344,02_medio_baixo,Patrimônio médio-baixo - rural/agropecuário + ...,TO,6,1,164000.000,0.166667,98,1,0.010204,16.333333,15.646259,0.061224,1.000000
108,01_baixo,Baixo patrimônio - outros ativos diversificado,CE,3,1,72162.760,0.333333,190,4,0.021053,15.833333,31.228070,0.015789,0.250000
49,01_baixo,Baixo patrimônio - financeiro concentrado,TO,6,2,28131.925,0.333333,94,2,0.021277,15.666667,31.205674,0.063830,1.000000
251,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,PB,2,1,150352.570,0.500000,113,4,0.035398,14.125000,46.460177,0.017699,0.250000
385,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,PA,3,1,150333.700,0.333333,114,3,0.026316,12.666667,30.701754,0.026316,0.333333


In [62]:
# ============================================================
# 3.11 Filtro mínimo para UF dentro do estrato
# ============================================================
#
# Este é o filtro mais importante do Eixo 3.
# Ele evita destacar casos em que o índice fica artificialmente alto
# por causa de poucos candidatos ou poucos eleitos no estrato-UF.
# ============================================================

MIN_CANDIDATOS_CLUSTER_ESTRATO_UF = 25
MIN_CANDIDATOS_ESTRATO_UF = 50
MIN_ELEITOS_ESTRATO_UF = 3

desempenho_cluster_estrato_uf_filtrado = (
    desempenho_cluster_estrato_uf
    .loc[
        (
            desempenho_cluster_estrato_uf["candidatos_cluster_uf"]
            >= MIN_CANDIDATOS_CLUSTER_ESTRATO_UF
        )
        & (
            desempenho_cluster_estrato_uf["candidatos_estrato_uf"]
            >= MIN_CANDIDATOS_ESTRATO_UF
        )
        & (
            desempenho_cluster_estrato_uf["eleitos_estrato_uf"]
            >= MIN_ELEITOS_ESTRATO_UF
        )
        & (
            desempenho_cluster_estrato_uf["taxa_estrato_uf"] > 0
        )
    ]
    .copy()
)

desempenho_cluster_estrato_uf_filtrado = desempenho_cluster_estrato_uf_filtrado.sort_values(
    ["indice_vs_estrato_uf", "delta_pp_vs_estrato_uf", "candidatos_cluster_uf"],
    ascending=[False, False, False]
)

desempenho_cluster_estrato_uf_filtrado.head(20)

,estrato,perfil_cluster,SG_UF,candidatos_cluster_uf,eleitos_cluster_uf,patrimonio_mediano_cluster_uf,taxa_cluster_uf,candidatos_estrato_uf,eleitos_estrato_uf,taxa_estrato_uf,indice_vs_estrato_uf,delta_pp_vs_estrato_uf,participacao_candidatos_estrato_uf,participacao_eleitos_estrato_uf
390,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,RJ,38,10,164691.555,0.263158,300,18,0.060000,4.385965,20.315789,0.126667,0.555556
33,01_baixo,Baixo patrimônio - financeiro concentrado,MG,42,5,15000.000,0.119048,441,12,0.027211,4.375000,9.183673,0.095238,0.416667
262,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,SP,62,6,196270.495,0.096774,521,14,0.026871,3.601382,6.990279,0.119002,0.428571
409,02_medio_baixo,Patrimônio médio-baixo - veicular concentrado,MG,37,4,160000.000,0.108108,387,14,0.036176,2.988417,7.193240,0.095607,0.285714
389,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,PR,25,2,180454.080,0.080000,256,7,0.027344,2.925714,5.265625,0.097656,0.285714
48,01_baixo,Baixo patrimônio - financeiro concentrado,SP,140,7,5000.000,0.050000,745,13,0.017450,2.865385,3.255034,0.187919,0.538462
498,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,RJ,41,10,482549.040,0.243902,303,27,0.089109,2.737127,15.479353,0.135314,0.370370
27,01_baixo,Baixo patrimônio - financeiro concentrado,BA,52,2,5000.000,0.038462,339,5,0.014749,2.607692,2.371228,0.153392,0.400000
259,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,RS,25,5,191171.750,0.200000,237,19,0.080169,2.494737,11.983122,0.105485,0.263158
598,03_medio_alto,Patrimônio médio-alto - veicular diversificado,MG,25,5,430000.000,0.200000,401,35,0.087282,2.291429,11.271820,0.062344,0.142857


In [63]:
# ============================================================
# 3.12 Maiores sobreperformances por UF dentro do estrato
# ============================================================

top_sobreperformance_estrato_uf = (
    desempenho_cluster_estrato_uf_filtrado
    .sort_values(
        ["indice_vs_estrato_uf", "delta_pp_vs_estrato_uf", "candidatos_cluster_uf"],
        ascending=[False, False, False]
    )
    .head(30)
    .reset_index(drop=True)
)

top_sobreperformance_estrato_uf[
    [
        "estrato",
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_estrato_uf",
        "indice_vs_estrato_uf",
        "delta_pp_vs_estrato_uf",
        "participacao_candidatos_estrato_uf",
        "participacao_eleitos_estrato_uf"
    ]
]

,estrato,SG_UF,perfil_cluster,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_estrato_uf,indice_vs_estrato_uf,delta_pp_vs_estrato_uf,participacao_candidatos_estrato_uf,participacao_eleitos_estrato_uf
0,02_medio_baixo,RJ,Patrimônio médio-baixo - veicular + financeiro,38,10,0.263158,0.060000,4.385965,20.315789,0.126667,0.555556
1,01_baixo,MG,Baixo patrimônio - financeiro concentrado,42,5,0.119048,0.027211,4.375000,9.183673,0.095238,0.416667
2,02_medio_baixo,SP,Patrimônio médio-baixo - imobiliário concentra...,62,6,0.096774,0.026871,3.601382,6.990279,0.119002,0.428571
3,02_medio_baixo,MG,Patrimônio médio-baixo - veicular concentrado,37,4,0.108108,0.036176,2.988417,7.193240,0.095607,0.285714
4,02_medio_baixo,PR,Patrimônio médio-baixo - veicular + financeiro,25,2,0.080000,0.027344,2.925714,5.265625,0.097656,0.285714
5,01_baixo,SP,Baixo patrimônio - financeiro concentrado,140,7,0.050000,0.017450,2.865385,3.255034,0.187919,0.538462
6,03_medio_alto,RJ,Patrimônio médio-alto - imobiliário concentrad...,41,10,0.243902,0.089109,2.737127,15.479353,0.135314,0.370370
7,01_baixo,BA,Baixo patrimônio - financeiro concentrado,52,2,0.038462,0.014749,2.607692,2.371228,0.153392,0.400000
8,02_medio_baixo,RS,Patrimônio médio-baixo - imobiliário concentra...,25,5,0.200000,0.080169,2.494737,11.983122,0.105485,0.263158
9,03_medio_alto,MG,Patrimônio médio-alto - veicular diversificado,25,5,0.200000,0.087282,2.291429,11.271820,0.062344,0.142857


In [64]:
# ============================================================
# 3.13 Maiores subperformances por UF dentro do estrato
# ============================================================

bottom_sobreperformance_estrato_uf = (
    desempenho_cluster_estrato_uf_filtrado
    .sort_values(
        ["indice_vs_estrato_uf", "delta_pp_vs_estrato_uf", "candidatos_cluster_uf"],
        ascending=[True, True, False]
    )
    .head(30)
    .reset_index(drop=True)
)

bottom_sobreperformance_estrato_uf[
    [
        "estrato",
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_estrato_uf",
        "indice_vs_estrato_uf",
        "delta_pp_vs_estrato_uf",
        "participacao_candidatos_estrato_uf",
        "participacao_eleitos_estrato_uf"
    ]
]

,estrato,SG_UF,perfil_cluster,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_estrato_uf,indice_vs_estrato_uf,delta_pp_vs_estrato_uf,participacao_candidatos_estrato_uf,participacao_eleitos_estrato_uf
0,04_alto,RJ,Alto patrimônio - imobiliário concentrado com ...,40,0,0.0,0.161812,0.0,-16.181230,0.129450,0.0
1,03_medio_alto,PB,Patrimônio médio-alto - imobiliário concentrad...,31,0,0.0,0.142857,0.0,-14.285714,0.340659,0.0
2,03_medio_alto,MT,Patrimônio médio-alto - imobiliário concentrad...,32,0,0.0,0.090909,0.0,-9.090909,0.323232,0.0
3,03_medio_alto,GO,Patrimônio médio-alto - imobiliário concentrad...,48,0,0.0,0.090361,0.0,-9.036145,0.289157,0.0
4,03_medio_alto,RJ,Patrimônio médio-alto - imobiliário concentrad...,76,0,0.0,0.089109,0.0,-8.910891,0.250825,0.0
5,03_medio_alto,PR,Patrimônio médio-alto - imobiliário concentrad...,65,0,0.0,0.076271,0.0,-7.627119,0.275424,0.0
6,03_medio_alto,SC,Patrimônio médio-alto - imobiliário concentrad...,33,0,0.0,0.073529,0.0,-7.352941,0.161765,0.0
7,02_medio_baixo,AM,Patrimônio médio-baixo - imobiliário concentra...,33,0,0.0,0.070707,0.0,-7.070707,0.333333,0.0
8,02_medio_baixo,AM,Patrimônio médio-baixo - imobiliário concentra...,25,0,0.0,0.070707,0.0,-7.070707,0.252525,0.0
9,01_baixo,AP,Baixo patrimônio - imobiliário concentrado,26,0,0.0,0.066667,0.0,-6.666667,0.346667,0.0


In [65]:
# ============================================================
# 3.14 Melhores UFs de cada cluster controlando por estrato
# ============================================================

top_ufs_por_cluster_controlado = (
    desempenho_cluster_estrato_uf_filtrado
    .sort_values(
        ["perfil_cluster", "indice_vs_estrato_uf", "delta_pp_vs_estrato_uf"],
        ascending=[True, False, False]
    )
    .groupby("perfil_cluster", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_ufs_por_cluster_controlado[
    [
        "perfil_cluster",
        "estrato",
        "SG_UF",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "taxa_estrato_uf",
        "indice_vs_estrato_uf",
        "delta_pp_vs_estrato_uf"
    ]
].head(40)

,perfil_cluster,estrato,SG_UF,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,taxa_estrato_uf,indice_vs_estrato_uf,delta_pp_vs_estrato_uf
0,Alto patrimônio - imobiliário + financeiro,04_alto,DF,31,4,0.129032,0.083832,1.539171,4.519992
1,Alto patrimônio - imobiliário + financeiro,04_alto,RJ,53,13,0.245283,0.161812,1.515849,8.347072
2,Alto patrimônio - imobiliário + financeiro,04_alto,SP,139,26,0.187050,0.133034,1.406030,5.401598
3,Alto patrimônio - imobiliário + financeiro,04_alto,MG,49,11,0.224490,0.160287,1.400548,6.420271
4,Alto patrimônio - imobiliário + financeiro,04_alto,SC,34,7,0.205882,0.167488,1.229239,3.839467
5,Alto patrimônio - imobiliário + rural/agropecu...,04_alto,TO,37,10,0.270270,0.190476,1.418919,7.979408
6,Alto patrimônio - imobiliário + rural/agropecu...,04_alto,RS,25,6,0.240000,0.187793,1.278000,5.220657
7,Alto patrimônio - imobiliário + rural/agropecu...,04_alto,BA,42,13,0.309524,0.277273,1.116315,3.225108
8,Alto patrimônio - imobiliário + rural/agropecu...,04_alto,GO,47,6,0.127660,0.162679,0.784731,-3.501985
9,Alto patrimônio - imobiliário + rural/agropecu...,04_alto,RO,27,2,0.074074,0.097826,0.757202,-2.375201


In [66]:
# ============================================================
# 3.15 Tabelas finais do Eixo 3
# ============================================================

tabela_eixo3_presenca = presenca_cluster_uf.copy()

tabela_eixo3_desempenho_uf = desempenho_cluster_uf_filtrado[
    [
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "candidatos_uf",
        "eleitos_uf",
        "taxa_uf",
        "indice_vs_uf",
        "delta_pp_vs_uf",
        "participacao_candidatos_uf",
        "participacao_eleitos_uf",
        "patrimonio_mediano_cluster_uf"
    ]
].copy()

tabela_eixo3_desempenho_estrato_uf = desempenho_cluster_estrato_uf_filtrado[
    [
        "estrato",
        "SG_UF",
        "perfil_cluster",
        "candidatos_cluster_uf",
        "eleitos_cluster_uf",
        "taxa_cluster_uf",
        "candidatos_estrato_uf",
        "eleitos_estrato_uf",
        "taxa_estrato_uf",
        "indice_vs_estrato_uf",
        "delta_pp_vs_estrato_uf",
        "participacao_candidatos_estrato_uf",
        "participacao_eleitos_estrato_uf",
        "patrimonio_mediano_cluster_uf"
    ]
].copy()

tabela_eixo3_desempenho_estrato_uf.head(20)

,estrato,SG_UF,perfil_cluster,candidatos_cluster_uf,eleitos_cluster_uf,taxa_cluster_uf,candidatos_estrato_uf,eleitos_estrato_uf,taxa_estrato_uf,indice_vs_estrato_uf,delta_pp_vs_estrato_uf,participacao_candidatos_estrato_uf,participacao_eleitos_estrato_uf,patrimonio_mediano_cluster_uf
390,02_medio_baixo,RJ,Patrimônio médio-baixo - veicular + financeiro,38,10,0.263158,300,18,0.060000,4.385965,20.315789,0.126667,0.555556,164691.555
33,01_baixo,MG,Baixo patrimônio - financeiro concentrado,42,5,0.119048,441,12,0.027211,4.375000,9.183673,0.095238,0.416667,15000.000
262,02_medio_baixo,SP,Patrimônio médio-baixo - imobiliário concentra...,62,6,0.096774,521,14,0.026871,3.601382,6.990279,0.119002,0.428571,196270.495
409,02_medio_baixo,MG,Patrimônio médio-baixo - veicular concentrado,37,4,0.108108,387,14,0.036176,2.988417,7.193240,0.095607,0.285714,160000.000
389,02_medio_baixo,PR,Patrimônio médio-baixo - veicular + financeiro,25,2,0.080000,256,7,0.027344,2.925714,5.265625,0.097656,0.285714,180454.080
48,01_baixo,SP,Baixo patrimônio - financeiro concentrado,140,7,0.050000,745,13,0.017450,2.865385,3.255034,0.187919,0.538462,5000.000
498,03_medio_alto,RJ,Patrimônio médio-alto - imobiliário concentrad...,41,10,0.243902,303,27,0.089109,2.737127,15.479353,0.135314,0.370370,482549.040
27,01_baixo,BA,Baixo patrimônio - financeiro concentrado,52,2,0.038462,339,5,0.014749,2.607692,2.371228,0.153392,0.400000,5000.000
259,02_medio_baixo,RS,Patrimônio médio-baixo - imobiliário concentra...,25,5,0.200000,237,19,0.080169,2.494737,11.983122,0.105485,0.263158,191171.750
598,03_medio_alto,MG,Patrimônio médio-alto - veicular diversificado,25,5,0.200000,401,35,0.087282,2.291429,11.271820,0.062344,0.142857,430000.000


In [67]:
# ============================================================
# 3.16 Salvamento das tabelas do Eixo 3
# ============================================================

from pathlib import Path

PASTA_SAIDAS = Path("results")
PASTA_SAIDAS.mkdir(exist_ok=True)

desempenho_uf.to_csv(
    PASTA_SAIDAS / "eixo3_desempenho_geral_por_uf.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo3_presenca.to_csv(
    PASTA_SAIDAS / "eixo3_presenca_cluster_por_uf.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo3_desempenho_uf.to_csv(
    PASTA_SAIDAS / "eixo3_desempenho_cluster_por_uf.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo3_desempenho_estrato_uf.to_csv(
    PASTA_SAIDAS / "eixo3_desempenho_cluster_por_uf_dentro_estrato.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

In [68]:
# ============================================================
# 3.17 Configurações para gráficos do Eixo 3
# ============================================================

import plotly.express as px
import textwrap

PASTA_GRAFICOS = Path("plot")
PASTA_GRAFICOS.mkdir(exist_ok=True)

if "RENDERIZAR_NO_NOTEBOOK" not in globals():
    RENDERIZAR_NO_NOTEBOOK = False

def quebra_label(texto, largura=55):
    if pd.isna(texto):
        return texto
    return "<br>".join(textwrap.wrap(str(texto), width=largura))

In [69]:
# ============================================================
# 3.19 Gráfico — Maiores sobreperformances cluster x UF
# ============================================================

N_TOP_UF = 30

df_plot_top_uf = (
    top_sobreperformance_uf
    .head(N_TOP_UF)
    .copy()
)

df_plot_top_uf["cluster_uf_label"] = (
    df_plot_top_uf["SG_UF"]
    + " — "
    + df_plot_top_uf["perfil_cluster"].apply(lambda x: quebra_label(x, 55))
)

fig = px.bar(
    df_plot_top_uf.sort_values("indice_vs_uf", ascending=True),
    x="indice_vs_uf",
    y="cluster_uf_label",
    orientation="h",
    hover_data={
        "SG_UF": True,
        "perfil_cluster": True,
        "cluster_uf_label": False,
        "candidatos_cluster_uf": True,
        "eleitos_cluster_uf": True,
        "taxa_cluster_uf": ":.2%",
        "taxa_uf": ":.2%",
        "indice_vs_uf": ":.2f",
        "delta_pp_vs_uf": ":.2f"
    },
    labels={
        "indice_vs_uf": "Índice vs média da UF",
        "cluster_uf_label": "UF — Cluster"
    },
    title="Maiores sobreperformances eleitorais de clusters por UF"
)

fig.add_vline(
    x=1,
    line_dash="dash",
    annotation_text="média da UF",
    annotation_position="top"
)

fig.update_layout(height=1000)

fig.write_html(PASTA_GRAFICOS / "eixo3_top_sobreperformance_cluster_uf.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [70]:
# ============================================================
# 3.20 Gráfico — Heatmap de sobreperformance cluster x UF
# Top clusters mais frequentes
# ============================================================

TOP_CLUSTERS_HEATMAP = 20

clusters_mais_frequentes_geo = (
    df_geo["perfil_cluster"]
    .value_counts()
    .head(TOP_CLUSTERS_HEATMAP)
    .index
)

df_heat_geo = (
    desempenho_cluster_uf_filtrado
    .loc[
        desempenho_cluster_uf_filtrado["perfil_cluster"]
        .isin(clusters_mais_frequentes_geo)
    ]
    .copy()
)

matriz_geo = (
    df_heat_geo
    .pivot_table(
        index="perfil_cluster",
        columns="SG_UF",
        values="indice_vs_uf",
        fill_value=np.nan,
        observed=True
    )
)

ordem_clusters_geo = (
    df_geo["perfil_cluster"]
    .value_counts()
    .loc[clusters_mais_frequentes_geo]
    .index
)

matriz_geo = matriz_geo.loc[ordem_clusters_geo]
matriz_geo.index = [quebra_label(idx, 65) for idx in matriz_geo.index]

fig = px.imshow(
    matriz_geo,
    aspect="auto",
    text_auto=".1f",
    labels={
        "x": "UF",
        "y": "Cluster",
        "color": "Índice vs UF"
    },
    title=f"Sobreperformance dos clusters por UF — Top {TOP_CLUSTERS_HEATMAP} clusters"
)

fig.update_layout(height=900)

fig.write_html(PASTA_GRAFICOS / "eixo3_heatmap_indice_cluster_uf.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [71]:
# ============================================================
# 3.21 Gráfico — Sobreperformance por UF dentro do estrato
# ============================================================

N_TOP_ESTRATO_UF = 30

df_plot_top_estrato_uf = (
    top_sobreperformance_estrato_uf
    .head(N_TOP_ESTRATO_UF)
    .copy()
)

df_plot_top_estrato_uf["label"] = (
    df_plot_top_estrato_uf["estrato"]
    + " | "
    + df_plot_top_estrato_uf["SG_UF"]
    + " — "
    + df_plot_top_estrato_uf["perfil_cluster"].apply(lambda x: quebra_label(x, 55))
)

fig = px.bar(
    df_plot_top_estrato_uf.sort_values("indice_vs_estrato_uf", ascending=True),
    x="indice_vs_estrato_uf",
    y="label",
    orientation="h",
    hover_data={
        "estrato": True,
        "SG_UF": True,
        "perfil_cluster": True,
        "label": False,
        "candidatos_cluster_uf": True,
        "eleitos_cluster_uf": True,
        "taxa_cluster_uf": ":.2%",
        "taxa_estrato_uf": ":.2%",
        "indice_vs_estrato_uf": ":.2f",
        "delta_pp_vs_estrato_uf": ":.2f"
    },
    labels={
        "indice_vs_estrato_uf": "Índice vs média do estrato na UF",
        "label": "Estrato | UF — Cluster"
    },
    title="Maiores sobreperformances de clusters por UF dentro do estrato"
)

fig.add_vline(
    x=1,
    line_dash="dash",
    annotation_text="média do estrato na UF",
    annotation_position="top"
)

fig.update_layout(height=1100)

fig.write_html(PASTA_GRAFICOS / "eixo3_top_sobreperformance_cluster_uf_dentro_estrato.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [72]:
# ============================================================
# 3.22 Gráfico — Heatmap controlado por estrato
# Um HTML por estrato
# ============================================================

for estrato_atual in sorted(df_geo["estrato"].dropna().unique()):
    
    df_heat_controlado = (
        desempenho_cluster_estrato_uf_filtrado
        .loc[
            desempenho_cluster_estrato_uf_filtrado["estrato"] == estrato_atual
        ]
        .copy()
    )
    
    if df_heat_controlado.empty:
        continue
    
    top_clusters_estrato_geo = (
        df_geo
        .loc[df_geo["estrato"] == estrato_atual, "perfil_cluster"]
        .value_counts()
        .head(15)
        .index
    )
    
    df_heat_controlado = df_heat_controlado.loc[
        df_heat_controlado["perfil_cluster"].isin(top_clusters_estrato_geo)
    ]
    
    matriz_controlada = (
        df_heat_controlado
        .pivot_table(
            index="perfil_cluster",
            columns="SG_UF",
            values="indice_vs_estrato_uf",
            fill_value=np.nan,
            observed=True
        )
    )
    
    if matriz_controlada.empty:
        continue
    
    ordem_clusters = (
        df_geo
        .loc[
            (df_geo["estrato"] == estrato_atual)
            & (df_geo["perfil_cluster"].isin(top_clusters_estrato_geo)),
            "perfil_cluster"
        ]
        .value_counts()
        .index
    )
    
    matriz_controlada = matriz_controlada.loc[ordem_clusters]
    matriz_controlada.index = [quebra_label(idx, 65) for idx in matriz_controlada.index]
    
    fig = px.imshow(
        matriz_controlada,
        aspect="auto",
        text_auto=".1f",
        labels={
            "x": "UF",
            "y": "Cluster",
            "color": "Índice vs estrato na UF"
        },
        title=f"Sobreperformance por UF dentro do estrato — {estrato_atual}"
    )
    
    fig.update_layout(height=800)
    
    nome_arquivo = f"eixo3_heatmap_indice_cluster_uf_dentro_estrato_{estrato_atual}.html"
    fig.write_html(PASTA_GRAFICOS / nome_arquivo)
    
    if RENDERIZAR_NO_NOTEBOOK:
        fig.show()

In [73]:
# ============================================================
# SEÇÃO 4 — EIXO 4
# Nichos eleitorais dos clusters por cargo
# ============================================================
#
# Pergunta central:
# Em quais cargos determinados perfis patrimoniais performam melhor?
#
# Importante:
# A distribuição por cargo é apenas contextual.
# O foco analítico é desempenho eleitoral.
#
# Métricas principais:
# taxa_cluster_cargo = eleitos do cluster no cargo / candidatos do cluster no cargo
# taxa_cargo = eleitos no cargo / candidatos no cargo
# indice_vs_cargo = taxa_cluster_cargo / taxa_cargo
#
# Controle adicional:
# taxa_estrato_cargo = eleitos do estrato no cargo / candidatos do estrato no cargo
# indice_vs_estrato_cargo = taxa_cluster_cargo / taxa_estrato_cargo
# ============================================================

In [74]:
# ============================================================
# 4.0 Checagens iniciais
# ============================================================

if "df_integrado" not in globals():
    raise NameError(
        "df_integrado não existe. Execute antes a seção de integração."
    )

colunas_necessarias_eixo4 = [
    "SQ_CANDIDATO",
    "DS_CARGO",
    "estrato",
    "perfil_cluster",
    "eleito",
    "patrimonio_total"
]

colunas_faltantes_eixo4 = [
    col for col in colunas_necessarias_eixo4
    if col not in df_integrado.columns
]

if colunas_faltantes_eixo4:
    raise ValueError(f"Colunas faltantes para o Eixo 4: {colunas_faltantes_eixo4}")

df_cargo = df_integrado.copy()

df_cargo["DS_CARGO"] = (
    df_cargo["DS_CARGO"]
    .astype(str)
    .str.upper()
    .str.strip()
)

print("Base por cargo:", df_cargo.shape)
print("Cargos:", df_cargo["DS_CARGO"].nunique())
print("Clusters:", df_cargo["perfil_cluster"].nunique())
print("Estratos:", df_cargo["estrato"].nunique())
print("Taxa média:", round(df_cargo["eleito"].mean(), 4))

Base por cargo: (18219, 41)
Cargos: 10
Clusters: 35
Estratos: 5
Taxa média: 0.0884


In [75]:
# ============================================================
# 4.1 Desempenho eleitoral geral por cargo
# ============================================================

desempenho_cargo = (
    df_cargo
    .groupby("DS_CARGO", observed=True)
    .agg(
        candidatos_cargo=("SQ_CANDIDATO", "count"),
        eleitos_cargo=("eleito", "sum"),
        clusters_presentes=("perfil_cluster", "nunique"),
        estratos_presentes=("estrato", "nunique"),
        patrimonio_mediano_cargo=("patrimonio_total", "median"),
        patrimonio_medio_cargo=("patrimonio_total", "mean")
    )
    .reset_index()
)

desempenho_cargo["taxa_cargo"] = (
    desempenho_cargo["eleitos_cargo"]
    / desempenho_cargo["candidatos_cargo"]
)

desempenho_cargo = desempenho_cargo.sort_values(
    "taxa_cargo",
    ascending=False
)

desempenho_cargo

,DS_CARGO,candidatos_cargo,eleitos_cargo,clusters_presentes,estratos_presentes,patrimonio_mediano_cargo,patrimonio_medio_cargo,taxa_cargo
1,2º SUPLENTE,184,26,33,5,460626.470,1.121968e+07,0.141304
0,1º SUPLENTE,194,25,33,5,632559.600,8.035638e+06,0.128866
7,SENADOR,213,27,32,5,785349.660,2.832260e+06,0.126761
8,VICE-GOVERNADOR,189,19,30,5,495101.790,4.179807e+06,0.100529
3,DEPUTADO ESTADUAL,10056,975,35,5,240900.000,8.564096e+05,0.096957
5,GOVERNADOR,193,18,33,5,738632.430,1.007059e+07,0.093264
9,VICE-PRESIDENTE,11,1,8,3,1005728.420,2.867645e+06,0.090909
4,DEPUTADO FEDERAL,6772,498,35,5,305212.055,1.130507e+06,0.073538
2,DEPUTADO DISTRITAL,394,22,31,5,310000.000,7.013752e+05,0.055838
6,PRESIDENTE,13,0,8,4,2317554.730,1.117567e+07,0.000000


In [76]:
# ============================================================
# 4.2 Distribuição dos clusters por cargo
# Apenas contextual
# ============================================================

presenca_cluster_cargo = (
    df_cargo
    .groupby(["perfil_cluster", "DS_CARGO"], observed=True)
    .agg(
        candidatos_cluster_cargo=("SQ_CANDIDATO", "count")
    )
    .reset_index()
)

total_cluster = (
    df_cargo
    .groupby("perfil_cluster", observed=True)
    .agg(
        candidatos_cluster_total=("SQ_CANDIDATO", "count")
    )
    .reset_index()
)

total_cargo = (
    df_cargo
    .groupby("DS_CARGO", observed=True)
    .agg(
        candidatos_cargo=("SQ_CANDIDATO", "count")
    )
    .reset_index()
)

presenca_cluster_cargo = (
    presenca_cluster_cargo
    .merge(total_cluster, on="perfil_cluster", how="left")
    .merge(total_cargo, on="DS_CARGO", how="left")
)

presenca_cluster_cargo["perc_cargo_no_cluster"] = (
    presenca_cluster_cargo["candidatos_cluster_cargo"]
    / presenca_cluster_cargo["candidatos_cluster_total"]
)

presenca_cluster_cargo["perc_cluster_no_cargo"] = (
    presenca_cluster_cargo["candidatos_cluster_cargo"]
    / presenca_cluster_cargo["candidatos_cargo"]
)

presenca_cluster_cargo = presenca_cluster_cargo.sort_values(
    ["perfil_cluster", "candidatos_cluster_cargo"],
    ascending=[True, False]
)

presenca_cluster_cargo.head(20)

,perfil_cluster,DS_CARGO,candidatos_cluster_cargo,candidatos_cluster_total,candidatos_cargo,perc_cargo_no_cluster,perc_cluster_no_cargo
3,Alto patrimônio - imobiliário + financeiro,DEPUTADO ESTADUAL,332,677,10056,0.490399,0.033015
4,Alto patrimônio - imobiliário + financeiro,DEPUTADO FEDERAL,274,677,6772,0.404727,0.040461
2,Alto patrimônio - imobiliário + financeiro,DEPUTADO DISTRITAL,19,677,394,0.028065,0.048223
8,Alto patrimônio - imobiliário + financeiro,VICE-GOVERNADOR,13,677,189,0.019202,0.068783
5,Alto patrimônio - imobiliário + financeiro,GOVERNADOR,12,677,193,0.017725,0.062176
7,Alto patrimônio - imobiliário + financeiro,SENADOR,10,677,213,0.014771,0.046948
1,Alto patrimônio - imobiliário + financeiro,2º SUPLENTE,8,677,184,0.011817,0.043478
0,Alto patrimônio - imobiliário + financeiro,1º SUPLENTE,6,677,194,0.008863,0.030928
9,Alto patrimônio - imobiliário + financeiro,VICE-PRESIDENTE,2,677,11,0.002954,0.181818
6,Alto patrimônio - imobiliário + financeiro,PRESIDENTE,1,677,13,0.001477,0.076923


In [77]:
# ============================================================
# 4.3 Taxa de eleição por cluster e cargo
# ============================================================

desempenho_cluster_cargo = (
    df_cargo
    .groupby(["perfil_cluster", "DS_CARGO"], observed=True)
    .agg(
        candidatos_cluster_cargo=("SQ_CANDIDATO", "count"),
        eleitos_cluster_cargo=("eleito", "sum"),
        patrimonio_mediano_cluster_cargo=("patrimonio_total", "median"),
        estratos_presentes=("estrato", "nunique")
    )
    .reset_index()
)

desempenho_cluster_cargo["taxa_cluster_cargo"] = (
    desempenho_cluster_cargo["eleitos_cluster_cargo"]
    / desempenho_cluster_cargo["candidatos_cluster_cargo"]
)

desempenho_cluster_cargo = desempenho_cluster_cargo.merge(
    desempenho_cargo[
        [
            "DS_CARGO",
            "candidatos_cargo",
            "eleitos_cargo",
            "taxa_cargo"
        ]
    ],
    on="DS_CARGO",
    how="left"
)

desempenho_cluster_cargo["indice_vs_cargo"] = np.where(
    desempenho_cluster_cargo["taxa_cargo"] > 0,
    desempenho_cluster_cargo["taxa_cluster_cargo"]
    / desempenho_cluster_cargo["taxa_cargo"],
    np.nan
)

desempenho_cluster_cargo["delta_pp_vs_cargo"] = (
    desempenho_cluster_cargo["taxa_cluster_cargo"]
    - desempenho_cluster_cargo["taxa_cargo"]
) * 100

desempenho_cluster_cargo["participacao_candidatos_cargo"] = (
    desempenho_cluster_cargo["candidatos_cluster_cargo"]
    / desempenho_cluster_cargo["candidatos_cargo"]
)

desempenho_cluster_cargo["participacao_eleitos_cargo"] = np.where(
    desempenho_cluster_cargo["eleitos_cargo"] > 0,
    desempenho_cluster_cargo["eleitos_cluster_cargo"]
    / desempenho_cluster_cargo["eleitos_cargo"],
    np.nan
)

desempenho_cluster_cargo = desempenho_cluster_cargo.sort_values(
    ["indice_vs_cargo", "delta_pp_vs_cargo"],
    ascending=[False, False]
)

desempenho_cluster_cargo.head(20)

,perfil_cluster,DS_CARGO,candidatos_cluster_cargo,eleitos_cluster_cargo,patrimonio_mediano_cluster_cargo,estratos_presentes,taxa_cluster_cargo,candidatos_cargo,eleitos_cargo,taxa_cargo,indice_vs_cargo,delta_pp_vs_cargo,participacao_candidatos_cargo,participacao_eleitos_cargo
57,Alto patrimônio - imobiliário diversificado co...,DEPUTADO DISTRITAL,1,1,1.348204e+06,1,1.000000,394,22,0.055838,17.909091,94.416244,0.002538,0.045455
19,Alto patrimônio - imobiliário + rural/agropecu...,VICE-PRESIDENTE,1,1,1.005728e+06,1,1.000000,11,1,0.090909,11.000000,90.909091,0.090909,1.000000
187,Patrimônio médio-baixo - imobiliário + veicular,DEPUTADO DISTRITAL,4,2,2.678793e+05,1,0.500000,394,22,0.055838,8.954545,44.416244,0.010152,0.090909
270,Patrimônio ultra-alto - rural/agropecuário con...,GOVERNADOR,3,2,2.074429e+07,1,0.666667,193,18,0.093264,7.148148,57.340242,0.015544,0.111111
151,Patrimônio médio-alto - imobiliário concentrad...,VICE-GOVERNADOR,3,2,7.249087e+05,1,0.666667,189,19,0.100529,6.631579,56.613757,0.015873,0.105263
275,Patrimônio ultra-alto - societário concentrado,GOVERNADOR,8,4,1.193948e+08,1,0.500000,193,18,0.093264,5.361111,40.673575,0.041451,0.222222
255,Patrimônio ultra-alto - financeiro + societário,DEPUTADO ESTADUAL,8,4,1.968841e+07,1,0.500000,10056,975,0.096957,5.156923,40.304296,0.000796,0.004103
145,Patrimônio médio-alto - imobiliário concentrad...,DEPUTADO DISTRITAL,7,2,3.971969e+05,1,0.285714,394,22,0.055838,5.116883,22.987672,0.017766,0.090909
200,Patrimônio médio-baixo - imobiliário concentra...,VICE-GOVERNADOR,2,1,2.978828e+05,1,0.500000,189,19,0.100529,4.973684,39.947090,0.010582,0.052632
235,Patrimônio médio-baixo - veicular + financeiro,DEPUTADO DISTRITAL,11,3,1.732860e+05,1,0.272727,394,22,0.055838,4.884298,21.688971,0.027919,0.136364


In [78]:
# ============================================================
# 4.4 Filtro mínimo para interpretação cluster x cargo
# ============================================================

MIN_CANDIDATOS_CLUSTER_CARGO = 25
MIN_CANDIDATOS_CARGO = 100
MIN_ELEITOS_CARGO = 5

desempenho_cluster_cargo_filtrado = (
    desempenho_cluster_cargo
    .loc[
        (
            desempenho_cluster_cargo["candidatos_cluster_cargo"]
            >= MIN_CANDIDATOS_CLUSTER_CARGO
        )
        & (
            desempenho_cluster_cargo["candidatos_cargo"]
            >= MIN_CANDIDATOS_CARGO
        )
        & (
            desempenho_cluster_cargo["eleitos_cargo"]
            >= MIN_ELEITOS_CARGO
        )
        & (
            desempenho_cluster_cargo["taxa_cargo"] > 0
        )
    ]
    .copy()
)

desempenho_cluster_cargo_filtrado = desempenho_cluster_cargo_filtrado.sort_values(
    ["indice_vs_cargo", "delta_pp_vs_cargo", "candidatos_cluster_cargo"],
    ascending=[False, False, False]
)

desempenho_cluster_cargo_filtrado.head(20)

,perfil_cluster,DS_CARGO,candidatos_cluster_cargo,eleitos_cluster_cargo,patrimonio_mediano_cluster_cargo,estratos_presentes,taxa_cluster_cargo,candidatos_cargo,eleitos_cargo,taxa_cargo,indice_vs_cargo,delta_pp_vs_cargo,participacao_candidatos_cargo,participacao_eleitos_cargo
59,Alto patrimônio - imobiliário diversificado co...,DEPUTADO FEDERAL,197,57,2756874.930,1,0.289340,6772,498,0.073538,3.934561,21.580200,0.029090,0.114458
58,Alto patrimônio - imobiliário diversificado co...,DEPUTADO ESTADUAL,198,74,2086149.140,1,0.373737,10056,975,0.096957,3.854670,27.678033,0.019690,0.075897
50,Alto patrimônio - imobiliário diversificado co...,DEPUTADO ESTADUAL,221,68,1802344.720,1,0.307692,10056,975,0.096957,3.173491,21.073527,0.021977,0.069744
3,Alto patrimônio - imobiliário + financeiro,DEPUTADO ESTADUAL,332,91,1184814.110,1,0.274096,10056,975,0.096957,2.826988,17.713934,0.033015,0.093333
4,Alto patrimônio - imobiliário + financeiro,DEPUTADO FEDERAL,274,55,1349952.395,1,0.200730,6772,498,0.073538,2.729605,12.719183,0.040461,0.110442
23,Alto patrimônio - imobiliário + societário com...,DEPUTADO ESTADUAL,487,116,1426948.000,1,0.238193,10056,975,0.096957,2.456686,14.123598,0.048429,0.118974
180,Patrimônio médio-alto - veicular diversificado,DEPUTADO FEDERAL,151,27,414884.000,1,0.178808,6772,498,0.073538,2.431501,10.526985,0.022298,0.054217
24,Alto patrimônio - imobiliário + societário com...,DEPUTADO FEDERAL,455,81,1627500.000,1,0.178022,6772,498,0.073538,2.420813,10.448388,0.067188,0.162651
147,Patrimônio médio-alto - imobiliário concentrad...,DEPUTADO FEDERAL,157,27,509302.560,1,0.171975,6772,498,0.073538,2.338577,9.843642,0.023184,0.054217
33,Alto patrimônio - imobiliário concentrado com ...,DEPUTADO ESTADUAL,254,56,1200000.000,1,0.220472,10056,975,0.096957,2.273919,12.351540,0.025259,0.057436


In [79]:
# ============================================================
# 4.5 Maiores sobreperformances por cargo
# ============================================================

top_sobreperformance_cargo = (
    desempenho_cluster_cargo_filtrado
    .sort_values(
        ["indice_vs_cargo", "delta_pp_vs_cargo", "candidatos_cluster_cargo"],
        ascending=[False, False, False]
    )
    .head(30)
    .reset_index(drop=True)
)

top_sobreperformance_cargo[
    [
        "DS_CARGO",
        "perfil_cluster",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "taxa_cargo",
        "indice_vs_cargo",
        "delta_pp_vs_cargo",
        "participacao_candidatos_cargo",
        "participacao_eleitos_cargo"
    ]
]

,DS_CARGO,perfil_cluster,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,taxa_cargo,indice_vs_cargo,delta_pp_vs_cargo,participacao_candidatos_cargo,participacao_eleitos_cargo
0,DEPUTADO FEDERAL,Alto patrimônio - imobiliário diversificado co...,197,57,0.289340,0.073538,3.934561,21.580200,0.029090,0.114458
1,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário diversificado co...,198,74,0.373737,0.096957,3.854670,27.678033,0.019690,0.075897
2,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário diversificado co...,221,68,0.307692,0.096957,3.173491,21.073527,0.021977,0.069744
3,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário + financeiro,332,91,0.274096,0.096957,2.826988,17.713934,0.033015,0.093333
4,DEPUTADO FEDERAL,Alto patrimônio - imobiliário + financeiro,274,55,0.200730,0.073538,2.729605,12.719183,0.040461,0.110442
5,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário + societário com...,487,116,0.238193,0.096957,2.456686,14.123598,0.048429,0.118974
6,DEPUTADO FEDERAL,Patrimônio médio-alto - veicular diversificado,151,27,0.178808,0.073538,2.431501,10.526985,0.022298,0.054217
7,DEPUTADO FEDERAL,Alto patrimônio - imobiliário + societário com...,455,81,0.178022,0.073538,2.420813,10.448388,0.067188,0.162651
8,DEPUTADO FEDERAL,Patrimônio médio-alto - imobiliário concentrad...,157,27,0.171975,0.073538,2.338577,9.843642,0.023184,0.054217
9,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário concentrado com ...,254,56,0.220472,0.096957,2.273919,12.351540,0.025259,0.057436


In [80]:
# ============================================================
# 4.6 Maiores subperformances por cargo
# ============================================================

bottom_sobreperformance_cargo = (
    desempenho_cluster_cargo_filtrado
    .sort_values(
        ["indice_vs_cargo", "delta_pp_vs_cargo", "candidatos_cluster_cargo"],
        ascending=[True, True, False]
    )
    .head(30)
    .reset_index(drop=True)
)

bottom_sobreperformance_cargo[
    [
        "DS_CARGO",
        "perfil_cluster",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "taxa_cargo",
        "indice_vs_cargo",
        "delta_pp_vs_cargo",
        "participacao_candidatos_cargo",
        "participacao_eleitos_cargo"
    ]
]

,DS_CARGO,perfil_cluster,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,taxa_cargo,indice_vs_cargo,delta_pp_vs_cargo,participacao_candidatos_cargo,participacao_eleitos_cargo
0,DEPUTADO ESTADUAL,Baixo patrimônio - imobiliário + veicular,193,0,0.000000,0.096957,0.000000,-9.695704,0.019193,0.000000
1,DEPUTADO ESTADUAL,Baixo patrimônio - outros ativos diversificado,129,0,0.000000,0.096957,0.000000,-9.695704,0.012828,0.000000
2,DEPUTADO FEDERAL,Baixo patrimônio - outros ativos diversificado,109,0,0.000000,0.073538,0.000000,-7.353810,0.016096,0.000000
3,DEPUTADO DISTRITAL,Baixo patrimônio - veicular concentrado com ru...,60,0,0.000000,0.055838,0.000000,-5.583756,0.152284,0.000000
4,DEPUTADO DISTRITAL,Patrimônio médio-baixo - imobiliário concentra...,30,0,0.000000,0.055838,0.000000,-5.583756,0.076142,0.000000
5,DEPUTADO FEDERAL,Patrimônio médio-baixo - imobiliário concentra...,354,2,0.005650,0.073538,0.076827,-6.788838,0.052274,0.004016
6,DEPUTADO ESTADUAL,Baixo patrimônio - veicular concentrado com ru...,1318,14,0.010622,0.096957,0.109555,-8.633489,0.131066,0.014359
7,DEPUTADO FEDERAL,Patrimônio médio-baixo - societário + veicular,118,1,0.008475,0.073538,0.115241,-6.506352,0.017425,0.002008
8,DEPUTADO FEDERAL,Baixo patrimônio - veicular concentrado com ru...,760,7,0.009211,0.073538,0.125248,-6.432757,0.112227,0.014056
9,DEPUTADO ESTADUAL,Patrimônio médio-baixo - imobiliário concentra...,699,9,0.012876,0.096957,0.132796,-8.408150,0.069511,0.009231


In [81]:
# ============================================================
# 4.7 Melhores cargos para cada cluster
# ============================================================

top_cargos_por_cluster = (
    desempenho_cluster_cargo_filtrado
    .sort_values(
        ["perfil_cluster", "indice_vs_cargo", "delta_pp_vs_cargo"],
        ascending=[True, False, False]
    )
    .groupby("perfil_cluster", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_cargos_por_cluster[
    [
        "perfil_cluster",
        "DS_CARGO",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "taxa_cargo",
        "indice_vs_cargo",
        "delta_pp_vs_cargo"
    ]
].head(40)

,perfil_cluster,DS_CARGO,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,taxa_cargo,indice_vs_cargo,delta_pp_vs_cargo
0,Alto patrimônio - imobiliário + financeiro,DEPUTADO ESTADUAL,332,91,0.274096,0.096957,2.826988,17.713934
1,Alto patrimônio - imobiliário + financeiro,DEPUTADO FEDERAL,274,55,0.200730,0.073538,2.729605,12.719183
2,Alto patrimônio - imobiliário + rural/agropecu...,DEPUTADO ESTADUAL,325,64,0.196923,0.096957,2.031034,9.996604
3,Alto patrimônio - imobiliário + rural/agropecu...,DEPUTADO FEDERAL,254,34,0.133858,0.073538,1.820257,6.032017
4,Alto patrimônio - imobiliário + societário com...,DEPUTADO ESTADUAL,487,116,0.238193,0.096957,2.456686,14.123598
5,Alto patrimônio - imobiliário + societário com...,DEPUTADO FEDERAL,455,81,0.178022,0.073538,2.420813,10.448388
6,Alto patrimônio - imobiliário + societário com...,SENADOR,29,7,0.241379,0.126761,1.904215,11.461875
7,Alto patrimônio - imobiliário concentrado com ...,DEPUTADO ESTADUAL,254,56,0.220472,0.096957,2.273919,12.351540
8,Alto patrimônio - imobiliário concentrado com ...,DEPUTADO FEDERAL,217,24,0.110599,0.073538,1.503970,3.706098
9,Alto patrimônio - imobiliário concentrado com ...,DEPUTADO DISTRITAL,28,3,0.107143,0.055838,1.918831,5.130529


In [82]:
# ============================================================
# 4.8 Clusters mais fortes dentro de cada cargo
# ============================================================

top_clusters_por_cargo = (
    desempenho_cluster_cargo_filtrado
    .sort_values(
        ["DS_CARGO", "indice_vs_cargo", "delta_pp_vs_cargo"],
        ascending=[True, False, False]
    )
    .groupby("DS_CARGO", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_clusters_por_cargo[
    [
        "DS_CARGO",
        "perfil_cluster",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "taxa_cargo",
        "indice_vs_cargo",
        "delta_pp_vs_cargo"
    ]
].head(40)

,DS_CARGO,perfil_cluster,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,taxa_cargo,indice_vs_cargo,delta_pp_vs_cargo
0,DEPUTADO DISTRITAL,Alto patrimônio - imobiliário concentrado com ...,28,3,0.107143,0.055838,1.918831,5.130529
1,DEPUTADO DISTRITAL,Patrimônio médio-alto - imobiliário concentrad...,42,3,0.071429,0.055838,1.279221,1.559101
2,DEPUTADO DISTRITAL,Baixo patrimônio - veicular concentrado com ru...,60,0,0.000000,0.055838,0.000000,-5.583756
3,DEPUTADO DISTRITAL,Patrimônio médio-baixo - imobiliário concentra...,30,0,0.000000,0.055838,0.000000,-5.583756
4,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário diversificado co...,198,74,0.373737,0.096957,3.854670,27.678033
5,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário diversificado co...,221,68,0.307692,0.096957,3.173491,21.073527
6,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário + financeiro,332,91,0.274096,0.096957,2.826988,17.713934
7,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário + societário com...,487,116,0.238193,0.096957,2.456686,14.123598
8,DEPUTADO ESTADUAL,Alto patrimônio - imobiliário concentrado com ...,254,56,0.220472,0.096957,2.273919,12.351540
9,DEPUTADO FEDERAL,Alto patrimônio - imobiliário diversificado co...,197,57,0.289340,0.073538,3.934561,21.580200


In [83]:
# ============================================================
# 4.9 Desempenho por estrato e cargo
# ============================================================

desempenho_estrato_cargo = (
    df_cargo
    .groupby(["estrato", "DS_CARGO"], observed=True)
    .agg(
        candidatos_estrato_cargo=("SQ_CANDIDATO", "count"),
        eleitos_estrato_cargo=("eleito", "sum"),
        patrimonio_mediano_estrato_cargo=("patrimonio_total", "median")
    )
    .reset_index()
)

desempenho_estrato_cargo["taxa_estrato_cargo"] = (
    desempenho_estrato_cargo["eleitos_estrato_cargo"]
    / desempenho_estrato_cargo["candidatos_estrato_cargo"]
)

desempenho_estrato_cargo = desempenho_estrato_cargo.sort_values(
    ["estrato", "DS_CARGO"]
)

desempenho_estrato_cargo.head(20)

,estrato,DS_CARGO,candidatos_estrato_cargo,eleitos_estrato_cargo,patrimonio_mediano_estrato_cargo,taxa_estrato_cargo
0,01_baixo,1º SUPLENTE,33,2,53500.00,0.060606
1,01_baixo,2º SUPLENTE,38,0,49628.00,0.000000
2,01_baixo,DEPUTADO DISTRITAL,110,2,38500.00,0.018182
3,01_baixo,DEPUTADO ESTADUAL,3142,72,32000.00,0.022915
4,01_baixo,DEPUTADO FEDERAL,1935,39,30066.88,0.020155
5,01_baixo,GOVERNADOR,31,2,36295.00,0.064516
6,01_baixo,PRESIDENTE,3,0,8548.13,0.000000
7,01_baixo,SENADOR,20,0,41334.22,0.000000
8,01_baixo,VICE-GOVERNADOR,49,1,25000.00,0.020408
9,01_baixo,VICE-PRESIDENTE,4,0,5956.34,0.000000


In [84]:
# ============================================================
# 4.10 Sobreperformance por cargo dentro do estrato
# ============================================================

desempenho_cluster_estrato_cargo = (
    df_cargo
    .groupby(["estrato", "perfil_cluster", "DS_CARGO"], observed=True)
    .agg(
        candidatos_cluster_cargo=("SQ_CANDIDATO", "count"),
        eleitos_cluster_cargo=("eleito", "sum"),
        patrimonio_mediano_cluster_cargo=("patrimonio_total", "median")
    )
    .reset_index()
)

desempenho_cluster_estrato_cargo["taxa_cluster_cargo"] = (
    desempenho_cluster_estrato_cargo["eleitos_cluster_cargo"]
    / desempenho_cluster_estrato_cargo["candidatos_cluster_cargo"]
)

desempenho_cluster_estrato_cargo = desempenho_cluster_estrato_cargo.merge(
    desempenho_estrato_cargo[
        [
            "estrato",
            "DS_CARGO",
            "candidatos_estrato_cargo",
            "eleitos_estrato_cargo",
            "taxa_estrato_cargo"
        ]
    ],
    on=["estrato", "DS_CARGO"],
    how="left"
)

desempenho_cluster_estrato_cargo["indice_vs_estrato_cargo"] = np.where(
    desempenho_cluster_estrato_cargo["taxa_estrato_cargo"] > 0,
    desempenho_cluster_estrato_cargo["taxa_cluster_cargo"]
    / desempenho_cluster_estrato_cargo["taxa_estrato_cargo"],
    np.nan
)

desempenho_cluster_estrato_cargo["delta_pp_vs_estrato_cargo"] = (
    desempenho_cluster_estrato_cargo["taxa_cluster_cargo"]
    - desempenho_cluster_estrato_cargo["taxa_estrato_cargo"]
) * 100

desempenho_cluster_estrato_cargo["participacao_candidatos_estrato_cargo"] = (
    desempenho_cluster_estrato_cargo["candidatos_cluster_cargo"]
    / desempenho_cluster_estrato_cargo["candidatos_estrato_cargo"]
)

desempenho_cluster_estrato_cargo["participacao_eleitos_estrato_cargo"] = np.where(
    desempenho_cluster_estrato_cargo["eleitos_estrato_cargo"] > 0,
    desempenho_cluster_estrato_cargo["eleitos_cluster_cargo"]
    / desempenho_cluster_estrato_cargo["eleitos_estrato_cargo"],
    np.nan
)

desempenho_cluster_estrato_cargo = desempenho_cluster_estrato_cargo.sort_values(
    ["indice_vs_estrato_cargo", "delta_pp_vs_estrato_cargo"],
    ascending=[False, False]
)

desempenho_cluster_estrato_cargo.head(20)

,estrato,perfil_cluster,DS_CARGO,candidatos_cluster_cargo,eleitos_cluster_cargo,patrimonio_mediano_cluster_cargo,taxa_cluster_cargo,candidatos_estrato_cargo,eleitos_estrato_cargo,taxa_estrato_cargo,indice_vs_estrato_cargo,delta_pp_vs_estrato_cargo,participacao_candidatos_estrato_cargo,participacao_eleitos_estrato_cargo
69,02_medio_baixo,Patrimônio médio-baixo - imobiliário + veicular,SENADOR,3,1,244436.920,0.333333,34,1,0.029412,11.333333,30.392157,0.088235,1.000000
240,04_alto,Alto patrimônio - imobiliário diversificado co...,DEPUTADO DISTRITAL,1,1,1348204.430,1.000000,95,9,0.094737,10.555556,90.526316,0.010526,0.111111
78,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,VICE-GOVERNADOR,2,1,297882.805,0.500000,34,2,0.058824,8.500000,44.117647,0.058824,0.500000
65,02_medio_baixo,Patrimônio médio-baixo - imobiliário + veicular,DEPUTADO DISTRITAL,4,2,267879.310,0.500000,83,5,0.060241,8.300000,43.975904,0.048193,0.400000
0,01_baixo,Baixo patrimônio - dinheiro em espécie concent...,1º SUPLENTE,2,1,37000.000,0.500000,33,2,0.060606,8.250000,43.939394,0.060606,0.500000
53,01_baixo,Baixo patrimônio - veicular concentrado com fi...,VICE-GOVERNADOR,7,1,24924.230,0.142857,49,1,0.020408,7.000000,12.244898,0.142857,1.000000
8,01_baixo,Baixo patrimônio - financeiro concentrado,DEPUTADO DISTRITAL,9,1,3260.970,0.111111,110,2,0.018182,6.111111,9.292929,0.081818,0.500000
48,01_baixo,Baixo patrimônio - veicular concentrado com fi...,DEPUTADO DISTRITAL,9,1,64762.010,0.111111,110,2,0.018182,6.111111,9.292929,0.081818,0.500000
202,04_alto,Alto patrimônio - imobiliário + rural/agropecu...,VICE-PRESIDENTE,1,1,1005728.420,1.000000,6,1,0.166667,6.000000,83.333333,0.166667,1.000000
6,01_baixo,Baixo patrimônio - financeiro concentrado,1º SUPLENTE,3,1,27263.850,0.333333,33,2,0.060606,5.500000,27.272727,0.090909,0.500000


In [85]:
# ============================================================
# 4.11 Filtro mínimo para cargo dentro do estrato
# ============================================================
#
# Este é o filtro mais importante do Eixo 4.
# Ele evita destacar casos em que poucos candidatos ou poucos eleitos
# distorcem o índice.
# ============================================================

MIN_CANDIDATOS_CLUSTER_ESTRATO_CARGO = 25
MIN_CANDIDATOS_ESTRATO_CARGO = 50
MIN_ELEITOS_ESTRATO_CARGO = 3

desempenho_cluster_estrato_cargo_filtrado = (
    desempenho_cluster_estrato_cargo
    .loc[
        (
            desempenho_cluster_estrato_cargo["candidatos_cluster_cargo"]
            >= MIN_CANDIDATOS_CLUSTER_ESTRATO_CARGO
        )
        & (
            desempenho_cluster_estrato_cargo["candidatos_estrato_cargo"]
            >= MIN_CANDIDATOS_ESTRATO_CARGO
        )
        & (
            desempenho_cluster_estrato_cargo["eleitos_estrato_cargo"]
            >= MIN_ELEITOS_ESTRATO_CARGO
        )
        & (
            desempenho_cluster_estrato_cargo["taxa_estrato_cargo"] > 0
        )
    ]
    .copy()
)

desempenho_cluster_estrato_cargo_filtrado = desempenho_cluster_estrato_cargo_filtrado.sort_values(
    ["indice_vs_estrato_cargo", "delta_pp_vs_estrato_cargo", "candidatos_cluster_cargo"],
    ascending=[False, False, False]
)

desempenho_cluster_estrato_cargo_filtrado.head(20)

,estrato,perfil_cluster,DS_CARGO,candidatos_cluster_cargo,eleitos_cluster_cargo,patrimonio_mediano_cluster_cargo,taxa_cluster_cargo,candidatos_estrato_cargo,eleitos_estrato_cargo,taxa_estrato_cargo,indice_vs_estrato_cargo,delta_pp_vs_estrato_cargo,participacao_candidatos_estrato_cargo,participacao_eleitos_estrato_cargo
75,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,DEPUTADO FEDERAL,119,16,192005.000,0.134454,1431,49,0.034242,3.926599,10.021199,0.083159,0.326531
9,01_baixo,Baixo patrimônio - financeiro concentrado,DEPUTADO ESTADUAL,367,28,7045.180,0.076294,3142,72,0.022915,3.329398,5.337894,0.116805,0.388889
114,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,DEPUTADO ESTADUAL,204,32,172252.575,0.156863,2497,119,0.047657,3.291481,10.920556,0.081698,0.268908
115,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,DEPUTADO FEDERAL,121,11,155463.120,0.090909,1431,49,0.034242,2.654917,5.666730,0.084556,0.224490
178,03_medio_alto,Patrimônio médio-alto - veicular diversificado,DEPUTADO FEDERAL,151,27,414884.000,0.178808,1522,110,0.072273,2.474052,10.653462,0.099212,0.245455
145,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,DEPUTADO FEDERAL,157,27,509302.560,0.171975,1522,110,0.072273,2.379502,9.970120,0.103154,0.245455
10,01_baixo,Baixo patrimônio - financeiro concentrado,DEPUTADO FEDERAL,280,12,6375.135,0.042857,1935,39,0.020155,2.126374,2.270210,0.144703,0.307692
107,02_medio_baixo,Patrimônio médio-baixo - societário + veicular,DEPUTADO ESTADUAL,188,19,170000.000,0.101064,2497,119,0.047657,2.120642,5.340664,0.075290,0.159664
74,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,DEPUTADO ESTADUAL,149,15,203000.000,0.100671,2497,119,0.047657,2.112402,5.301395,0.059672,0.126050
67,02_medio_baixo,Patrimônio médio-baixo - imobiliário + veicular,DEPUTADO FEDERAL,145,10,211139.660,0.068966,1431,49,0.034242,2.014075,3.472373,0.101328,0.204082


In [86]:
# ============================================================
# 4.12 Maiores sobreperformances por cargo dentro do estrato
# ============================================================

top_sobreperformance_estrato_cargo = (
    desempenho_cluster_estrato_cargo_filtrado
    .sort_values(
        [
            "indice_vs_estrato_cargo",
            "delta_pp_vs_estrato_cargo",
            "candidatos_cluster_cargo"
        ],
        ascending=[False, False, False]
    )
    .head(30)
    .reset_index(drop=True)
)

top_sobreperformance_estrato_cargo[
    [
        "estrato",
        "DS_CARGO",
        "perfil_cluster",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "taxa_estrato_cargo",
        "indice_vs_estrato_cargo",
        "delta_pp_vs_estrato_cargo",
        "participacao_candidatos_estrato_cargo",
        "participacao_eleitos_estrato_cargo"
    ]
]

,estrato,DS_CARGO,perfil_cluster,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,taxa_estrato_cargo,indice_vs_estrato_cargo,delta_pp_vs_estrato_cargo,participacao_candidatos_estrato_cargo,participacao_eleitos_estrato_cargo
0,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - imobiliário concentra...,119,16,0.134454,0.034242,3.926599,10.021199,0.083159,0.326531
1,01_baixo,DEPUTADO ESTADUAL,Baixo patrimônio - financeiro concentrado,367,28,0.076294,0.022915,3.329398,5.337894,0.116805,0.388889
2,02_medio_baixo,DEPUTADO ESTADUAL,Patrimônio médio-baixo - veicular + financeiro,204,32,0.156863,0.047657,3.291481,10.920556,0.081698,0.268908
3,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - veicular + financeiro,121,11,0.090909,0.034242,2.654917,5.666730,0.084556,0.224490
4,03_medio_alto,DEPUTADO FEDERAL,Patrimônio médio-alto - veicular diversificado,151,27,0.178808,0.072273,2.474052,10.653462,0.099212,0.245455
5,03_medio_alto,DEPUTADO FEDERAL,Patrimônio médio-alto - imobiliário concentrad...,157,27,0.171975,0.072273,2.379502,9.970120,0.103154,0.245455
6,01_baixo,DEPUTADO FEDERAL,Baixo patrimônio - financeiro concentrado,280,12,0.042857,0.020155,2.126374,2.270210,0.144703,0.307692
7,02_medio_baixo,DEPUTADO ESTADUAL,Patrimônio médio-baixo - societário + veicular,188,19,0.101064,0.047657,2.120642,5.340664,0.075290,0.159664
8,02_medio_baixo,DEPUTADO ESTADUAL,Patrimônio médio-baixo - imobiliário concentra...,149,15,0.100671,0.047657,2.112402,5.301395,0.059672,0.126050
9,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - imobiliário + veicular,145,10,0.068966,0.034242,2.014075,3.472373,0.101328,0.204082


In [87]:
# ============================================================
# 4.13 Maiores subperformances por cargo dentro do estrato
# ============================================================

bottom_sobreperformance_estrato_cargo = (
    desempenho_cluster_estrato_cargo_filtrado
    .sort_values(
        [
            "indice_vs_estrato_cargo",
            "delta_pp_vs_estrato_cargo",
            "candidatos_cluster_cargo"
        ],
        ascending=[True, True, False]
    )
    .head(30)
    .reset_index(drop=True)
)

bottom_sobreperformance_estrato_cargo[
    [
        "estrato",
        "DS_CARGO",
        "perfil_cluster",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "taxa_estrato_cargo",
        "indice_vs_estrato_cargo",
        "delta_pp_vs_estrato_cargo",
        "participacao_candidatos_estrato_cargo",
        "participacao_eleitos_estrato_cargo"
    ]
]

,estrato,DS_CARGO,perfil_cluster,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,taxa_estrato_cargo,indice_vs_estrato_cargo,delta_pp_vs_estrato_cargo,participacao_candidatos_estrato_cargo,participacao_eleitos_estrato_cargo
0,02_medio_baixo,DEPUTADO DISTRITAL,Patrimônio médio-baixo - imobiliário concentra...,30,0,0.000000,0.060241,0.000000,-6.024096,0.361446,0.000000
1,01_baixo,DEPUTADO ESTADUAL,Baixo patrimônio - imobiliário + veicular,193,0,0.000000,0.022915,0.000000,-2.291534,0.061426,0.000000
2,01_baixo,DEPUTADO ESTADUAL,Baixo patrimônio - outros ativos diversificado,129,0,0.000000,0.022915,0.000000,-2.291534,0.041057,0.000000
3,01_baixo,DEPUTADO FEDERAL,Baixo patrimônio - outros ativos diversificado,109,0,0.000000,0.020155,0.000000,-2.015504,0.056331,0.000000
4,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - imobiliário concentra...,354,2,0.005650,0.034242,0.164995,-2.859207,0.247379,0.040816
5,04_alto,DEPUTADO FEDERAL,Alto patrimônio - imobiliário concentrado com ...,250,8,0.032000,0.155715,0.205504,-12.371507,0.138045,0.028369
6,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - societário + veicular,118,1,0.008475,0.034242,0.247492,-2.576721,0.082460,0.020408
7,03_medio_alto,DEPUTADO FEDERAL,Patrimônio médio-alto - imobiliário + veicular...,157,3,0.019108,0.072273,0.264389,-5.316504,0.103154,0.027273
8,02_medio_baixo,DEPUTADO ESTADUAL,Patrimônio médio-baixo - imobiliário concentra...,699,9,0.012876,0.047657,0.270170,-3.478165,0.279936,0.075630
9,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - imobiliário concentra...,384,5,0.013021,0.034242,0.380261,-2.122096,0.268344,0.102041


In [88]:
# ============================================================
# 4.14 Melhores cargos de cada cluster controlando por estrato
# ============================================================

top_cargos_por_cluster_controlado = (
    desempenho_cluster_estrato_cargo_filtrado
    .sort_values(
        [
            "perfil_cluster",
            "indice_vs_estrato_cargo",
            "delta_pp_vs_estrato_cargo"
        ],
        ascending=[True, False, False]
    )
    .groupby("perfil_cluster", observed=True)
    .head(5)
    .reset_index(drop=True)
)

top_cargos_por_cluster_controlado[
    [
        "perfil_cluster",
        "estrato",
        "DS_CARGO",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "taxa_estrato_cargo",
        "indice_vs_estrato_cargo",
        "delta_pp_vs_estrato_cargo"
    ]
].head(40)

,perfil_cluster,estrato,DS_CARGO,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,taxa_estrato_cargo,indice_vs_estrato_cargo,delta_pp_vs_estrato_cargo
0,Alto patrimônio - imobiliário + financeiro,04_alto,DEPUTADO FEDERAL,274,55,0.200730,0.155715,1.289085,4.501485
1,Alto patrimônio - imobiliário + financeiro,04_alto,DEPUTADO ESTADUAL,332,91,0.274096,0.241722,1.133933,3.237453
2,Alto patrimônio - imobiliário + rural/agropecu...,04_alto,DEPUTADO FEDERAL,254,34,0.133858,0.155715,0.859636,-2.185681
3,Alto patrimônio - imobiliário + rural/agropecu...,04_alto,DEPUTADO ESTADUAL,325,64,0.196923,0.241722,0.814668,-4.479878
4,Alto patrimônio - imobiliário + societário com...,04_alto,SENADOR,29,7,0.241379,0.166667,1.448276,7.471264
5,Alto patrimônio - imobiliário + societário com...,04_alto,DEPUTADO FEDERAL,455,81,0.178022,0.155715,1.143255,2.230690
6,Alto patrimônio - imobiliário + societário com...,04_alto,DEPUTADO ESTADUAL,487,116,0.238193,0.241722,0.985401,-0.352884
7,Alto patrimônio - imobiliário concentrado com ...,04_alto,DEPUTADO ESTADUAL,254,56,0.220472,0.241722,0.912091,-2.124941
8,Alto patrimônio - imobiliário concentrado com ...,04_alto,DEPUTADO FEDERAL,217,24,0.110599,0.155715,0.710266,-4.511600
9,Alto patrimônio - imobiliário concentrado com ...,04_alto,DEPUTADO DISTRITAL,28,3,0.107143,0.094737,1.130952,1.240602


In [89]:
# ============================================================
# 4.15 Tabelas finais do Eixo 4
# ============================================================

tabela_eixo4_presenca = presenca_cluster_cargo.copy()

tabela_eixo4_desempenho_cargo = desempenho_cluster_cargo_filtrado[
    [
        "DS_CARGO",
        "perfil_cluster",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "candidatos_cargo",
        "eleitos_cargo",
        "taxa_cargo",
        "indice_vs_cargo",
        "delta_pp_vs_cargo",
        "participacao_candidatos_cargo",
        "participacao_eleitos_cargo",
        "patrimonio_mediano_cluster_cargo"
    ]
].copy()

tabela_eixo4_desempenho_estrato_cargo = desempenho_cluster_estrato_cargo_filtrado[
    [
        "estrato",
        "DS_CARGO",
        "perfil_cluster",
        "candidatos_cluster_cargo",
        "eleitos_cluster_cargo",
        "taxa_cluster_cargo",
        "candidatos_estrato_cargo",
        "eleitos_estrato_cargo",
        "taxa_estrato_cargo",
        "indice_vs_estrato_cargo",
        "delta_pp_vs_estrato_cargo",
        "participacao_candidatos_estrato_cargo",
        "participacao_eleitos_estrato_cargo",
        "patrimonio_mediano_cluster_cargo"
    ]
].copy()

tabela_eixo4_desempenho_estrato_cargo.head(20)

,estrato,DS_CARGO,perfil_cluster,candidatos_cluster_cargo,eleitos_cluster_cargo,taxa_cluster_cargo,candidatos_estrato_cargo,eleitos_estrato_cargo,taxa_estrato_cargo,indice_vs_estrato_cargo,delta_pp_vs_estrato_cargo,participacao_candidatos_estrato_cargo,participacao_eleitos_estrato_cargo,patrimonio_mediano_cluster_cargo
75,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - imobiliário concentra...,119,16,0.134454,1431,49,0.034242,3.926599,10.021199,0.083159,0.326531,192005.000
9,01_baixo,DEPUTADO ESTADUAL,Baixo patrimônio - financeiro concentrado,367,28,0.076294,3142,72,0.022915,3.329398,5.337894,0.116805,0.388889,7045.180
114,02_medio_baixo,DEPUTADO ESTADUAL,Patrimônio médio-baixo - veicular + financeiro,204,32,0.156863,2497,119,0.047657,3.291481,10.920556,0.081698,0.268908,172252.575
115,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - veicular + financeiro,121,11,0.090909,1431,49,0.034242,2.654917,5.666730,0.084556,0.224490,155463.120
178,03_medio_alto,DEPUTADO FEDERAL,Patrimônio médio-alto - veicular diversificado,151,27,0.178808,1522,110,0.072273,2.474052,10.653462,0.099212,0.245455,414884.000
145,03_medio_alto,DEPUTADO FEDERAL,Patrimônio médio-alto - imobiliário concentrad...,157,27,0.171975,1522,110,0.072273,2.379502,9.970120,0.103154,0.245455,509302.560
10,01_baixo,DEPUTADO FEDERAL,Baixo patrimônio - financeiro concentrado,280,12,0.042857,1935,39,0.020155,2.126374,2.270210,0.144703,0.307692,6375.135
107,02_medio_baixo,DEPUTADO ESTADUAL,Patrimônio médio-baixo - societário + veicular,188,19,0.101064,2497,119,0.047657,2.120642,5.340664,0.075290,0.159664,170000.000
74,02_medio_baixo,DEPUTADO ESTADUAL,Patrimônio médio-baixo - imobiliário concentra...,149,15,0.100671,2497,119,0.047657,2.112402,5.301395,0.059672,0.126050,203000.000
67,02_medio_baixo,DEPUTADO FEDERAL,Patrimônio médio-baixo - imobiliário + veicular,145,10,0.068966,1431,49,0.034242,2.014075,3.472373,0.101328,0.204082,211139.660


In [90]:
# ============================================================
# 4.16 Salvamento das tabelas do Eixo 4
# ============================================================

from pathlib import Path

PASTA_SAIDAS = Path("results")
PASTA_SAIDAS.mkdir(exist_ok=True)

desempenho_cargo.to_csv(
    PASTA_SAIDAS / "eixo4_desempenho_geral_por_cargo.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo4_presenca.to_csv(
    PASTA_SAIDAS / "eixo4_presenca_cluster_por_cargo.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo4_desempenho_cargo.to_csv(
    PASTA_SAIDAS / "eixo4_desempenho_cluster_por_cargo.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo4_desempenho_estrato_cargo.to_csv(
    PASTA_SAIDAS / "eixo4_desempenho_cluster_por_cargo_dentro_estrato.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

In [91]:
# ============================================================
# 4.17 Configurações para gráficos do Eixo 4
# ============================================================

import plotly.express as px
import textwrap

PASTA_GRAFICOS = Path("plot")
PASTA_GRAFICOS.mkdir(exist_ok=True)

if "RENDERIZAR_NO_NOTEBOOK" not in globals():
    RENDERIZAR_NO_NOTEBOOK = False

def quebra_label(texto, largura=55):
    if pd.isna(texto):
        return texto
    return "<br>".join(textwrap.wrap(str(texto), width=largura))

In [92]:
# ============================================================
# 4.18 Gráfico — Taxa média de eleição por cargo
# ============================================================

fig = px.bar(
    desempenho_cargo.sort_values("taxa_cargo", ascending=True),
    x="taxa_cargo",
    y="DS_CARGO",
    orientation="h",
    text=desempenho_cargo.sort_values("taxa_cargo", ascending=True)["taxa_cargo"].map(lambda x: f"{x:.1%}"),
    hover_data={
        "DS_CARGO": True,
        "candidatos_cargo": True,
        "eleitos_cargo": True,
        "taxa_cargo": ":.2%",
        "patrimonio_mediano_cargo": ":,.2f"
    },
    labels={
        "taxa_cargo": "Taxa de eleição",
        "DS_CARGO": "Cargo"
    },
    title="Taxa média de eleição por cargo"
)

fig.update_xaxes(tickformat=".0%")
fig.update_layout(height=600)

fig.write_html(PASTA_GRAFICOS / "eixo4_taxa_media_eleicao_por_cargo.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [93]:
# ============================================================
# 4.19 Gráfico — Maiores sobreperformances cluster x cargo
# ============================================================

N_TOP_CARGO = 30

df_plot_top_cargo = (
    top_sobreperformance_cargo
    .head(N_TOP_CARGO)
    .copy()
)

df_plot_top_cargo["cluster_cargo_label"] = (
    df_plot_top_cargo["DS_CARGO"]
    + " — "
    + df_plot_top_cargo["perfil_cluster"].apply(lambda x: quebra_label(x, 55))
)

fig = px.bar(
    df_plot_top_cargo.sort_values("indice_vs_cargo", ascending=True),
    x="indice_vs_cargo",
    y="cluster_cargo_label",
    orientation="h",
    hover_data={
        "DS_CARGO": True,
        "perfil_cluster": True,
        "cluster_cargo_label": False,
        "candidatos_cluster_cargo": True,
        "eleitos_cluster_cargo": True,
        "taxa_cluster_cargo": ":.2%",
        "taxa_cargo": ":.2%",
        "indice_vs_cargo": ":.2f",
        "delta_pp_vs_cargo": ":.2f"
    },
    labels={
        "indice_vs_cargo": "Índice vs média do cargo",
        "cluster_cargo_label": "Cargo — Cluster"
    },
    title="Maiores sobreperformances eleitorais de clusters por cargo"
)

fig.add_vline(
    x=1,
    line_dash="dash",
    annotation_text="média do cargo",
    annotation_position="top"
)

fig.update_layout(height=1000)

fig.write_html(PASTA_GRAFICOS / "eixo4_top_sobreperformance_cluster_cargo.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [94]:
# ============================================================
# 4.20 Gráfico — Heatmap de sobreperformance cluster x cargo
# Top clusters mais frequentes
# ============================================================

TOP_CLUSTERS_HEATMAP_CARGO = 20

clusters_mais_frequentes_cargo = (
    df_cargo["perfil_cluster"]
    .value_counts()
    .head(TOP_CLUSTERS_HEATMAP_CARGO)
    .index
)

df_heat_cargo = (
    desempenho_cluster_cargo_filtrado
    .loc[
        desempenho_cluster_cargo_filtrado["perfil_cluster"]
        .isin(clusters_mais_frequentes_cargo)
    ]
    .copy()
)

matriz_cargo = (
    df_heat_cargo
    .pivot_table(
        index="perfil_cluster",
        columns="DS_CARGO",
        values="indice_vs_cargo",
        fill_value=np.nan,
        observed=True
    )
)

ordem_clusters_cargo = (
    df_cargo["perfil_cluster"]
    .value_counts()
    .loc[clusters_mais_frequentes_cargo]
    .index
)

matriz_cargo = matriz_cargo.loc[ordem_clusters_cargo]
matriz_cargo.index = [quebra_label(idx, 65) for idx in matriz_cargo.index]

fig = px.imshow(
    matriz_cargo,
    aspect="auto",
    text_auto=".1f",
    labels={
        "x": "Cargo",
        "y": "Cluster",
        "color": "Índice vs cargo"
    },
    title=f"Sobreperformance dos clusters por cargo — Top {TOP_CLUSTERS_HEATMAP_CARGO} clusters"
)

fig.update_layout(height=900)

fig.write_html(PASTA_GRAFICOS / "eixo4_heatmap_indice_cluster_cargo.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [95]:
# ============================================================
# 4.21 Gráfico — Sobreperformance por cargo dentro do estrato
# ============================================================

N_TOP_ESTRATO_CARGO = 30

df_plot_top_estrato_cargo = (
    top_sobreperformance_estrato_cargo
    .head(N_TOP_ESTRATO_CARGO)
    .copy()
)

df_plot_top_estrato_cargo["label"] = (
    df_plot_top_estrato_cargo["estrato"]
    + " | "
    + df_plot_top_estrato_cargo["DS_CARGO"]
    + " — "
    + df_plot_top_estrato_cargo["perfil_cluster"].apply(lambda x: quebra_label(x, 55))
)

fig = px.bar(
    df_plot_top_estrato_cargo.sort_values("indice_vs_estrato_cargo", ascending=True),
    x="indice_vs_estrato_cargo",
    y="label",
    orientation="h",
    hover_data={
        "estrato": True,
        "DS_CARGO": True,
        "perfil_cluster": True,
        "label": False,
        "candidatos_cluster_cargo": True,
        "eleitos_cluster_cargo": True,
        "taxa_cluster_cargo": ":.2%",
        "taxa_estrato_cargo": ":.2%",
        "indice_vs_estrato_cargo": ":.2f",
        "delta_pp_vs_estrato_cargo": ":.2f"
    },
    labels={
        "indice_vs_estrato_cargo": "Índice vs média do estrato no cargo",
        "label": "Estrato | Cargo — Cluster"
    },
    title="Maiores sobreperformances de clusters por cargo dentro do estrato"
)

fig.add_vline(
    x=1,
    line_dash="dash",
    annotation_text="média do estrato no cargo",
    annotation_position="top"
)

fig.update_layout(height=1100)

fig.write_html(PASTA_GRAFICOS / "eixo4_top_sobreperformance_cluster_cargo_dentro_estrato.html")

if RENDERIZAR_NO_NOTEBOOK:
    fig.show()

In [96]:
# ============================================================
# 4.22 Gráfico — Heatmap controlado por estrato
# Um HTML por estrato
# ============================================================

for estrato_atual in sorted(df_cargo["estrato"].dropna().unique()):
    
    df_heat_controlado = (
        desempenho_cluster_estrato_cargo_filtrado
        .loc[
            desempenho_cluster_estrato_cargo_filtrado["estrato"] == estrato_atual
        ]
        .copy()
    )
    
    if df_heat_controlado.empty:
        continue
    
    top_clusters_estrato_cargo = (
        df_cargo
        .loc[df_cargo["estrato"] == estrato_atual, "perfil_cluster"]
        .value_counts()
        .head(15)
        .index
    )
    
    df_heat_controlado = df_heat_controlado.loc[
        df_heat_controlado["perfil_cluster"].isin(top_clusters_estrato_cargo)
    ]
    
    matriz_controlada = (
        df_heat_controlado
        .pivot_table(
            index="perfil_cluster",
            columns="DS_CARGO",
            values="indice_vs_estrato_cargo",
            fill_value=np.nan,
            observed=True
        )
    )
    
    if matriz_controlada.empty:
        continue
    
    ordem_clusters = (
        df_cargo
        .loc[
            (df_cargo["estrato"] == estrato_atual)
            & (df_cargo["perfil_cluster"].isin(top_clusters_estrato_cargo)),
            "perfil_cluster"
        ]
        .value_counts()
        .index
    )
    
    matriz_controlada = matriz_controlada.loc[ordem_clusters]
    matriz_controlada.index = [quebra_label(idx, 65) for idx in matriz_controlada.index]
    
    fig = px.imshow(
        matriz_controlada,
        aspect="auto",
        text_auto=".1f",
        labels={
            "x": "Cargo",
            "y": "Cluster",
            "color": "Índice vs estrato no cargo"
        },
        title=f"Sobreperformance por cargo dentro do estrato — {estrato_atual}"
    )
    
    fig.update_layout(height=800)
    
    nome_arquivo = f"eixo4_heatmap_indice_cluster_cargo_dentro_estrato_{estrato_atual}.html"
    fig.write_html(PASTA_GRAFICOS / nome_arquivo)
    
    if RENDERIZAR_NO_NOTEBOOK:
        fig.show()

In [97]:
# ============================================================
# SEÇÃO 5 — EIXO 5
# Perfil demográfico dos clusters-chave
# ============================================================
#
# Pergunta central:
# Quem compõe os clusters patrimoniais que se mostraram mais relevantes
# nos eixos de desempenho eleitoral?
#
# Diferença em relação à estrutura antiga:
# Não vamos analisar gênero, raça, escolaridade e ocupação para todos
# os clusters de forma descritiva ampla.
#
# Agora essas variáveis entram como apoio interpretativo para entender
# os clusters que tiveram sobreperformance eleitoral.
# ============================================================

In [98]:
# ============================================================
# 5.0 Checagens iniciais
# ============================================================

if "df_integrado" not in globals():
    raise NameError(
        "df_integrado não existe. Execute antes a seção de integração."
    )

colunas_base_eixo5 = [
    "SQ_CANDIDATO",
    "perfil_cluster",
    "estrato",
    "eleito"
]

colunas_faltantes_eixo5 = [
    col for col in colunas_base_eixo5
    if col not in df_integrado.columns
]

if colunas_faltantes_eixo5:
    raise ValueError(f"Colunas faltantes para o Eixo 5: {colunas_faltantes_eixo5}")

df_demo = df_integrado.copy()

print("Base:", df_demo.shape)
print("Clusters:", df_demo["perfil_cluster"].nunique())
print("Estratos:", df_demo["estrato"].nunique())
print("Eleitos:", df_demo["eleito"].sum())

Base: (18219, 41)
Clusters: 35
Estratos: 5
Eleitos: 1611


In [99]:
# ============================================================
# 5.1 Seleção automática dos clusters-chave
# ============================================================
#
# A lista é formada a partir dos clusters mais relevantes dos Eixos 2, 3 e 4.
# Se alguma tabela não existir no ambiente, ela é ignorada.
# ============================================================

clusters_interessantes = []

# Eixo 2: clusters com maior sobreperformance dentro do estrato
if "top_sobreperformance_estrato" in globals():
    clusters_interessantes += (
        top_sobreperformance_estrato["perfil_cluster"]
        .head(8)
        .dropna()
        .tolist()
    )

# Eixo 3: clusters com maior sobreperformance por UF dentro do estrato
if "top_sobreperformance_estrato_uf" in globals():
    clusters_interessantes += (
        top_sobreperformance_estrato_uf["perfil_cluster"]
        .head(8)
        .dropna()
        .tolist()
    )

# Eixo 4: clusters com maior sobreperformance por cargo dentro do estrato
if "top_sobreperformance_estrato_cargo" in globals():
    clusters_interessantes += (
        top_sobreperformance_estrato_cargo["perfil_cluster"]
        .head(8)
        .dropna()
        .tolist()
    )

clusters_interessantes = list(dict.fromkeys(clusters_interessantes))

print("Clusters selecionados automaticamente:", len(clusters_interessantes))

clusters_interessantes

Clusters selecionados automaticamente: 9


['Patrimônio médio-baixo - veicular + financeiro',
 'Baixo patrimônio - financeiro concentrado',
 'Patrimônio médio-baixo - imobiliário concentrado com financeiro e societário',
 'Patrimônio médio-alto - imobiliário concentrado com financeiro e societário',
 'Patrimônio médio-alto - veicular diversificado',
 'Baixo patrimônio - veicular concentrado com financeiro e societário',
 'Alto patrimônio - imobiliário diversificado com societário e créditos e direitos',
 'Patrimônio médio-baixo - societário + veicular',
 'Patrimônio médio-baixo - veicular concentrado']

In [100]:
# ============================================================
# 5.2 Ajuste manual opcional da lista de clusters-chave
# ============================================================
#
# Use esta célula se quiser fixar manualmente os clusters que serão analisados.
# Por padrão, ela mantém a seleção automática.
# ============================================================

clusters_interessantes_manual = None

# Exemplo, caso queira forçar manualmente:
# clusters_interessantes_manual = [
#     "Baixo patrimônio - financeiro concentrado",
#     "Patrimônio médio-baixo - veicular + financeiro",
#     "Patrimônio médio-alto - veicular diversificado",
#     "Alto patrimônio - imobiliário diversificado com societário e créditos",
# ]

if clusters_interessantes_manual is not None:
    clusters_interessantes = clusters_interessantes_manual

df_clusters_chave = (
    df_demo
    .loc[df_demo["perfil_cluster"].isin(clusters_interessantes)]
    .copy()
)

print("Candidatos nos clusters-chave:", len(df_clusters_chave))
print("Eleitos nos clusters-chave:", df_clusters_chave["eleito"].sum())
print("Clusters-chave:", df_clusters_chave["perfil_cluster"].nunique())

Candidatos nos clusters-chave: 3720
Eleitos nos clusters-chave: 465
Clusters-chave: 9


In [101]:
# ============================================================
# 5.3 Resumo eleitoral dos clusters-chave
# ============================================================

resumo_clusters_chave = (
    df_clusters_chave
    .groupby(["estrato", "perfil_cluster"], observed=True)
    .agg(
        candidatos=("SQ_CANDIDATO", "count"),
        eleitos=("eleito", "sum"),
        taxa_eleicao=("eleito", "mean")
    )
    .reset_index()
    .sort_values(["estrato", "taxa_eleicao"], ascending=[True, False])
)

resumo_clusters_chave

,estrato,perfil_cluster,candidatos,eleitos,taxa_eleicao
0,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,0.060870
1,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,0.035129
4,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,0.133333
2,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,0.117647
3,02_medio_baixo,Patrimônio médio-baixo - societário + veicular,326,21,0.064417
5,02_medio_baixo,Patrimônio médio-baixo - veicular concentrado,364,13,0.035714
6,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,392,77,0.196429
7,03_medio_alto,Patrimônio médio-alto - veicular diversificado,398,74,0.185930
8,04_alto,Alto patrimônio - imobiliário diversificado co...,474,141,0.297468


In [102]:
# ============================================================
# 5.4 Variáveis demográficas e interpretativas disponíveis
# ============================================================

variaveis_demo_candidatas = [
    "DS_GENERO",
    "DS_COR_RACA",
    "DS_GRAU_INSTRUCAO",
    "DS_OCUPACAO"
]

variaveis_contexto_candidatas = [
    "SG_PARTIDO",
    "DS_CARGO",
    "SG_UF"
]

variaveis_demo = [
    col for col in variaveis_demo_candidatas
    if col in df_demo.columns
]

variaveis_contexto = [
    col for col in variaveis_contexto_candidatas
    if col in df_demo.columns
]

print("Variáveis demográficas disponíveis:", variaveis_demo)
print("Variáveis contextuais disponíveis:", variaveis_contexto)

Variáveis demográficas disponíveis: ['DS_GENERO', 'DS_COR_RACA', 'DS_GRAU_INSTRUCAO', 'DS_OCUPACAO']
Variáveis contextuais disponíveis: ['SG_PARTIDO', 'DS_CARGO', 'SG_UF']


In [103]:
# ============================================================
# 5.5 Padronização simples das categorias
# ============================================================

def limpar_categoria_analise(serie):
    return (
        serie
        .astype(str)
        .str.upper()
        .str.strip()
        .replace({
            "NAN": "NÃO INFORMADO",
            "NONE": "NÃO INFORMADO",
            "": "NÃO INFORMADO",
            "#NULO#": "NÃO INFORMADO",
            "NÃO DIVULGÁVEL": "NÃO INFORMADO"
        })
    )

for col in variaveis_demo + variaveis_contexto:
    df_demo[col + "_LIMPA"] = limpar_categoria_analise(df_demo[col])
    df_clusters_chave[col + "_LIMPA"] = limpar_categoria_analise(df_clusters_chave[col])

In [104]:
# ============================================================
# 5.6 Função para composição demográfica dos clusters-chave
# ============================================================

def composicao_variavel_clusters(df, variavel):
    var = variavel + "_LIMPA"
    
    # composição no cluster
    comp_cluster = (
        df
        .groupby(["estrato", "perfil_cluster", var], observed=True)
        .agg(candidatos_cluster_categoria=("SQ_CANDIDATO", "count"))
        .reset_index()
    )
    
    total_cluster = (
        df
        .groupby(["estrato", "perfil_cluster"], observed=True)
        .agg(candidatos_cluster=("SQ_CANDIDATO", "count"))
        .reset_index()
    )
    
    comp_cluster = comp_cluster.merge(
        total_cluster,
        on=["estrato", "perfil_cluster"],
        how="left"
    )
    
    comp_cluster["perc_cluster"] = (
        comp_cluster["candidatos_cluster_categoria"]
        / comp_cluster["candidatos_cluster"]
    )
    
    # composição no estrato
    comp_estrato = (
        df_demo
        .groupby(["estrato", var], observed=True)
        .agg(candidatos_estrato_categoria=("SQ_CANDIDATO", "count"))
        .reset_index()
    )
    
    total_estrato = (
        df_demo
        .groupby("estrato", observed=True)
        .agg(candidatos_estrato=("SQ_CANDIDATO", "count"))
        .reset_index()
    )
    
    comp_estrato = comp_estrato.merge(
        total_estrato,
        on="estrato",
        how="left"
    )
    
    comp_estrato["perc_estrato"] = (
        comp_estrato["candidatos_estrato_categoria"]
        / comp_estrato["candidatos_estrato"]
    )
    
    # composição na base geral
    comp_base = (
        df_demo
        .groupby(var, observed=True)
        .agg(candidatos_base_categoria=("SQ_CANDIDATO", "count"))
        .reset_index()
    )
    
    comp_base["candidatos_base"] = len(df_demo)
    comp_base["perc_base"] = (
        comp_base["candidatos_base_categoria"]
        / comp_base["candidatos_base"]
    )
    
    resultado = (
        comp_cluster
        .merge(
            comp_estrato[["estrato", var, "perc_estrato"]],
            on=["estrato", var],
            how="left"
        )
        .merge(
            comp_base[[var, "perc_base"]],
            on=var,
            how="left"
        )
    )
    
    resultado["variavel"] = variavel
    resultado = resultado.rename(columns={var: "categoria"})
    
    resultado["indice_vs_estrato"] = np.where(
        resultado["perc_estrato"] > 0,
        resultado["perc_cluster"] / resultado["perc_estrato"],
        np.nan
    )
    
    resultado["indice_vs_base"] = np.where(
        resultado["perc_base"] > 0,
        resultado["perc_cluster"] / resultado["perc_base"],
        np.nan
    )
    
    resultado["delta_pp_vs_estrato"] = (
        resultado["perc_cluster"] - resultado["perc_estrato"]
    ) * 100
    
    resultado["delta_pp_vs_base"] = (
        resultado["perc_cluster"] - resultado["perc_base"]
    ) * 100
    
    return resultado[
        [
            "variavel",
            "estrato",
            "perfil_cluster",
            "categoria",
            "candidatos_cluster_categoria",
            "candidatos_cluster",
            "perc_cluster",
            "perc_estrato",
            "perc_base",
            "indice_vs_estrato",
            "indice_vs_base",
            "delta_pp_vs_estrato",
            "delta_pp_vs_base"
        ]
    ]

In [105]:
# ============================================================
# 5.7 Composição demográfica dos clusters-chave
# ============================================================

tabelas_demo = []

for variavel in variaveis_demo:
    tabelas_demo.append(
        composicao_variavel_clusters(df_clusters_chave, variavel)
    )

tabela_eixo5_demografia = pd.concat(
    tabelas_demo,
    ignore_index=True
)

tabela_eixo5_demografia = tabela_eixo5_demografia.sort_values(
    [
        "variavel",
        "perfil_cluster",
        "perc_cluster"
    ],
    ascending=[True, True, False]
)

tabela_eixo5_demografia.head(30)

,variavel,estrato,perfil_cluster,categoria,candidatos_cluster_categoria,candidatos_cluster,perc_cluster,perc_estrato,perc_base,indice_vs_estrato,indice_vs_base,delta_pp_vs_estrato,delta_pp_vs_base
61,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,BRANCA,373,474,0.786920,0.678685,0.550469,1.159478,1.429544,10.823502,23.645054
64,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,PARDA,87,474,0.183544,0.268254,0.327351,0.684218,0.560696,-8.470966,-14.380626
60,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,AMARELA,7,474,0.014768,0.006122,0.004446,2.412096,3.321691,0.864548,1.032202
63,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,NÃO INFORMADO,3,474,0.006329,0.003401,0.002580,1.860759,2.453407,0.292775,0.374939
65,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,PRETA,3,474,0.006329,0.041723,0.110544,0.151692,0.057254,-3.539424,-10.421482
62,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,INDÍGENA,1,474,0.002110,0.001814,0.004611,1.162975,0.457580,0.029565,-0.250087
19,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,BRANCA,358,690,0.518841,0.452749,0.550469,1.145978,0.942542,6.609128,-3.162871
22,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,PARDA,200,690,0.289855,0.364958,0.327351,0.794215,0.885458,-7.510299,-3.749550
23,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,PRETA,122,690,0.176812,0.168127,0.110544,1.051657,1.599469,0.868485,6.626766
20,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,INDÍGENA,7,690,0.010145,0.007829,0.004611,1.295894,2.200362,0.231641,0.553436


In [106]:
# ============================================================
# 5.8 Principais traços demográficos de cada cluster-chave
# ============================================================
#
# Mostra as categorias mais presentes em cada cluster para cada variável.
# ============================================================

TOP_CATEGORIAS_DEMO = 3

top_demografia_clusters = (
    tabela_eixo5_demografia
    .sort_values(
        ["variavel", "perfil_cluster", "perc_cluster"],
        ascending=[True, True, False]
    )
    .groupby(["variavel", "perfil_cluster"], observed=True)
    .head(TOP_CATEGORIAS_DEMO)
    .reset_index(drop=True)
)

top_demografia_clusters[
    [
        "variavel",
        "perfil_cluster",
        "categoria",
        "candidatos_cluster_categoria",
        "candidatos_cluster",
        "perc_cluster",
        "perc_estrato",
        "indice_vs_estrato",
        "delta_pp_vs_estrato"
    ]
].head(40)

,variavel,perfil_cluster,categoria,candidatos_cluster_categoria,candidatos_cluster,perc_cluster,perc_estrato,indice_vs_estrato,delta_pp_vs_estrato
0,DS_COR_RACA,Alto patrimônio - imobiliário diversificado co...,BRANCA,373,474,0.786920,0.678685,1.159478,10.823502
1,DS_COR_RACA,Alto patrimônio - imobiliário diversificado co...,PARDA,87,474,0.183544,0.268254,0.684218,-8.470966
2,DS_COR_RACA,Alto patrimônio - imobiliário diversificado co...,AMARELA,7,474,0.014768,0.006122,2.412096,0.864548
3,DS_COR_RACA,Baixo patrimônio - financeiro concentrado,BRANCA,358,690,0.518841,0.452749,1.145978,6.609128
4,DS_COR_RACA,Baixo patrimônio - financeiro concentrado,PARDA,200,690,0.289855,0.364958,0.794215,-7.510299
5,DS_COR_RACA,Baixo patrimônio - financeiro concentrado,PRETA,122,690,0.176812,0.168127,1.051657,0.868485
6,DS_COR_RACA,Baixo patrimônio - veicular concentrado com fi...,BRANCA,244,427,0.571429,0.452749,1.262130,11.867927
7,DS_COR_RACA,Baixo patrimônio - veicular concentrado com fi...,PARDA,118,427,0.276347,0.364958,0.757201,-8.861146
8,DS_COR_RACA,Baixo patrimônio - veicular concentrado com fi...,PRETA,56,427,0.131148,0.168127,0.780052,-3.697921
9,DS_COR_RACA,Patrimônio médio-alto - imobiliário concentrad...,BRANCA,267,392,0.681122,0.581715,1.170886,9.940704


In [107]:
# ============================================================
# 5.9 Categorias mais sobre-representadas nos clusters-chave
# ============================================================
#
# Aqui o foco não é a categoria mais frequente,
# mas a categoria mais acima do padrão do próprio estrato.
# ============================================================

MIN_CANDIDATOS_CATEGORIA_CLUSTER = 10

sobre_representacao_demo = (
    tabela_eixo5_demografia
    .loc[
        tabela_eixo5_demografia["candidatos_cluster_categoria"]
        >= MIN_CANDIDATOS_CATEGORIA_CLUSTER
    ]
    .copy()
)

sobre_representacao_demo = sobre_representacao_demo.sort_values(
    [
        "indice_vs_estrato",
        "delta_pp_vs_estrato",
        "candidatos_cluster_categoria"
    ],
    ascending=[False, False, False]
)

sobre_representacao_demo[
    [
        "variavel",
        "estrato",
        "perfil_cluster",
        "categoria",
        "candidatos_cluster_categoria",
        "candidatos_cluster",
        "perc_cluster",
        "perc_estrato",
        "indice_vs_estrato",
        "delta_pp_vs_estrato"
    ]
].head(30)

,variavel,estrato,perfil_cluster,categoria,candidatos_cluster_categoria,candidatos_cluster,perc_cluster,perc_estrato,indice_vs_estrato,delta_pp_vs_estrato
427,DS_OCUPACAO,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,DEPUTADO,34,360,0.094444,0.027473,3.437778,6.697192
152,DS_OCUPACAO,01_baixo,Baixo patrimônio - financeiro concentrado,DEPUTADO,27,690,0.039130,0.011556,3.386045,2.757405
377,DS_OCUPACAO,02_medio_baixo,Patrimônio médio-baixo - societário + veicular,EMPRESÁRIO,130,326,0.398773,0.128046,3.114298,27.072714
313,DS_OCUPACAO,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,DEPUTADO,23,289,0.079585,0.027473,2.896886,5.211225
388,DS_OCUPACAO,02_medio_baixo,Patrimônio médio-baixo - societário + veicular,MÉDICO,14,326,0.042945,0.016245,2.643630,2.670016
163,DS_OCUPACAO,01_baixo,Baixo patrimônio - financeiro concentrado,"ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS",29,690,0.042029,0.016403,2.562335,2.562638
498,DS_OCUPACAO,02_medio_baixo,Patrimônio médio-baixo - veicular concentrado,ENGENHEIRO,11,364,0.030220,0.012900,2.342593,1.731964
181,DS_OCUPACAO,01_baixo,Baixo patrimônio - financeiro concentrado,MÉDICO,11,690,0.015942,0.006897,2.311594,0.904548
431,DS_OCUPACAO,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,ENGENHEIRO,10,360,0.027778,0.012900,2.153292,1.487763
552,DS_OCUPACAO,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,DEPUTADO,51,392,0.130102,0.063160,2.059865,6.694156


In [108]:
# ============================================================
# 5.10 Composição dos eleitos nos clusters-chave
# ============================================================

df_eleitos_clusters_chave = (
    df_clusters_chave
    .loc[df_clusters_chave["eleito"] == 1]
    .copy()
)

tabelas_demo_eleitos = []

for variavel in variaveis_demo:
    var = variavel + "_LIMPA"
    
    comp = (
        df_eleitos_clusters_chave
        .groupby(["estrato", "perfil_cluster", var], observed=True)
        .agg(eleitos_cluster_categoria=("SQ_CANDIDATO", "count"))
        .reset_index()
        .rename(columns={var: "categoria"})
    )
    
    total = (
        df_eleitos_clusters_chave
        .groupby(["estrato", "perfil_cluster"], observed=True)
        .agg(eleitos_cluster=("SQ_CANDIDATO", "count"))
        .reset_index()
    )
    
    comp = comp.merge(
        total,
        on=["estrato", "perfil_cluster"],
        how="left"
    )
    
    comp["perc_eleitos_cluster"] = (
        comp["eleitos_cluster_categoria"]
        / comp["eleitos_cluster"]
    )
    
    comp["variavel"] = variavel
    
    tabelas_demo_eleitos.append(comp)

tabela_eixo5_demografia_eleitos = pd.concat(
    tabelas_demo_eleitos,
    ignore_index=True
)

tabela_eixo5_demografia_eleitos = tabela_eixo5_demografia_eleitos[
    [
        "variavel",
        "estrato",
        "perfil_cluster",
        "categoria",
        "eleitos_cluster_categoria",
        "eleitos_cluster",
        "perc_eleitos_cluster"
    ]
].sort_values(
    ["variavel", "perfil_cluster", "perc_eleitos_cluster"],
    ascending=[True, True, False]
)

tabela_eixo5_demografia_eleitos.head(30)

,variavel,estrato,perfil_cluster,categoria,eleitos_cluster_categoria,eleitos_cluster,perc_eleitos_cluster
46,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,BRANCA,110,141,0.780142
48,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,PARDA,25,141,0.177305
49,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,PRETA,3,141,0.021277
47,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,NÃO INFORMADO,2,141,0.014184
45,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,AMARELA,1,141,0.007092
18,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,BRANCA,30,42,0.714286
21,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,PRETA,6,42,0.142857
20,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,PARDA,5,42,0.119048
19,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,INDÍGENA,1,42,0.023810
23,DS_COR_RACA,01_baixo,Baixo patrimônio - veicular concentrado com fi...,PARDA,8,15,0.533333


In [109]:
# ============================================================
# 5.11 Principais características dos eleitos nos clusters-chave
# ============================================================

TOP_CATEGORIAS_ELEITOS = 3

top_demografia_eleitos_clusters = (
    tabela_eixo5_demografia_eleitos
    .sort_values(
        ["variavel", "perfil_cluster", "perc_eleitos_cluster"],
        ascending=[True, True, False]
    )
    .groupby(["variavel", "perfil_cluster"], observed=True)
    .head(TOP_CATEGORIAS_ELEITOS)
    .reset_index(drop=True)
)

top_demografia_eleitos_clusters.head(40)

,variavel,estrato,perfil_cluster,categoria,eleitos_cluster_categoria,eleitos_cluster,perc_eleitos_cluster
0,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,BRANCA,110,141,0.780142
1,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,PARDA,25,141,0.177305
2,DS_COR_RACA,04_alto,Alto patrimônio - imobiliário diversificado co...,PRETA,3,141,0.021277
3,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,BRANCA,30,42,0.714286
4,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,PRETA,6,42,0.142857
5,DS_COR_RACA,01_baixo,Baixo patrimônio - financeiro concentrado,PARDA,5,42,0.119048
6,DS_COR_RACA,01_baixo,Baixo patrimônio - veicular concentrado com fi...,PARDA,8,15,0.533333
7,DS_COR_RACA,01_baixo,Baixo patrimônio - veicular concentrado com fi...,BRANCA,6,15,0.400000
8,DS_COR_RACA,01_baixo,Baixo patrimônio - veicular concentrado com fi...,PRETA,1,15,0.066667
9,DS_COR_RACA,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,BRANCA,56,77,0.727273


In [110]:
# ============================================================
# 5.12 Variáveis contextuais dos clusters-chave
# Ocupação já pode estar em variáveis_demo; aqui entram partido, cargo e UF.
# ============================================================

tabelas_contexto = []

for variavel in variaveis_contexto:
    tabelas_contexto.append(
        composicao_variavel_clusters(df_clusters_chave, variavel)
    )

if tabelas_contexto:
    tabela_eixo5_contexto = pd.concat(
        tabelas_contexto,
        ignore_index=True
    )
    
    tabela_eixo5_contexto = tabela_eixo5_contexto.sort_values(
        [
            "variavel",
            "perfil_cluster",
            "perc_cluster"
        ],
        ascending=[True, True, False]
    )
else:
    tabela_eixo5_contexto = pd.DataFrame()

tabela_eixo5_contexto.head(30)

,variavel,estrato,perfil_cluster,categoria,candidatos_cluster_categoria,candidatos_cluster,perc_cluster,perc_estrato,perc_base,indice_vs_estrato,indice_vs_base,delta_pp_vs_estrato,delta_pp_vs_base
342,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,DEPUTADO ESTADUAL,198,474,0.417722,0.479365,0.551951,0.871406,0.756809,-6.164356,-13.422974
343,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,DEPUTADO FEDERAL,197,474,0.415612,0.410658,0.371700,1.012064,1.118138,0.495422,4.391194
344,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,GOVERNADOR,20,474,0.042194,0.018367,0.010593,2.297234,3.983079,2.382675,3.160076
346,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,SENADOR,16,474,0.033755,0.023129,0.011691,1.459419,2.887265,1.062602,2.206418
339,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,1º SUPLENTE,13,474,0.027426,0.014966,0.010648,1.832566,2.575656,1.246017,1.677794
347,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,VICE-GOVERNADOR,13,474,0.027426,0.015646,0.010374,1.752889,2.643795,1.177990,1.705237
340,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,2º SUPLENTE,12,474,0.025316,0.013605,0.010099,1.860759,2.506742,1.171101,1.521711
345,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,PRESIDENTE,3,474,0.006329,0.001361,0.000714,4.651899,8.870010,0.496857,0.561557
341,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,DEPUTADO DISTRITAL,1,474,0.002110,0.021542,0.021626,0.097935,0.097555,-1.943225,-1.951607
348,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,VICE-PRESIDENTE,1,474,0.002110,0.001361,0.000604,1.550633,3.494246,0.074916,0.150594


In [111]:
# ============================================================
# 5.13 Top categorias contextuais por cluster-chave
# ============================================================

TOP_CATEGORIAS_CONTEXTO = 5

if not tabela_eixo5_contexto.empty:
    top_contexto_clusters = (
        tabela_eixo5_contexto
        .sort_values(
            ["variavel", "perfil_cluster", "perc_cluster"],
            ascending=[True, True, False]
        )
        .groupby(["variavel", "perfil_cluster"], observed=True)
        .head(TOP_CATEGORIAS_CONTEXTO)
        .reset_index(drop=True)
    )
else:
    top_contexto_clusters = pd.DataFrame()

top_contexto_clusters.head(50)

,variavel,estrato,perfil_cluster,categoria,candidatos_cluster_categoria,candidatos_cluster,perc_cluster,perc_estrato,perc_base,indice_vs_estrato,indice_vs_base,delta_pp_vs_estrato,delta_pp_vs_base
0,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,DEPUTADO ESTADUAL,198,474,0.417722,0.479365,0.551951,0.871406,0.756809,-6.164356,-13.422974
1,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,DEPUTADO FEDERAL,197,474,0.415612,0.410658,0.371700,1.012064,1.118138,0.495422,4.391194
2,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,GOVERNADOR,20,474,0.042194,0.018367,0.010593,2.297234,3.983079,2.382675,3.160076
3,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,SENADOR,16,474,0.033755,0.023129,0.011691,1.459419,2.887265,1.062602,2.206418
4,DS_CARGO,04_alto,Alto patrimônio - imobiliário diversificado co...,1º SUPLENTE,13,474,0.027426,0.014966,0.010648,1.832566,2.575656,1.246017,1.677794
...,...,...,...,...,...,...,...,...,...,...,...,...,...
45,SG_PARTIDO,04_alto,Alto patrimônio - imobiliário diversificado co...,UNIÃO,48,474,0.101266,0.086395,0.063615,1.172132,1.591857,1.487126,3.765092
46,SG_PARTIDO,04_alto,Alto patrimônio - imobiliário diversificado co...,PP,47,474,0.099156,0.074150,0.054010,1.337243,1.835900,2.500646,4.514657
47,SG_PARTIDO,04_alto,Alto patrimônio - imobiliário diversificado co...,MDB,45,474,0.094937,0.068254,0.053570,1.390933,1.772184,2.668274,4.136626
48,SG_PARTIDO,04_alto,Alto patrimônio - imobiliário diversificado co...,PL,45,474,0.094937,0.085941,0.066634,1.104673,1.424754,0.899567,2.830297


In [112]:
# ============================================================
# 5.14 Tabela-resumo textual por cluster-chave
# ============================================================

def resumir_top_categorias(tabela, variavel, n=3, coluna_perc="perc_cluster"):
    if tabela.empty:
        return pd.DataFrame()
    
    temp = (
        tabela
        .loc[tabela["variavel"] == variavel]
        .sort_values(["perfil_cluster", coluna_perc], ascending=[True, False])
        .groupby("perfil_cluster", observed=True)
        .head(n)
        .copy()
    )
    
    temp["item"] = (
        temp["categoria"].astype(str)
        + " ("
        + (temp[coluna_perc] * 100).round(1).astype(str)
        + "%)"
    )
    
    return (
        temp
        .groupby("perfil_cluster", observed=True)["item"]
        .apply(lambda x: "; ".join(x))
        .reset_index(name=f"top_{variavel.lower()}")
    )

resumo_interpretativo_clusters = resumo_clusters_chave.copy()

for variavel in variaveis_demo:
    resumo_var = resumir_top_categorias(
        tabela_eixo5_demografia,
        variavel,
        n=3,
        coluna_perc="perc_cluster"
    )
    
    resumo_interpretativo_clusters = resumo_interpretativo_clusters.merge(
        resumo_var,
        on="perfil_cluster",
        how="left"
    )

resumo_interpretativo_clusters

,estrato,perfil_cluster,candidatos,eleitos,taxa_eleicao,top_ds_genero,top_ds_cor_raca,top_ds_grau_instrucao,top_ds_ocupacao
0,01_baixo,Baixo patrimônio - financeiro concentrado,690,42,0.060870,MASCULINO (58.8%); FEMININO (41.2%),BRANCA (51.9%); PARDA (29.0%); PRETA (17.7%),SUPERIOR COMPLETO (57.4%); ENSINO MÉDIO COMPLE...,OUTROS (20.3%); EMPRESÁRIO (7.5%); ADVOGADO (7...
1,01_baixo,Baixo patrimônio - veicular concentrado com fi...,427,15,0.035129,MASCULINO (64.6%); FEMININO (35.4%),BRANCA (57.1%); PARDA (27.6%); PRETA (13.1%),SUPERIOR COMPLETO (67.2%); ENSINO MÉDIO COMPLE...,OUTROS (12.9%); VEREADOR (9.4%); EMPRESÁRIO (8...
2,02_medio_baixo,Patrimônio médio-baixo - veicular + financeiro,360,48,0.133333,MASCULINO (68.6%); FEMININO (31.4%),BRANCA (63.6%); PARDA (29.7%); PRETA (6.4%),SUPERIOR COMPLETO (75.6%); ENSINO MÉDIO COMPLE...,ADVOGADO (11.7%); DEPUTADO (9.4%); OUTROS (9.2%)
3,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,289,34,0.117647,MASCULINO (66.1%); FEMININO (33.9%),BRANCA (62.6%); PARDA (25.3%); PRETA (9.7%),SUPERIOR COMPLETO (74.4%); ENSINO MÉDIO COMPLE...,ADVOGADO (13.5%); EMPRESÁRIO (8.7%); OUTROS (8...
4,02_medio_baixo,Patrimônio médio-baixo - societário + veicular,326,21,0.064417,MASCULINO (79.8%); FEMININO (20.2%),BRANCA (57.1%); PARDA (37.1%); PRETA (5.2%),SUPERIOR COMPLETO (58.0%); ENSINO MÉDIO COMPLE...,EMPRESÁRIO (39.9%); OUTROS (8.6%); ADVOGADO (8...
5,02_medio_baixo,Patrimônio médio-baixo - veicular concentrado,364,13,0.035714,MASCULINO (77.2%); FEMININO (22.8%),BRANCA (48.9%); PARDA (40.7%); PRETA (9.3%),SUPERIOR COMPLETO (60.4%); ENSINO MÉDIO COMPLE...,EMPRESÁRIO (16.5%); OUTROS (11.5%); ADVOGADO (...
6,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,392,77,0.196429,MASCULINO (71.7%); FEMININO (28.3%),BRANCA (68.1%); PARDA (24.7%); PRETA (6.1%),SUPERIOR COMPLETO (82.1%); SUPERIOR INCOMPLETO...,ADVOGADO (15.1%); DEPUTADO (13.0%); EMPRESÁRIO...
7,03_medio_alto,Patrimônio médio-alto - veicular diversificado,398,74,0.185930,MASCULINO (79.1%); FEMININO (20.9%),BRANCA (64.8%); PARDA (28.1%); PRETA (6.3%),SUPERIOR COMPLETO (68.3%); ENSINO MÉDIO COMPLE...,EMPRESÁRIO (23.4%); DEPUTADO (12.3%); ADVOGADO...
8,04_alto,Alto patrimônio - imobiliário diversificado co...,474,141,0.297468,MASCULINO (88.6%); FEMININO (11.4%),BRANCA (78.7%); PARDA (18.4%); AMARELA (1.5%),SUPERIOR COMPLETO (85.0%); ENSINO MÉDIO COMPLE...,EMPRESÁRIO (23.0%); DEPUTADO (18.4%); ADVOGADO...


In [113]:
# ============================================================
# 5.15 Salvamento das tabelas do novo Eixo 5
# ============================================================

from pathlib import Path

PASTA_SAIDAS = Path("results")
PASTA_SAIDAS.mkdir(exist_ok=True)

resumo_clusters_chave.to_csv(
    PASTA_SAIDAS / "eixo5_resumo_clusters_chave.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo5_demografia.to_csv(
    PASTA_SAIDAS / "eixo5_demografia_clusters_chave.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

tabela_eixo5_demografia_eleitos.to_csv(
    PASTA_SAIDAS / "eixo5_demografia_eleitos_clusters_chave.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

if not tabela_eixo5_contexto.empty:
    tabela_eixo5_contexto.to_csv(
        PASTA_SAIDAS / "eixo5_contexto_clusters_chave.csv",
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

resumo_interpretativo_clusters.to_csv(
    PASTA_SAIDAS / "eixo5_resumo_interpretativo_clusters_chave.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

In [114]:
# ============================================================
# 5.16 Configurações para gráficos do Eixo 5
# ============================================================

import plotly.express as px
import textwrap

PASTA_GRAFICOS = Path("plot")
PASTA_GRAFICOS.mkdir(exist_ok=True)

if "RENDERIZAR_NO_NOTEBOOK" not in globals():
    RENDERIZAR_NO_NOTEBOOK = False

def quebra_label(texto, largura=55):
    if pd.isna(texto):
        return texto
    return "<br>".join(textwrap.wrap(str(texto), width=largura))

In [115]:
# ============================================================
# 5.17 Gráfico — Composição demográfica dos clusters-chave
# Um HTML por variável demográfica
# ============================================================

for variavel in variaveis_demo:
    
    df_plot = (
        tabela_eixo5_demografia
        .loc[tabela_eixo5_demografia["variavel"] == variavel]
        .copy()
    )
    
    if df_plot.empty:
        continue
    
    # Limita categorias para evitar poluição visual
    categorias_top = (
        df_plot
        .groupby("categoria", observed=True)["candidatos_cluster_categoria"]
        .sum()
        .sort_values(ascending=False)
        .head(12)
        .index
    )
    
    df_plot = df_plot.loc[df_plot["categoria"].isin(categorias_top)].copy()
    df_plot["perfil_cluster_quebrado"] = df_plot["perfil_cluster"].apply(
        lambda x: quebra_label(x, 65)
    )
    
    fig = px.bar(
        df_plot,
        x="perfil_cluster_quebrado",
        y="perc_cluster",
        color="categoria",
        barmode="stack",
        hover_data={
            "perfil_cluster": True,
            "perfil_cluster_quebrado": False,
            "categoria": True,
            "candidatos_cluster_categoria": True,
            "candidatos_cluster": True,
            "perc_cluster": ":.2%",
            "perc_estrato": ":.2%",
            "indice_vs_estrato": ":.2f",
            "delta_pp_vs_estrato": ":.2f"
        },
        labels={
            "perfil_cluster_quebrado": "Cluster",
            "perc_cluster": "% do cluster",
            "categoria": variavel
        },
        title=f"Composição de {variavel} nos clusters-chave"
    )
    
    fig.update_yaxes(tickformat=".0%")
    fig.update_layout(height=800)
    
    nome_arquivo = f"eixo5_composicao_{variavel.lower()}_clusters_chave.html"
    fig.write_html(PASTA_GRAFICOS / nome_arquivo)
    
    if RENDERIZAR_NO_NOTEBOOK:
        fig.show()

In [116]:
# ============================================================
# 5.18 Gráfico — Sobre-representação demográfica vs estrato
# ============================================================

for variavel in variaveis_demo:
    
    df_plot = (
        sobre_representacao_demo
        .loc[sobre_representacao_demo["variavel"] == variavel]
        .copy()
    )
    
    if df_plot.empty:
        continue
    
    df_plot = (
        df_plot
        .sort_values(
            ["indice_vs_estrato", "delta_pp_vs_estrato"],
            ascending=[False, False]
        )
        .head(30)
        .copy()
    )
    
    df_plot["label"] = (
        df_plot["categoria"].astype(str)
        + " — "
        + df_plot["perfil_cluster"].apply(lambda x: quebra_label(x, 55))
    )
    
    fig = px.bar(
        df_plot.sort_values("indice_vs_estrato", ascending=True),
        x="indice_vs_estrato",
        y="label",
        orientation="h",
        hover_data={
            "estrato": True,
            "perfil_cluster": True,
            "categoria": True,
            "candidatos_cluster_categoria": True,
            "candidatos_cluster": True,
            "perc_cluster": ":.2%",
            "perc_estrato": ":.2%",
            "indice_vs_estrato": ":.2f",
            "delta_pp_vs_estrato": ":.2f"
        },
        labels={
            "indice_vs_estrato": "Índice vs composição do estrato",
            "label": "Categoria — Cluster"
        },
        title=f"Categorias de {variavel} sobre-representadas nos clusters-chave"
    )
    
    fig.add_vline(
        x=1,
        line_dash="dash",
        annotation_text="composição do estrato",
        annotation_position="top"
    )
    
    fig.update_layout(height=1000)
    
    nome_arquivo = f"eixo5_sobre_representacao_{variavel.lower()}_vs_estrato.html"
    fig.write_html(PASTA_GRAFICOS / nome_arquivo)
    
    if RENDERIZAR_NO_NOTEBOOK:
        fig.show()

In [117]:
# ============================================================
# 5.19 Gráfico — Composição dos eleitos nos clusters-chave
# Um HTML por variável demográfica
# ============================================================

for variavel in variaveis_demo:
    
    df_plot = (
        tabela_eixo5_demografia_eleitos
        .loc[tabela_eixo5_demografia_eleitos["variavel"] == variavel]
        .copy()
    )
    
    if df_plot.empty:
        continue
    
    categorias_top = (
        df_plot
        .groupby("categoria", observed=True)["eleitos_cluster_categoria"]
        .sum()
        .sort_values(ascending=False)
        .head(12)
        .index
    )
    
    df_plot = df_plot.loc[df_plot["categoria"].isin(categorias_top)].copy()
    df_plot["perfil_cluster_quebrado"] = df_plot["perfil_cluster"].apply(
        lambda x: quebra_label(x, 65)
    )
    
    fig = px.bar(
        df_plot,
        x="perfil_cluster_quebrado",
        y="perc_eleitos_cluster",
        color="categoria",
        barmode="stack",
        hover_data={
            "perfil_cluster": True,
            "perfil_cluster_quebrado": False,
            "categoria": True,
            "eleitos_cluster_categoria": True,
            "eleitos_cluster": True,
            "perc_eleitos_cluster": ":.2%"
        },
        labels={
            "perfil_cluster_quebrado": "Cluster",
            "perc_eleitos_cluster": "% dos eleitos do cluster",
            "categoria": variavel
        },
        title=f"Composição dos eleitos por {variavel} nos clusters-chave"
    )
    
    fig.update_yaxes(tickformat=".0%")
    fig.update_layout(height=800)
    
    nome_arquivo = f"eixo5_composicao_eleitos_{variavel.lower()}_clusters_chave.html"
    fig.write_html(PASTA_GRAFICOS / nome_arquivo)
    
    if RENDERIZAR_NO_NOTEBOOK:
        fig.show()